# Import

In [1]:
# ======================== CONFIGURATION ========================
# Options: 'ETTh1', 'ETTh2', 'ETTm1', 'ETTm2',
#          'exchange_rate', 'weather', 'ECL', 'national_illness'
Dataset_Name = 'ETTh2'

PROJECT_ROOT = '/mnt/d/Project/PivotFormer'

# ============================================================
# DATASET CONFIGURATION MAP
# ------------------------------------------------------------
# Maps Dataset_Name -> {
#   folder   : subfolder under data/raw/
#   file     : exact CSV filename
#   urls     : list of candidate download URLs (tried in order)
#   split    : 'ett_hourly'  -> fixed 8640/2880/2880
#              'ett_minute'  -> fixed 34560/11520/11520
#              'ratio_7_1_2' -> 70% / 10% / 20% (TSLib standard)
#   lookback : input window length for this dataset
#   horizons : forecast horizons for this dataset
# }
#
# NOTE ON URLS:
#   The raw.githubusercontent.com paths for non-ETT datasets on the
#   official thuml/Time-Series-Library repo return 404 (the repo uses
#   Google Drive for those files, not git). Hugging Face mirrors of
#   TSLib are used as the primary source. Multiple URLs are tried in
#   order for resilience.
# ============================================================
DATASET_CONFIG = {
    'ETTh1': {
        'folder':   'ETTh1',
        'file':     'ETTh1.csv',
        'urls': [
            'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv',
        ],
        'split':    'ett_hourly',
        'lookback': 96,
        'horizons': [96, 192, 336, 720],
    },
    'ETTh2': {
        'folder':   'ETTh2',
        'file':     'ETTh2.csv',
        'urls': [
            'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv',
        ],
        'split':    'ett_hourly',
        'lookback': 96,
        'horizons': [96, 192, 336, 720],
    },
    'ETTm1': {
        'folder':   'ETTm1',
        'file':     'ETTm1.csv',
        'urls': [
            'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv',
        ],
        'split':    'ett_minute',
        'lookback': 96,
        'horizons': [96, 192, 336, 720],
    },
    'ETTm2': {
        'folder':   'ETTm2',
        'file':     'ETTm2.csv',
        'urls': [
            'https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv',
        ],
        'split':    'ett_minute',
        'lookback': 96,
        'horizons': [96, 192, 336, 720],
    },
    'exchange_rate': {
        'folder':   'exchange_rate',
        'file':     'exchange_rate.csv',
        'urls': [
            'https://huggingface.co/datasets/Leopegasus/Time-Series-Library/resolve/main/exchange_rate/exchange_rate.csv',
            'https://huggingface.co/datasets/pkr7098/time_series_datasets/resolve/main/exchange_rate.csv',
        ],
        'split':    'ratio_7_1_2',
        'lookback': 96,
        'horizons': [96, 192, 336, 720],
    },
    'weather': {
        'folder':   'weather',
        'file':     'weather.csv',
        'urls': [
            'https://huggingface.co/datasets/Leopegasus/Time-Series-Library/resolve/main/weather/weather.csv',
            'https://huggingface.co/datasets/pkr7098/time_series_datasets/resolve/main/weather.csv',
        ],
        'split':    'ratio_7_1_2',
        'lookback': 96,
        'horizons': [96, 192, 336, 720],
    },
    'ECL': {
        'folder':   'electricity',
        'file':     'electricity.csv',
        'urls': [
            'https://huggingface.co/datasets/Leopegasus/Time-Series-Library/resolve/main/electricity/electricity.csv',
            'https://huggingface.co/datasets/pkr7098/time_series_datasets/resolve/main/electricity.csv',
        ],
        'split':    'ratio_7_1_2',
        'lookback': 96,
        'horizons': [96, 192, 336, 720],
    },
    'national_illness': {
        'folder':   'illness',
        'file':     'national_illness.csv',
        'urls': [
            'https://huggingface.co/datasets/Leopegasus/Time-Series-Library/resolve/main/illness/national_illness.csv',
            'https://huggingface.co/datasets/pkr7098/time_series_datasets/resolve/main/national_illness.csv',
        ],
        'split':    'ratio_7_1_2',
        'lookback': 36,
        'horizons': [24, 36, 48, 60],
    },
}

if Dataset_Name not in DATASET_CONFIG:
    raise ValueError(
        f"Unknown Dataset_Name '{Dataset_Name}'. "
        f"Choose from: {list(DATASET_CONFIG.keys())}")

_CFG            = DATASET_CONFIG[Dataset_Name]
DATASET_FOLDER  = _CFG['folder']
DATASET_FILE    = _CFG['file']
DATASET_URLS    = _CFG['urls']
DATASET_SPLIT   = _CFG['split']

LOOKBACK = _CFG['lookback']
HORIZONS = _CFG['horizons']

SEEDS = [0, 1, 2, 3, 4, 5]

EPOCHS = 40
BATCH_SIZE = 32
LR = 1e-4
DROPOUT = 0.1
D_MODEL = 16
PATIENCE = 5

print("=" * 60)
print(f"[CONFIG] Dataset   : {Dataset_Name}")
print(f"[CONFIG] Folder    : {DATASET_FOLDER}")
print(f"[CONFIG] File      : {DATASET_FILE}")
print(f"[CONFIG] Split     : {DATASET_SPLIT}")
print(f"[CONFIG] LOOKBACK  : {LOOKBACK}")
print(f"[CONFIG] HORIZONS  : {HORIZONS}")
print(f"[CONFIG] #URLs     : {len(DATASET_URLS)}")
print("=" * 60)


# =============================== IMPORT ================================
import os, sys, subprocess, time, json, random, glob, shutil, gc
os.environ['TF_DISABLE_GRAPPLER'] = '1'
import requests, pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt, seaborn as sns


# ======================== ENVIRONMENT DETECTION ========================
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Running on Colab: {IN_COLAB}")


# ======================== SETUP ========================
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_folder = '/content/drive/MyDrive/PivotFormer'
    PROJECT_ROOT = drive_folder
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.chdir(PROJECT_ROOT)
    print("Working directory (Colab):", os.getcwd())

    !pip install -q patool sktime reformer-pytorch
    !pip install -q tensorflow numpy pandas scikit-learn seaborn matplotlib plotly tqdm requests
    !python -m pip install -q python-docx

    if not os.path.exists('Time-Series-Library-main'):
        !git clone https://github.com/thuml/Time-Series-Library.git Time-Series-Library-main
    !pip install -r Time-Series-Library-main/requirements.txt 2>/dev/null
    !pip install scipy statsmodels pandas numpy scikit-learn

else:
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.chdir(PROJECT_ROOT)
    print("Working directory (local):", os.getcwd())

    if not os.path.exists('Time-Series-Library-main'):
        try:
            import subprocess as _sp
            _sp.run(['git', 'clone',
                     'https://github.com/thuml/Time-Series-Library.git',
                     'Time-Series-Library-main'], check=True)
        except Exception as e:
            print(f"[WARN] Could not clone TSLib repo: {e}")
            print("[WARN] Ensure 'Time-Series-Library-main' exists in PROJECT_ROOT")

    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'


# ---------------- DOWNLOAD HELPER WITH FALLBACKS ----------------
def _try_download(dataset_file, candidate_urls, timeout=60):
    """
    Try each URL in order. On success, save the file and return True.
    On total failure, print manual-download instructions and return False.
    """
    data_dir = os.path.dirname(dataset_file)
    os.makedirs(data_dir, exist_ok=True)

    for i, url in enumerate(candidate_urls, start=1):
        print(f"  [try {i}/{len(candidate_urls)}] {url}")
        try:
            resp = requests.get(url, timeout=timeout, allow_redirects=True)
        except Exception as e:
            print(f"    -> request error: {e}")
            continue

        if resp.status_code != 200:
            print(f"    -> failed (status={resp.status_code})")
            continue

        # sanity check: the payload should look like a CSV (a few KB+)
        if len(resp.content) < 1024:
            print(f"    -> payload too small ({len(resp.content)} bytes), skipping")
            continue

        # If server returned HTML (login page, error page), reject
        first_bytes = resp.content[:200].lower()
        if b'<html' in first_bytes or b'<!doctype' in first_bytes:
            print("    -> payload is HTML, not CSV, skipping")
            continue

        with open(dataset_file, 'wb') as f:
            f.write(resp.content)
        size_mb = len(resp.content) / 1024 / 1024
        print(f"    -> saved {dataset_file} ({size_mb:.2f} MB)")
        return True

    # All URLs failed
    print("\n" + "=" * 60)
    print(f"[ERROR] Could not download {dataset_file}")
    print("=" * 60)
    print("Please download the file manually from ONE of these sources")
    print("and place it at the path shown below:\n")
    print(f"  Target path: {dataset_file}\n")
    print("Candidate sources (choose one that works in your browser):")
    for j, url in enumerate(candidate_urls, start=1):
        print(f"  {j}. {url}")
    print()
    print("After placing the file, re-run this cell.")
    print("=" * 60)
    return False


# ------------- resolve project paths -------------
tslib_root = os.path.join(PROJECT_ROOT, 'Time-Series-Library-main')
data_root  = os.path.join(PROJECT_ROOT, 'data', 'raw', DATASET_FOLDER)
final_dir  = os.path.join(PROJECT_ROOT, 'results', f'{Dataset_Name}_baselines')
os.makedirs(data_root, exist_ok=True)
os.makedirs(final_dir, exist_ok=True)

print(f"\nPaths set:\n  tslib_root = {tslib_root}\n  data_root  = {data_root}\n  final_dir  = {final_dir}")


# ------------- ensure dataset file is present -------------
data_file = os.path.join(data_root, DATASET_FILE)
if not os.path.exists(data_file):
    print(f"\n[INFO] Dataset file not found: {data_file}")
    print(f"[INFO] Attempting automatic download ({len(DATASET_URLS)} URLs) ...")
    ok = _try_download(data_file, DATASET_URLS)
    if not ok:
        raise RuntimeError(
            f"Dataset file could not be downloaded. "
            f"Place '{DATASET_FILE}' at: {data_file}")
else:
    size_mb = os.path.getsize(data_file) / 1024 / 1024
    print(f"\n[INFO] Dataset file already present: {data_file} "
          f"({size_mb:.2f} MB)")


# ------------- inspect dataset -------------
df = pd.read_csv(data_file)
print(f"[INFO] Dataset shape: {df.shape}")
print(f"[INFO] Columns      : {list(df.columns)[:10]}"
      f"{'...' if len(df.columns) > 10 else ''}")
print(df.head())


# =============================================================================
# DATA LOADING
# =============================================================================
def load_multivariate_data(dataset_name, pred_len,
                           seq_len=LOOKBACK, data_root=None):
    """
    Load and window a multivariate time series.

    The file name and split ratio are resolved from DATASET_CONFIG:
      - exchange_rate, weather, ECL, national_illness -> 70/10/20
      - ETTh1/ETTh2                                   -> fixed (8640, 2880, 2880)
      - ETTm1/ETTm2                                   -> fixed (34560, 11520, 11520)
    """
    if dataset_name in DATASET_CONFIG:
        file_name  = DATASET_CONFIG[dataset_name]['file']
        split_type = DATASET_CONFIG[dataset_name]['split']
    else:
        file_name  = f"{dataset_name}.csv"
        split_type = 'ratio_7_1_2'

    if data_root is None:
        csv_path = file_name
    else:
        csv_path = os.path.join(data_root, file_name)

    df   = pd.read_csv(csv_path)
    cols = [c for c in df.columns if c.lower() != 'date']
    data = df[cols].values.astype(np.float32)

    if split_type == 'ett_hourly':
        train_len, val_len, test_len = 8640, 2880, 2880
    elif split_type == 'ett_minute':
        train_len, val_len, test_len = 34560, 11520, 11520
    else:  # 'ratio_7_1_2'
        n         = len(data)
        train_len = int(n * 0.7)
        val_len   = int(n * 0.1)
        test_len  = n - train_len - val_len

    scaler     = StandardScaler()
    train_data = scaler.fit_transform(data[:train_len])
    val_data   = scaler.transform(data[train_len:train_len + val_len])
    test_data  = scaler.transform(
        data[train_len + val_len: train_len + val_len + test_len])

    def make_windows(arr, sl, pl):
        X, Y = [], []
        for i in range(len(arr) - sl - pl + 1):
            X.append(arr[i:i + sl])
            Y.append(arr[i + sl:i + sl + pl])
        return (np.array(X, dtype=np.float32),
                np.array(Y, dtype=np.float32))

    trX, trY = make_windows(train_data, seq_len, pred_len)
    vaX, vaY = make_windows(val_data,   seq_len, pred_len)
    teX, teY = make_windows(test_data,  seq_len, pred_len)
    return trX, trY, vaX, vaY, teX, teY, scaler

[CONFIG] Dataset   : ETTh2
[CONFIG] Folder    : ETTh2
[CONFIG] File      : ETTh2.csv
[CONFIG] Split     : ett_hourly
[CONFIG] LOOKBACK  : 96
[CONFIG] HORIZONS  : [96, 192, 336, 720]
[CONFIG] #URLs     : 1
Running on Colab: False
Working directory (local): /mnt/d/Project/PivotFormer

Paths set:
  tslib_root = /mnt/d/Project/PivotFormer/Time-Series-Library-main
  data_root  = /mnt/d/Project/PivotFormer/data/raw/ETTh2
  final_dir  = /mnt/d/Project/PivotFormer/results/ETTh2_baselines

[INFO] Dataset file already present: /mnt/d/Project/PivotFormer/data/raw/ETTh2/ETTh2.csv (2.31 MB)
[INFO] Dataset shape: (17420, 8)
[INFO] Columns      : ['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']
                  date       HUFL    HULL       MUFL   MULL   LUFL   LULL  \
0  2016-07-01 00:00:00  41.130001  12.481  36.535999  9.355  4.424  1.311   
1  2016-07-01 01:00:00  37.528000  10.136  33.936001  7.532  4.435  1.215   
2  2016-07-01 02:00:00  37.946999  11.309  35.330002  9.007  2.100

# Load Dataset

In [2]:
# =============================================================================
# UNIFIED DATA LOADING CELL
# -----------------------------------------------------------------------------
# Provides a single data loading pipeline used by BOTH FKBFormer training
# and the TSLib baseline evaluation. The pipeline matches TSLib's
# Dataset_ETT_hour, Dataset_ETT_minute, and Dataset_Custom exactly:
#
#   1. Fixed borders for ETT datasets (12 + 4 + 4 months).
#   2. 70/10/20 split for custom datasets.
#   3. seq_len overlap at validation and test boundaries so that every
#      window has a complete input context.
#   4. StandardScaler fit on the training slice only, then applied to
#      all splits.
#
# This makes FKBFormer and the TSLib baselines see byte-identical data
# during training, validation, and test.
# =============================================================================
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Helper: resolve dataset file name
# ------------------------------------------------------------
def _resolve_dataset_file(dataset_name):
    """Return the actual CSV filename for a given dataset name."""
    if 'DATASET_CONFIG' in globals() and dataset_name in DATASET_CONFIG:
        return DATASET_CONFIG[dataset_name]['file']
    return f"{dataset_name}.csv"


# ------------------------------------------------------------
# Helper: resolve split type
# ------------------------------------------------------------
def _resolve_split_type(dataset_name):
    """Return the split mode: 'ett_hourly', 'ett_minute', or 'ratio_7_1_2'."""
    if 'DATASET_CONFIG' in globals() and dataset_name in DATASET_CONFIG:
        return DATASET_CONFIG[dataset_name]['split']
    if dataset_name.startswith('ETTh'):
        return 'ett_hourly'
    if dataset_name.startswith('ETTm'):
        return 'ett_minute'
    return 'ratio_7_1_2'


# ------------------------------------------------------------
# Helper: compute TSLib-compatible borders
# ------------------------------------------------------------
def _compute_tslib_borders(n_samples, split_type, seq_len):
    """
    Return (border1s, border2s) matching TSLib's data loaders.

    border1s[i] and border2s[i] define the range of rows used for split i
    (0=train, 1=val, 2=test). Validation and test ranges include seq_len
    samples of overlap from the previous split so that every window has
    a complete input context.
    """
    if split_type == 'ett_hourly':
        num_train = 12 * 30 * 24            # 8640
        num_vali  = 4 * 30 * 24             # 2880
        num_test  = 4 * 30 * 24             # 2880
    elif split_type == 'ett_minute':
        num_train = 12 * 30 * 24 * 4        # 34560
        num_vali  = 4 * 30 * 24 * 4         # 11520
        num_test  = 4 * 30 * 24 * 4         # 11520
    else:  # 'ratio_7_1_2'
        num_train = int(n_samples * 0.7)
        num_test  = int(n_samples * 0.2)
        num_vali  = n_samples - num_train - num_test

    raw_borders = [0,
                   num_train,
                   num_train + num_vali,
                   num_train + num_vali + num_test]

    # Clamp to actual number of rows (ETT CSVs may have 17420 rows)
    end = n_samples
    border1s = [raw_borders[0],
                max(0, raw_borders[1] - seq_len),
                max(0, raw_borders[2] - seq_len)]
    border2s = [min(raw_borders[1], end),
                min(raw_borders[2], end),
                min(raw_borders[3], end)]

    return border1s, border2s


# ------------------------------------------------------------
# Main loader: TSLib-compatible
# ------------------------------------------------------------
def load_multivariate_data(dataset_name, pred_len,
                           seq_len=LOOKBACK, data_root=None):
    """
    Load and window a multivariate time series exactly like TSLib.

    Parameters
    ----------
    dataset_name : str
        ETTh1, ETTh2, ETTm1, ETTm2, exchange_rate, weather, ECL, or
        national_illness.
    pred_len : int
        Forecast horizon.
    seq_len : int
        Lookback window length. Defaults to LOOKBACK from the config cell.
    data_root : str or None
        Folder containing the CSV. If None, the filename is used as a
        relative path.

    Returns
    -------
    trX, trY, vaX, vaY, teX, teY, scaler
    """
    file_name  = _resolve_dataset_file(dataset_name)
    split_type = _resolve_split_type(dataset_name)

    csv_path = (file_name if data_root is None
                else os.path.join(data_root, file_name))

    df   = pd.read_csv(csv_path)
    cols = [c for c in df.columns if c.lower() != 'date']
    data = df[cols].values.astype(np.float32)

    n = len(data)
    border1s, border2s = _compute_tslib_borders(n, split_type, seq_len)

    # Fit scaler on the TRAINING slice only (no seq_len overlap).
    # This matches TSLib's behavior.
    scaler = StandardScaler()
    scaler.fit(data[border1s[0]:border2s[0]])
    data_scaled = scaler.transform(data)

    tr_data = data_scaled[border1s[0]:border2s[0]]
    va_data = data_scaled[border1s[1]:border2s[1]]
    te_data = data_scaled[border1s[2]:border2s[2]]

    def make_windows(arr, sl, pl):
        X, Y = [], []
        for i in range(len(arr) - sl - pl + 1):
            X.append(arr[i:i + sl])
            Y.append(arr[i + sl:i + sl + pl])
        return (np.array(X, dtype=np.float32),
                np.array(Y, dtype=np.float32))

    trX, trY = make_windows(tr_data, seq_len, pred_len)
    vaX, vaY = make_windows(va_data, seq_len, pred_len)
    teX, teY = make_windows(te_data, seq_len, pred_len)

    return trX, trY, vaX, vaY, teX, teY, scaler


# ------------------------------------------------------------
# RMSSE denominator: seasonal-naive MSE per channel on training only
# ------------------------------------------------------------
def compute_train_seasonal_naive_mse_per_channel(dataset_name, data_root,
                                                  seasonal_period=24):
    """
    Per-channel seasonal-naive MSE on the TRAINING slice.

    The scaler is fit on the same training slice that
    load_multivariate_data uses, so the RMSSE denominator is
    consistent with the model evaluation.
    """
    file_name  = _resolve_dataset_file(dataset_name)
    split_type = _resolve_split_type(dataset_name)

    csv_path = os.path.join(data_root, file_name) if data_root else file_name
    df   = pd.read_csv(csv_path)
    cols = [c for c in df.columns if c.lower() != 'date']
    data = df[cols].values.astype(np.float32)

    n = len(data)
    border1s, border2s = _compute_tslib_borders(n, split_type, 0)
    train_len = border2s[0] - border1s[0]

    scaler       = StandardScaler()
    train_scaled = scaler.fit_transform(data[:train_len])

    if len(train_scaled) <= seasonal_period:
        return np.ones(train_scaled.shape[1], dtype=np.float64)

    forecasts = train_scaled[:-seasonal_period]
    actuals   = train_scaled[seasonal_period:]
    per_ch    = np.mean((actuals - forecasts) ** 2, axis=0)
    return per_ch.astype(np.float64)


# ------------------------------------------------------------
# Sanity check: print window counts for the current config
# ------------------------------------------------------------
def _print_window_counts_safe():
    """Print train/val/test window counts if config is ready."""
    if 'Dataset_Name' not in globals():
        return
    if 'HORIZONS' not in globals() or 'LOOKBACK' not in globals():
        return
    data_root_val = globals().get('data_root', None)
    print("=" * 70)
    print(f"[SANITY] Window counts for {Dataset_Name}")
    print("=" * 70)
    print(f"{'Horizon':>8s} | {'Train':>8s} | {'Val':>8s} | {'Test':>8s}")
    print("-" * 40)
    for h in HORIZONS:
        try:
            trX, trY, vaX, vaY, teX, teY, _ = load_multivariate_data(
                Dataset_Name, pred_len=h, seq_len=LOOKBACK,
                data_root=data_root_val)
            print(f"{h:>8d} | {len(trX):>8d} | {len(vaX):>8d} | {len(teX):>8d}")
        except Exception as e:
            print(f"{h:>8d} | ERROR: {e}")
    print("=" * 70)


_print_window_counts_safe()

[SANITY] Window counts for ETTh2
 Horizon |    Train |      Val |     Test
----------------------------------------
      96 |     8449 |     2785 |     2785
     192 |     8353 |     2689 |     2689
     336 |     8209 |     2545 |     2545
     720 |     7825 |     2161 |     2161


In [3]:
from pathlib import Path

dl_path = Path(PROJECT_ROOT) / 'Time-Series-Library-main' / 'data_provider' / 'data_loader.py'
text = dl_path.read_text(encoding='utf-8')

fixed = text.replace(".drop(['date'], 1)", ".drop(['date'], axis=1)")
fixed = fixed.replace('.drop(["date"], 1)', '.drop(["date"], axis=1)')

if fixed != text:
    dl_path.write_text(fixed, encoding='utf-8')
    print(f"[FIXED] Patched pandas drop(..., axis=1) in: {dl_path}")
else:
    print("[INFO] No matching pattern found (already patched).")

[INFO] No matching pattern found (already patched).


# TSlib Baselines

In [ ]:
# v16.2: original model configs, loop and report; isolated native evaluation.
# + Added TimesNet (CNN/2D-Variation) and TimeMixer (Multi-scale MLP) for Q1 completeness.
# + Saves final .npy/.json/.csv directly to ./results/{Dataset_Name}_baselines/ and cleans up evaluation_* dir.
from pathlib import Path
import tempfile
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, random, os, sys, subprocess, time, gc, re, math, json, shutil
import glob as glb
import matplotlib.pyplot as plt, pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


_REQUIRED_GLOBALS = [
    'load_multivariate_data',
    'compute_train_seasonal_naive_mse_per_channel',
    'Dataset_Name', 'LOOKBACK', 'HORIZONS', 'SEEDS',
    'EPOCHS', 'BATCH_SIZE', 'LR', 'PATIENCE',
    'PROJECT_ROOT', 'data_root', 'DATASET_FILE',
]
_missing = [g for g in _REQUIRED_GLOBALS if g not in globals()]
if _missing:
    raise RuntimeError(f"Missing required globals: {_missing}.")


# ============================================================
# CONFIG
# ============================================================
tslib_root = os.path.join(PROJECT_ROOT, 'Time-Series-Library-main')
OLD=Path(f'./results/{Dataset_Name}_baselines').resolve()
OUT=OLD.parent/(OLD.name); OUT.mkdir(parents=True,exist_ok=True)
RUN_DIR=Path(tempfile.mkdtemp(prefix='evaluation_',dir=OUT))
final_dir=str(OUT)
REPO=Path(tslib_root).resolve()
TRAIN_IF_MISSING = True
import hashlib

REPLAY_EXACT_CHECKPOINT=True
print('Output (final_dir):', final_dir)
print('Temp eval dir     :', str(RUN_DIR))
USE_CUDA = torch.cuda.is_available()
device   = torch.device('cuda' if USE_CUDA else 'cpu')
print(f"[INFO] CUDA: {USE_CUDA}")
if USE_CUDA:
    print(f"[INFO] GPU : {torch.cuda.get_device_name(0)}")


run_py = os.path.join(tslib_root, 'run.py')
if not os.path.isfile(run_py):
    raise FileNotFoundError(f"run.py not found: {run_py}")
def _check_model_file(m):
    return os.path.isfile(os.path.join(tslib_root, 'models', f'{m}.py'))
BASELINE_CONFIGS = {
    'PatchTST': {
        'd_model': 128, 'd_ff': 256, 'e_layers': 3, 'n_heads': 16,
        'patch_len': 16, 'stride': 8, 'dropout': 0.2, 'use_norm': 1,
        'train_epochs': EPOCHS, 'patience': PATIENCE,
        'batch_size': BATCH_SIZE, 'learning_rate': LR, 'extra': [],
    },
    'DLinear': {
        'd_model': 16, 'd_ff': 1, 'e_layers': 1, 'n_heads': 1,
        'moving_avg': 25, 'dropout': 0.05, 'use_norm': 1,
        'train_epochs': EPOCHS, 'patience': PATIENCE,
        'batch_size': BATCH_SIZE, 'learning_rate': LR, 'extra': [],
    },
    'Crossformer': {
        'd_model': 256, 'd_ff': 512, 'e_layers': 3, 'n_heads': 4,
        'seg_len': 6, 'win_size': 2, 'factor': 10,
        'channel_independence': 0, 'dropout': 0.2, 'use_norm': 1,
        'train_epochs': EPOCHS, 'patience': PATIENCE,
        'batch_size': BATCH_SIZE, 'learning_rate': 1e-4, 'extra': [],
    },
    'iTransformer': {
        'd_model': 512, 'd_ff': 2048, 'e_layers': 2, 'n_heads': 8,
        'dropout': 0.1, 'use_norm': 1,
        'train_epochs': 10, 'patience': 5,
        'batch_size': 16, 'learning_rate': 1e-4, 'extra': [],
    },
    'TimesNet': {
        'd_model': 16, 'd_ff': 32, 'e_layers': 2,
        'top_k': 5, 'num_kernels': 6,
        'dropout': 0.1, 'use_norm': 1,
        'train_epochs': 10, 'patience': 5,
        'batch_size': BATCH_SIZE, 'learning_rate': 1e-4, 'extra': [],
    },
    'TimeMixer': {
        'd_model': 16, 'd_ff': 32, 'e_layers': 2,
        'label_len': 0,
        'down_sampling_layers': 3, 'down_sampling_window': 2,
        'down_sampling_method': 'avg',
        'decomp_method': 'moving_avg', 'moving_avg': 25,
        'channel_independence': 1,
        'dropout': 0.1, 'use_norm': 1,
        'train_epochs': 10, 'patience': 5,
        'batch_size': 16, 'learning_rate': 1e-3, 'extra': [],
    },
}

DETERMINISTIC_MODELS = {'DLinear'}
RUN_SEEDS    = list(SEEDS)
RUN_HORIZONS = list(HORIZONS)


_available = [m for m in BASELINE_CONFIGS if _check_model_file(m)]
_missing_files = [m for m in BASELINE_CONFIGS if m not in _available]
if _missing_files:
    print(f"[WARN] Missing model files: {_missing_files}")
print(f"[INFO] Available: {_available}")


TSLIB_BASELINES = _available
if not TSLIB_BASELINES:
    raise RuntimeError("No baseline models found.")


models_tslib = list(TSLIB_BASELINES)
print(f"[INFO] Will run: {models_tslib}")


# ============================================================
# Dataset mappings
# ============================================================
TSLIB_DATA_ARG = {
    'ETTh1': 'ETTh1', 'ETTh2': 'ETTh2',
    'ETTm1': 'ETTm1', 'ETTm2': 'ETTm2',
    'exchange_rate': 'custom', 'weather': 'custom',
    'ECL': 'custom', 'national_illness': 'custom',
}
FREQ_MAP = {
    'ETTh1': 'h', 'ETTh2': 'h', 'ETTm1': 't', 'ETTm2': 't',
    'exchange_rate': 'd', 'weather': 't',
    'ECL': 'h', 'national_illness': 'w',
}
SEASONAL_MAP = {
    'ETTh1': 24, 'ETTh2': 24, 'ETTm1': 96, 'ETTm2': 96,
    'exchange_rate': 7, 'weather': 144,
    'ECL': 24, 'national_illness': 52,
}


if Dataset_Name not in TSLIB_DATA_ARG:
    raise ValueError(f"Unknown Dataset_Name: {Dataset_Name}")


tslib_data_arg  = TSLIB_DATA_ARG[Dataset_Name]
tslib_freq_arg  = FREQ_MAP[Dataset_Name]
seasonal_period = SEASONAL_MAP[Dataset_Name]
print(f"[INFO] Dataset={Dataset_Name} freq={tslib_freq_arg}")
# ============================================================
# Data probe
# ============================================================
csv_path = os.path.join(data_root, DATASET_FILE)
_df_probe = pd.read_csv(csv_path)
FEATURE_COLS = [c for c in _df_probe.columns if c.lower() != 'date']
N_CHANNELS = len(FEATURE_COLS)
TARGET_COL = 'OT' if 'OT' in _df_probe.columns else FEATURE_COLS[-1]
print(f"[INFO] {N_CHANNELS} channels, target={TARGET_COL}")
# RMSSE denominator
# ============================================================
train_naive_mse_per_channel = compute_train_seasonal_naive_mse_per_channel(
    Dataset_Name, data_root, seasonal_period=seasonal_period)
print(f"[INFO] RMSSE denom={np.mean(train_naive_mse_per_channel):.6f}")

CSV=Path(csv_path).resolve(); C=N_CHANNELS; sl=int(LOOKBACK); cols=FEATURE_COLS; freq=tslib_freq_arg
den=np.asarray(train_naive_mse_per_channel); configs={m:{k:v for k,v in cfg.items() if k!='extra'} for m,cfg in BASELINE_CONFIGS.items()}
if den.shape!=(C,): raise ValueError('Invalid RMSSE denominator')
csv_digest=hashlib.sha256(CSV.read_bytes()).hexdigest()
code_digest=hashlib.sha256(b''.join(p.read_bytes() for p in sorted(REPO.rglob('*.py')) if not {'results','checkpoints','.venv','venv'}.intersection(p.parts))).hexdigest()

WORKER = r'''
import sys, os, json, argparse, random, runpy
from pathlib import Path
import numpy as np, torch
work_dir = Path.cwd()
repo, specfile = sys.argv[1:3]
spec = json.loads(Path(specfile).read_text())
os.chdir(repo)
sys.path.insert(0, repo)
import exp.exp_long_term_forecasting as ex
origparse = argparse.ArgumentParser.parse_args
def parse(self, *a, **kw):
    # Supply configured values before argparse enforces fork-specific required flags.
    for action in self._actions:
        if action.option_strings and action.dest in spec['args']:
            action.required = False; action.default = spec['args'][action.dest]
    ns = origparse(self, *a, **kw)
    for k, v in spec['args'].items(): setattr(ns, k, v)
    random.seed(ns.seed); np.random.seed(ns.seed); torch.manual_seed(ns.seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(ns.seed)
    return ns
argparse.ArgumentParser.parse_args = parse
origmetric = ex.metric
calls = []
def metric(pred, true, *args, **kw):
    vals = origmetric(pred, true, *args, **kw); calls.append(1)
    if len(calls) != 1: raise RuntimeError('Ambiguous metric capture')
    np.save(work_dir / 'native_pred.npy', pred)
    np.save(work_dir / 'native_true.npy', true)
    return vals
ex.metric = metric
cls = ex.Exp_Long_Term_Forecast; oldtest = cls.test
def test(self, setting, *args, **kw):
    ck = spec['checkpoint']
    if ck:
        if setting != spec['setting']: raise RuntimeError('Checkpoint setting mismatch: ' + setting)
        self.model.load_state_dict(torch.load(ck, map_location=self.device), strict=True)
    (work_dir / 'effective.json').write_text(json.dumps(dict(args=vars(self.args), setting=setting,
        params=sum(p.numel() for p in self.model.parameters() if p.requires_grad)), default=str))
    return oldtest(self, setting, test=0)
cls.test = test
sys.argv = [str(Path(repo) / 'run.py'), '--task_name', 'long_term_forecast', '--is_training', str(spec['args']['is_training']),
 '--model_id', spec['args']['model_id'], '--model', spec['args']['model'], '--data', spec['args']['data']]
runpy.run_path(sys.argv[0], run_name='__main__')
if len(calls) != 1: raise RuntimeError('Expected exactly one native metric call')
'''
compile(WORKER,'native_worker','exec')
num=r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)'
def scores(text):
    pairs=re.findall(r'\bmse\s*:\s*'+num+r'\s*,\s*mae\s*:\s*'+num,text,re.I)
    return np.array(pairs[0],float) if len(pairs)==1 else None

def close(a,b): return b is not None and np.allclose(a,b,rtol=2e-5,atol=1e-6)
def run_one(m,h,s,row,ref):
    rid=f'{m}_{Dataset_Name}_{h}_seed{s}'; log=OLD/f'log_{rid}.txt'
    text=log.read_text(errors='replace') if log.is_file() else ''
    oklog=text.startswith('RETURNCODE: 0\n'); native=scores(text) if oklog else None
    if native is not None: row.update(old_native_mse=native[0],old_native_mae=native[1])
    settings=re.findall(r'>{3,}\s*testing\s*:\s*([^\r\n<]+?)\s*<{3,}',text,re.I)
    setting=settings[0].strip() if len(settings)==1 and rid+'_' in settings[0] else None
    if setting and (setting in ('.','..') or Path(setting).name!=setting): raise ValueError('invalid setting')
    res=REPO/'results'/setting if setting else None
    ck=REPO/'checkpoints'/setting/'checkpoint.pth' if setting else None
    B=len(ref)
    oldmetrics=OLD/(rid+'.npy'); oldpred=OLD/(rid+'_preds.npy'); params=None
    if oldmetrics.is_file():
        record=np.load(oldmetrics,allow_pickle=False)
        if record.ndim==1 and record.size>=7 and np.isfinite(record[6]) and record[6]>0: params=int(record[6])
    new=False
    signature=json.dumps([csv_digest,code_digest,Dataset_Name,rid,sl,configs[m],freq,'fixed','MSE',int(EPOCHS),int(PATIENCE),int(BATCH_SIZE),float(LR),C,list(cols)],sort_keys=True)
    previous=None
    for marker in sorted(OUT.glob('evaluation_*/work/'+rid+'/paired.json')):
        if marker.read_text()==signature:
            previous=marker.parent; break
    if previous is not None:
        p,t=(np.load(previous/n,mmap_mode='r') for n in ['native_pred.npy','native_true.npy'])
        native=scores((previous/'native.log').read_text(errors='replace')); source='previous validated native training'; params=json.loads((previous/'effective.json').read_text())['params']; row['origin']=str(previous)
    elif native is not None and res and all((res/n).is_file() for n in ['pred.npy','true.npy']):
        p,t=(np.load(res/n,mmap_mode='r') for n in ['pred.npy','true.npy']); source='native files'
    elif not new and m!='iTransformer' and native is not None and oldpred.is_file():
        obj=np.load(oldpred,allow_pickle=True).item(); p,t=obj['preds'],obj['trues']; source='v16 validated against native log'
    elif TRAIN_IF_MISSING or (REPLAY_EXACT_CHECKPOINT and native is not None and ck and ck.is_file()):
        new=not (REPLAY_EXACT_CHECKPOINT and native is not None and ck and ck.is_file())
        print(f"[{'TRAIN-MISSING' if new else 'REPLAY'}] {rid}",flush=True)
        import torch
        work=RUN_DIR/'work'/rid; work.mkdir(parents=True)
        args=dict(task_name='long_term_forecast',is_training=int(new),model=m,model_id=rid,data=Dataset_Name if Dataset_Name.startswith('ETT') else 'custom',
            des=rid,root_path=str(CSV.parent),data_path=CSV.name,features='M',target='OT' if 'OT' in cols else cols[-1],freq=freq,embed='fixed',
            seq_len=sl,pred_len=h,label_len=sl//2,enc_in=C,dec_in=C,c_out=C,d_layers=1,train_epochs=int(EPOCHS),patience=int(PATIENCE),batch_size=int(BATCH_SIZE),
            learning_rate=float(LR),loss='MSE',use_norm=1,seed=s,itr=1,num_workers=0,use_gpu=torch.cuda.is_available(),checkpoints=str(work/'checkpoints'))
        args.update(configs[m]); spec=dict(args=args,checkpoint=None if new else str(ck),setting=setting)
        (work/'spec.json').write_text(json.dumps(spec)); (work/'worker.py').write_text(WORKER)
        with (work/'native.log').open('w') as f:
            proc=subprocess.run([sys.executable,'-u',str(work/'worker.py'),str(REPO),str(work/'spec.json')],cwd=work,stdout=f,stderr=subprocess.STDOUT,timeout=7200)
        if proc.returncode:
            tail=(work/'native.log').read_text(errors='replace')[-2500:]
            raise RuntimeError(f'Native subprocess rc={proc.returncode}: {work}/native.log\n{tail}')
        current=scores((work/'native.log').read_text(errors='replace'))
        if current is None or (not new and not close(current,native)): raise ValueError('Native log missing/ambiguous or checkpoint replay disagrees with old log')
        native=current; params=json.loads((work/'effective.json').read_text())['params']; source='new native run' if new else 'native checkpoint replay'
        p,t=(np.load(work/n,mmap_mode='r') for n in ['native_pred.npy','native_true.npy'])
    else: raise ValueError('No eligible paired source (old native scores, if available, are log-only)')
    row['source']=source
    if p.shape!=t.shape or t.shape!=(B,h,C): raise ValueError(f'Shape mismatch {p.shape}, {t.shape}, expected {(B,h,C)}')
    # Compute primary metrics on exactly the arrays TSLib (or the validated cache)
    # supplied. Independently check scores on the common targets before export.
    sq=np.zeros(C); ab=0.; hits=0; native_sq=native_ab=0.; maxdiff=0.
    for i in range(0,B,64):
        a=p[i:i+64].astype('float64'); b=ref[i:i+64]; y=t[i:i+64]
        if not np.isfinite(a).all() or not np.isfinite(y).all(): raise ValueError('Nonfinite predictions/targets')
        maxdiff=max(maxdiff,float(np.max(np.abs(y-b))))
        if not np.allclose(y,b,rtol=0,atol=1e-6): raise ValueError('Target scale/channel/origin mismatch')
        en=a-y; native_sq+=np.square(en).sum(); native_ab+=np.abs(en).sum()
        e=a-b; sq+=(e*e).sum(axis=(0,1)); ab+=np.abs(e).sum(); hits+=(np.diff(a,axis=1)*np.diff(b,axis=1)>0).sum()
    mse=sq.sum()/p.size; mae=ab/p.size
    if not close([native_sq/p.size,native_ab/p.size],native) or not close([mse,mae],native): raise ValueError('MSE/MAE mismatch vs native log')
    valid_den=np.isfinite(den)&(den>0)
    rmsse=np.sqrt(sq[valid_den]/(B*h)/den[valid_den]).mean() if valid_den.any() else np.nan
    mda=hits/(B*(h-1)*C)
    pf=p.reshape(-1); tf=ref.reshape(-1); sp=stats.spearmanr(pf,tf)[0]; pr=stats.pearsonr(pf,tf)[0]
    rec=np.array([mse,mae,rmsse,mda,sp,pr,np.nan if params is None else params])
    # Save directly to OUT (./results/{Dataset_Name}_baselines/) instead of RUN_DIR.
    np.save(OUT/(rid+'_preds.npy'),dict(preds=np.asarray(p),trues=np.asarray(ref)))
    np.save(OUT/(rid+'.npy'),rec)
    if new: (work/'paired.json').write_text(signature)
    row.update(status='paired',target_max_abs_diff=maxdiff,native_MSE=float(native[0]),native_MAE=float(native[1]),
               params_source=('unavailable' if params is None else 'native model' if source in ('new native run','previous validated native training','native checkpoint replay') else 'legacy metric record'),
               **dict(zip(['MSE','MAE','RMSSE','MDA','Spearman','Pearson','Params'],rec)))

def _get_npy_path(model, horizon, seed):
    return os.path.join(
        final_dir, f"{model}_{Dataset_Name}_{horizon}_seed{seed}.npy")


def _get_preds_path(model, horizon, seed):
    return os.path.join(
        final_dir, f"{model}_{Dataset_Name}_{horizon}_seed{seed}_preds.npy")


print("\n" + "=" * 90)
print("START")
print("=" * 90)
_total = len(models_tslib) * len(RUN_HORIZONS) * len(RUN_SEEDS)
print(f"[INFO] {_total} (model, horizon, seed) entries to process")


all_results = {m: {h: {} for h in RUN_HORIZONS} for m in models_tslib}

rows=[]
for m in models_tslib:
    for h in RUN_HORIZONS:
        ref=np.asarray(load_multivariate_data(Dataset_Name,h,sl,data_root=data_root)[5],dtype=np.float32)
        if ref.ndim!=3 or ref.shape[1:]!=(h,C) or not len(ref) or not np.isfinite(ref).all(): raise ValueError('Invalid reference targets')
        for s in RUN_SEEDS:
            row=dict(model=m,horizon=h,seed=s,status='unavailable'); rows.append(row)
            try:
                run_one(m,int(h),int(s),row,ref)
                all_results[m][h][s]={k:float(row[k]) for k in ('MSE','MAE','RMSSE','MDA','Spearman','Pearson','Params')}
            except Exception as exc:
                row['reason']=str(exc)
                if (RUN_DIR/'work'/f'{m}_{Dataset_Name}_{h}_seed{s}').exists():
                    pd.DataFrame(rows).to_csv(OUT/'runs.csv',index=False)
                    raise RuntimeError(f'Stopped after native run failed: {exc}') from exc
            print(m,h,s,row['status'],row.get('reason',''),flush=True)
            pd.DataFrame(rows).to_csv(OUT/'runs.csv',index=False)
# Final .npy verification
# ============================================================
print("\n" + "=" * 90)
print("FINAL .NPY VERIFICATION")
print("=" * 90)
_missing_final = []
for m in models_tslib:
    for h in RUN_HORIZONS:
        for s in RUN_SEEDS:
            fp = _get_npy_path(m, h, s)
            if not os.path.exists(fp):
                _missing_final.append((m, h, s))


if _missing_final:
    print(f"  WARNING: {len(_missing_final)} missing:")
    by_model = {}
    for m, h, s in _missing_final:
        by_model.setdefault(m, []).append((h, s))
    for m, items in by_model.items():
        print(f"    {m:14s}  missing {len(items)}")
else:
    print(f"  All {_total} .npy present")
print("=" * 90)


# ============================================================
# std=0 diagnostic
# ============================================================
print("\n" + "=" * 90)
print("SEED-VARIANCE DIAGNOSTIC")
print("=" * 90)


_std0 = []; _expected = []
for m in models_tslib:
    is_det = (m in DETERMINISTIC_MODELS)
    print(f"\n{m}" + ("  (deterministic)" if is_det else "  (stochastic)"))
    for h in RUN_HORIZONS:
        recs = all_results[m].get(h, {})
        if not recs:
            print(f"  H={h:4d}  no data"); continue
        mses = [recs[s]['MSE'] for s in sorted(recs.keys())]
        std = float(np.std(mses))
        if std < 1e-8:
            if is_det:
                flag = "  [OK: deterministic]"; _expected.append((m,h))
            else:
                flag = "  [WARN: std=0]"; _std0.append((m,h,len(mses)))
        else: flag = ""
        print(f"  H={h:4d}  mean={np.mean(mses):.4f}  "
              f"std={std:.6f}{flag}")


if _std0: print(f"\nWARNING: {len(_std0)} stochastic entries with std=0")
elif _expected: print("\n[OK] std=0 only in deterministic models.")
else: print("\n[OK] All models show nonzero seed variance.")


json_path = os.path.join(final_dir,
                         f'all_models_{Dataset_Name}_results.json')
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"[JSON] {json_path}")


# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 90)
print(f"FULL METRICS LOG - {Dataset_Name}")
print("=" * 90)


log_data = {}
for m in models_tslib:
    print(f"\n{m}"); print("-" * 80)
    log_data[m] = {}
    for h in RUN_HORIZONS:
        recs = list(all_results[m].get(h, {}).values())
        if not recs:
            print(f"H={h:4d} | No data"); continue
        def arr(k): return np.array([r.get(k, np.nan) for r in recs])
        mse, mae   = arr('MSE'), arr('MAE')
        rmsse, mda = arr('RMSSE'), arr('MDA')
        sp, pr     = arr('Spearman'), arr('Pearson')
        pm, tm, gp = arr('Params'), arr('TrainTime_sec'), arr('PeakGPU_GB')
        print(f"H={h:4d} | Params={int(pm.mean()) if np.isfinite(pm.mean()) else -1:>10,} | "
              f"MSE={mse.mean():.4f}+/-{mse.std():.4f}  "
              f"MAE={mae.mean():.4f}+/-{mae.std():.4f}  "
              f"RMSSE={rmsse.mean():.4f}+/-{rmsse.std():.4f}  "
              f"MDA={mda.mean():.4f}+/-{mda.std():.4f}")
        log_data[m][h] = {
            'params': int(pm.mean()) if np.isfinite(pm.mean()) else -1,
            'MSE':   f"{mse.mean():.4f}+/-{mse.std():.4f}",
            'MAE':   f"{mae.mean():.4f}+/-{mae.std():.4f}",
            'RMSSE': f"{rmsse.mean():.4f}+/-{rmsse.std():.4f}",
            'MDA':   f"{mda.mean():.4f}+/-{mda.std():.4f}",
            'Spearman': f"{sp.mean():.4f}+/-{sp.std():.4f}",
            'Pearson':  f"{pr.mean():.4f}+/-{pr.std():.4f}",
            'TrainTime_sec': f"{(np.nanmean(tm) if np.isfinite(tm).any() else np.nan):.1f}+/-{(np.nanstd(tm) if np.isfinite(tm).any() else np.nan):.1f}",
            'PeakGPU_GB':    f"{(np.nanmean(gp) if np.isfinite(gp).any() else np.nan):.2f}+/-{(np.nanstd(gp) if np.isfinite(gp).any() else np.nan):.2f}",
        }


log_file = os.path.join(final_dir,
                        f'{Dataset_Name}_full_metrics_log.json')
with open(log_file, 'w') as f:
    json.dump(log_data, f, indent=2)
print(f"\n[LOG] {log_file}")
print(f"[ALL] {final_dir}")

# ============================================================
# Cleanup temporary evaluation directory
# ============================================================
gc.collect()
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    print(f"[CLEANUP] Removed temporary evaluation directory: {RUN_DIR}")

print("\n" + "=" * 90)
print("BASELINE CELL COMPLETE")
print("=" * 90)
print(f"  Models: {models_tslib}")
print(f"  Horizons: {RUN_HORIZONS}")
print(f"  Seeds: {RUN_SEEDS}")
print(f"  .npy complete: {len(_missing_final) == 0}")
if _missing_final:
    print(f"  WARNING: {len(_missing_final)} missing")
print("=" * 90)

Output (final_dir): /mnt/d/Project/PivotFormer/results/ETTh2_baselines
Temp eval dir     : /mnt/d/Project/PivotFormer/results/ETTh2_baselines/evaluation_w8jtegu4
[INFO] CUDA: True
[INFO] GPU : NVIDIA GeForce RTX 3090
[INFO] Available: ['PatchTST', 'DLinear', 'Crossformer', 'iTransformer', 'TimesNet', 'TimeMixer']
[INFO] Will run: ['PatchTST', 'DLinear', 'Crossformer', 'iTransformer', 'TimesNet', 'TimeMixer']
[INFO] Dataset=ETTh2 freq=h
[INFO] 7 channels, target=OT
[INFO] RMSSE denom=0.361395

START
[INFO] 144 (model, horizon, seed) entries to process
[TRAIN-MISSING] PatchTST_ETTh2_96_seed0
PatchTST 96 0 paired 
[TRAIN-MISSING] PatchTST_ETTh2_96_seed1
PatchTST 96 1 paired 
[TRAIN-MISSING] PatchTST_ETTh2_96_seed2
PatchTST 96 2 paired 
[TRAIN-MISSING] PatchTST_ETTh2_96_seed3
PatchTST 96 3 paired 
[TRAIN-MISSING] PatchTST_ETTh2_96_seed4
PatchTST 96 4 paired 
[TRAIN-MISSING] PatchTST_ETTh2_96_seed5
PatchTST 96 5 paired 
[TRAIN-MISSING] PatchTST_ETTh2_192_seed0
PatchTST 192 0 paired 
[TRAIN-

# PivotFormer

In [ ]:
# =============================================================================
# RUBICKFORMER - Channel-Independent Multi-Input Multi-Output
# -----------------------------------------------------------------------------
# v7 - Q1 ABLATION SUITE:
#   1. Combined Loss: alpha * MSE + (1 - alpha) * MAE
#   2. Learnable Attention Temperature (tau) for tanh logit de-ashing
#   3. Gated Residual Bypass around Conv1d (outlier / high-freq preservation)
#   4. Patching vs. No-Patching + Full Component Ablation Runner & Table
# =============================================================================

import sys, subprocess
try:
    import docx  # noqa: F401
except ImportError:
    print("[SETUP] Installing python-docx ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"],
                   check=False)
    print("[SETUP] python-docx installed.")

import os
import math
import json
import random
import time
import gc
import inspect
import traceback
import warnings
import numpy as np
import pandas as pd
from scipy import stats

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, TensorDataset

# =============================================================================
# 0. GUARD: required globals
# =============================================================================
_REQUIRED_GLOBALS = [
    'Dataset_Name', 'LOOKBACK', 'HORIZONS', 'SEEDS',
    'EPOCHS', 'BATCH_SIZE', 'LR', 'DROPOUT', 'D_MODEL', 'PATIENCE',
    'load_multivariate_data', 'compute_train_seasonal_naive_mse_per_channel',
]
_missing = [g for g in _REQUIRED_GLOBALS if g not in globals()]
if _missing:
    raise RuntimeError(
        f"Missing required globals: {_missing}.\n"
        f"Make sure the config cell and the data loading cell "
        f"have been executed BEFORE this cell."
    )

# =============================================================================
# 0b. GUARD: verify load_multivariate_data is the TSLib-compatible version
# =============================================================================
try:
    _ldm_src = inspect.getsource(load_multivariate_data)
except Exception as e:
    raise RuntimeError(
        f"Could not inspect load_multivariate_data: {e}") from e

if '_compute_tslib_borders' not in _ldm_src:
    raise RuntimeError(
        "\n" + "!" * 72 + "\n"
        "load_multivariate_data is NOT the TSLib-compatible version.\n"
        "The Config Cell has likely overwritten it with the simplified split\n"
        "(no seq_len overlap at validation/test boundaries). This would make\n"
        "the comparison with PatchTST / Autoformer / etc. unfair.\n"
        "\n"
        "Fix: re-run the Data Cell (unified data loading cell) BEFORE this\n"
        "     training cell, so that the TSLib-compatible version is active.\n"
        + "!" * 72
    )
print("[DATA] load_multivariate_data is TSLib-compatible (seq_len overlap).")

# =============================================================================
# 1. CONFIG & ABLATION FLAGS
# =============================================================================
PATCH_LEN = 8
STRIDE    = 4
N_HEADS   = 4
E_LAYERS  = 3
CONV_SIZE = 7
GRAD_CLIP = 1.0

# Architecture version — bumped for v7 capabilities
ARCH_VERSION = 'v7_q1_full'

# Normalization applied after softmax, before output_scale.
NORM_TYPE = 'rmsnorm'  # or 'layernorm'

# Defaults used by capacity-matched (_cm) variants.
CM_D_MODEL = 32
CM_N_HEADS = 4

# ---- NEW v7 Architectural & Loss Hyperparameters (Default Flags) ----
USE_LEARNABLE_TAU = True     # Ablation 2: Learnable temperature (tau) on attention
TAU_INIT          = 0.5      # Initial temperature (<1.0 sharpens tanh logits)
TAU_MIN           = 0.05     # Lower clamp bound for numerical stability

USE_CONV_RESIDUAL = True     # Ablation 3: Gated residual bypass around Conv1d
LOSS_ALPHA        = 0.8      # Ablation 1: alpha*MSE + (1-alpha)*MAE (1.0 = pure MSE)

# =============================================================================
# [ABLATION SUITE] Config
# -----------------------------------------------------------------------------
# ABLATION_PATCHING = True  -> run patch vs no-patch ablation
# ABLATION_Q1_SUITE = True  -> run complete 4-part Q1 ablation table
# =============================================================================
ABLATION_PATCHING = False      # set True to run Patch vs No-Patch only
ABLATION_Q1_SUITE = False      # set True to run all 4 Q1 ablations + table
ABLATION_HORIZONS = [96]       # horizons to use in the ablation
ABLATION_SEEDS    = [0]        # seeds to use in the ablation

# The two configs used by the patching ablation
_ABL_PATCH_CFG = {
    'variant_name': 'full_cm',
    'patch_len': 8,
    'stride': 4,
    'arch_version': 'v7_patch_postsoftmax_norm_rms',
}
_ABL_NOPATCH_CFG = {
    'variant_name': 'full_cm_nopatch',
    'patch_len': 1,
    'stride': 1,
    'arch_version': 'v7_nopatch_postsoftmax_norm_rms',
}

def _fit_patching(input_len, patch_len, stride, conv_size, min_patches=None):
    """RubickFormer patching helper.

    [PATCHING-ABLATION] Allows patch_len=1.
    With patch_len=1 and stride=1, each timestep becomes one token and
    patch_embedding reduces to Linear(1, d_model).
    """
    pl = max(1, int(patch_len))
    st = max(1, int(stride))
    if pl > input_len:
        pl = input_len
    if st > pl:
        st = pl
    while True:
        n_patches = (input_len - pl) // st + 1
        if n_patches < conv_size and pl > 1:
            pl = max(1, pl // 2)
            st = max(1, min(st, pl))
            continue
        break
    n_patches = (input_len - pl) // st + 1
    cs = min(int(conv_size), n_patches)
    return pl, st, cs

def _parse_variant(model_variant: str):
    """Parse variant name into independent flags."""
    use_revin         = 'no_revin' not in model_variant
    capacity_matched  = '_cm' in model_variant
    use_tslib_padding = '_tslib' in model_variant
    has_stable        = '_stable' in model_variant

    if has_stable:
        use_tslib_training = False
    elif use_tslib_padding:
        use_tslib_training = True
    else:
        use_tslib_training = False

    return use_revin, use_tslib_padding, use_tslib_training, capacity_matched

def _tslib_type1_lr(initial_lr, epoch):
    """Match TSLib's lradj='type1' exactly (1-indexed epoch)."""
    return initial_lr * (0.5 ** max(0, epoch - 2))

def set_seed(seed: int = 0):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _cache_path(final_dir, dataset_name, model_variant, horizon, seed):
    # Use a stable public filename; isolate variants to avoid overwriting baselines.
    directory = (final_dir if model_variant == 'full_cm'
                 else os.path.join(final_dir, model_variant))
    return os.path.join(directory, f'{dataset_name}_full_cm_H{horizon}_seed{seed}.npy')


def _save_npy_dict(cache_path: str, payload: dict) -> str:
    cache_path = os.path.abspath(cache_path)
    parent = os.path.dirname(cache_path) or '.'
    try:
        os.makedirs(parent, exist_ok=True)
    except Exception as e:
        raise RuntimeError(
            f"Could not create cache directory '{parent}': {e}") from e
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
            np.save(cache_path, payload, allow_pickle=True)
    except Exception:
        print(f"  !!! np.save FAILED for {cache_path}")
        traceback.print_exc()
        raise
    if not os.path.exists(cache_path):
        raise RuntimeError(
            f"np.save returned without error but {cache_path} does not exist.")
    size = os.path.getsize(cache_path)
    print(f"  Saved -> {cache_path}  ({size:,} bytes)")
    return cache_path

def _load_npy_dict(cache_path: str) -> dict:
    obj = np.load(cache_path, allow_pickle=True)
    if isinstance(obj, np.ndarray) and obj.dtype == object and obj.shape == ():
        return obj.item()
    return obj.item() if hasattr(obj, 'item') else obj

# =============================================================================
# 1b. [ABLATION 1] COMBINED LOSS (alpha * MSE + (1 - alpha) * MAE)
# =============================================================================
class CombinedLoss(nn.Module):
    """Convex combination of MSE and MAE loss:
       L = alpha * MSE(pred, target) + (1 - alpha) * MAE(pred, target)
    When alpha == 1.0, reduces strictly to standard nn.MSELoss().
    """
    def __init__(self, alpha: float = 0.8):
        super().__init__()
        if not (0.0 <= alpha <= 1.0):
            raise ValueError(f"alpha must be in [0.0, 1.0], got {alpha}")
        self.alpha = float(alpha)

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if self.alpha >= 1.0 - 1e-8:
            return F.mse_loss(pred, target)
        if self.alpha <= 1e-8:
            return F.l1_loss(pred, target)
        mse = F.mse_loss(pred, target)
        mae = F.l1_loss(pred, target)
        return self.alpha * mse + (1.0 - self.alpha) * mae

# =============================================================================
# 2. RevIN
# =============================================================================
class RevIN(nn.Module):
    def __init__(self, num_features: int = 1, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.gamma = nn.Parameter(torch.ones(num_features))
            self.beta  = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('gamma', None)
            self.register_parameter('beta',  None)
        self.mean  = None
        self.stdev = None

    def forward(self, x, reverse: bool = False):
        if reverse:
            if self.mean is None or self.stdev is None:
                raise RuntimeError("RevIN.forward(reverse=False) must be called first.")
            if self.affine:
                x = (x - self.beta.view(1, 1, -1)) / (
                    self.gamma.view(1, 1, -1) + self.eps)
            return x * self.stdev + self.mean
        else:
            self.mean  = x.mean(dim=1, keepdim=True).detach()
            self.stdev = torch.sqrt(
                x.var(dim=1, keepdim=True, unbiased=False) + self.eps).detach()
            x = (x - self.mean) / self.stdev
            if self.affine:
                x = x * self.gamma.view(1, 1, -1) + self.beta.view(1, 1, -1)
            return x

# =============================================================================
# 2b. RMSNorm
# =============================================================================
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, NeurIPS 2019)."""
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

# =============================================================================
# 3. RUBICKATTENTION BLOCK (with Learnable Temperature tau)
# =============================================================================
class RubickAttentionBlock(nn.Module):
    def __init__(self, n_heads, d_model, multi_layer=True, use_residual=True,
                 use_scaling=True, soft_max_axis=-1, dropout=0.1,
                 use_learnable_tau=False, tau_init=0.5, tau_min=0.05,
                 head_channels=None, selective=False):
        super().__init__()
        self.n_heads           = n_heads
        self.d_model           = d_model
        self.multi_layer       = multi_layer
        self.use_residual      = use_residual
        self.use_scaling       = use_scaling
        self.soft_max_axis     = soft_max_axis
        self.use_learnable_tau = use_learnable_tau
        self.tau_min           = tau_min

        self.W = nn.Parameter(torch.empty(d_model, d_model))
        self.b = nn.Parameter(torch.zeros(d_model))
        nn.init.xavier_uniform_(self.W)

        self.c = nn.Parameter(torch.empty(n_heads, d_model, d_model))
        nn.init.xavier_uniform_(self.c)

        # [ABLATION 2] Learnable per-head temperature tau to counteract tanh saturation
        if self.use_learnable_tau:
            self.temperature = nn.Parameter(torch.full((1, n_heads, 1, 1), float(tau_init)))
        else:
            self.register_parameter('temperature', None)

        if self.multi_layer and self.n_heads > 1:
            self.head_weights = nn.Parameter(torch.ones(n_heads))
        else:
            self.register_parameter('head_weights', None)

        self.register_parameter('channel_strength', None)
        self.register_parameter('head_profile', None)
        if selective and head_channels is not None and self.head_weights is not None:
            self.channel_strength = nn.Parameter(torch.zeros(head_channels))
            self.head_profile = nn.Parameter(torch.linspace(-1., 1., n_heads))
        if not selective and head_channels is not None and self.head_weights is not None:
            self.channel_delta = nn.Parameter(torch.zeros(head_channels, n_heads))
        else:
            self.register_parameter('channel_delta', None)

        if not self.multi_layer and self.n_heads > 1:
            self.output_projection = nn.Linear(self.n_heads * d_model, d_model)
        else:
            self.output_projection = None

        if self.use_residual and not self.multi_layer:
            self.layer_norm = nn.LayerNorm(d_model, eps=1e-6)
        else:
            self.layer_norm = None

        if self.multi_layer:
            if NORM_TYPE == 'rmsnorm':
                self.post_softmax_norm = RMSNorm(
                    n_heads * d_model, eps=1e-6)
            else:
                self.post_softmax_norm = nn.LayerNorm(
                    n_heads * d_model, eps=1e-6)
            self.output_scale = nn.Parameter(torch.ones(1))
        else:
            self.post_softmax_norm = None
            self.register_parameter('output_scale', None)

        self.dropout = nn.Dropout(dropout)

    def channel_gain(self):
        if self.channel_strength is not None:
            return 1.0 + self.channel_strength.tanh()[:, None] * self.head_profile.tanh()[None, :]
        if self.channel_delta is not None:
            return 1.0 + 0.2 * self.channel_delta.tanh()
        return None

    def forward(self, x):
        B, L, F = x.shape
        H = self.n_heads
        c_it = torch.tanh(torch.matmul(x, self.W) + self.b)
        a_it = torch.einsum('bld,hfd->bhld', c_it, self.c)
        if self.use_scaling:
            a_it = a_it / math.sqrt(F)

        # [ABLATION 2] Apply learnable temperature before softmax
        if self.use_learnable_tau and self.temperature is not None:
            tau = self.temperature.clamp(min=self.tau_min)
            a_it = a_it / tau

        a_it_soft = torch.softmax(a_it, dim=-1)
        a_it_soft = self.dropout(a_it_soft)
        x_expanded = x.unsqueeze(1)
        weighted = x_expanded * a_it_soft
        if self.head_weights is not None:
            gain = self.channel_gain()
            if gain is None:
                weighted = weighted * self.head_weights.view(1, H, 1, 1)
            else:
                # B here is batch_size * C, with channel varying fastest.
                C = gain.shape[0]
                hw = self.head_weights * gain  # [C, H]
                weighted = (weighted.reshape(-1, C, H, L, F)
                            * hw[None, :, :, None, None]).reshape(B, H, L, F)
        weighted = weighted.permute(0, 2, 1, 3).contiguous()
        concat = weighted.view(B, L, H * F)
        if self.multi_layer:
            out = torch.softmax(concat, dim=self.soft_max_axis)
            out = self.post_softmax_norm(out)
            return out * self.output_scale
        else:
            if self.output_projection is not None:
                out = self.output_projection(concat)
                out = torch.relu(out)
            else:
                out = concat
            if self.use_residual:
                out = out + x
            if self.layer_norm is not None:
                out = self.layer_norm(out)
            return out

class RubickEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, soft_max_axis, dropout=0.1,
                 use_scaling=True, use_learnable_tau=False,
                 tau_init=0.5, tau_min=0.05, head_channels=None, selective=False):
        super().__init__()
        self.attention = RubickAttentionBlock(
            n_heads=n_heads, d_model=d_model, multi_layer=True,
            use_residual=False, use_scaling=use_scaling,
            soft_max_axis=soft_max_axis, dropout=dropout,
            use_learnable_tau=use_learnable_tau,
            tau_init=tau_init, tau_min=tau_min, head_channels=head_channels, selective=selective
        )
        self.proj    = nn.Linear(n_heads * d_model, d_model)
        self.norm    = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.attention(x)
        x = self.proj(x)
        x = self.dropout(x)
        x = x + residual
        x = self.norm(x)
        return x

# =============================================================================
# 4. PER-CHANNEL ACTIVATION
# =============================================================================
class PerChannelActivation(nn.Module):
    def __init__(self, num_channels, activation_type='pelu'):
        super().__init__()
        self.num_channels    = num_channels
        self.activation_type = activation_type
        if activation_type == 'pelu':
            self.alpha = nn.Parameter(torch.ones(num_channels))
            self.beta  = nn.Parameter(torch.ones(num_channels))
        else:
            self.alpha = None
            self.beta  = None

    def forward(self, x):
        if self.activation_type == 'pelu' and self.alpha is not None:
            alpha = self.alpha.view(1, -1, 1)
            beta  = self.beta.view(1, -1, 1)
            return torch.where(x >= 0, beta * x, alpha * (torch.exp(x) - 1))
        return x

# =============================================================================
# 5. RUBICKFORMER (with Conv1d Residual Bypass & Learnable Tau support)
# =============================================================================
class RubickFormer(nn.Module):
    def __init__(self, input_len=336, pred_len=96, patch_len=16, stride=8,
                 d_model=64, n_heads=8, e_layers=3, dropout=0.1,
                 num_features=7, use_learnable_pe=True, conv_size=7,
                 use_revin=True, use_tslib_padding=False,
                 use_learnable_tau=False, tau_init=0.5, tau_min=0.05,
                 use_conv_residual=False, use_channel_heads='selective'):
        super().__init__()

        self.use_tslib_padding = use_tslib_padding
        self.use_learnable_tau = use_learnable_tau
        self.use_channel_heads = use_channel_heads
        self.use_conv_residual = use_conv_residual

        if use_tslib_padding:
            padded_len       = input_len + stride
            self.num_patches = (padded_len - patch_len) // stride + 1
            self.conv_size   = min(int(conv_size), self.num_patches)
        else:
            patch_len, stride, conv_size = _fit_patching(
                input_len, patch_len, stride, conv_size)
            self.num_patches = (input_len - patch_len) // stride + 1
            self.conv_size   = int(conv_size)

        self.input_len        = input_len
        self.pred_len         = pred_len
        self.patch_len        = patch_len
        self.stride           = stride
        self.d_model          = d_model
        self.num_features     = num_features
        self.use_learnable_pe = use_learnable_pe
        self.use_revin        = use_revin

        if use_revin:
            self.revin = RevIN(num_features=1, eps=1e-5, affine=True)
        else:
            self.revin = None

        self.patch_embedding = nn.Linear(patch_len, d_model)

        if use_learnable_pe:
            self.pos_embed = nn.Parameter(
                torch.randn(1, self.num_patches, d_model) * 0.02)
        else:
            pe       = torch.zeros(self.num_patches, d_model)
            position = torch.arange(self.num_patches).unsqueeze(1).float()
            div      = torch.exp(torch.arange(0, d_model, 2).float()
                                 * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div)
            pe[:, 1::2] = torch.cos(position * div)
            self.register_buffer('pe', pe.unsqueeze(0))

        self.dropout_layer = nn.Dropout(dropout)

        layers = []
        for i in range(e_layers):
            axis = -2 if i % 2 == 0 else -1
            layers.append(RubickEncoderLayer(
                d_model, n_heads, soft_max_axis=axis,
                dropout=dropout, use_scaling=True,
                use_learnable_tau=use_learnable_tau,
                tau_init=tau_init, tau_min=tau_min,
                head_channels=num_features if use_channel_heads else None,
                selective=(use_channel_heads == 'selective')))
        self.encoder = nn.Sequential(*layers)

        self.head_linear = nn.Linear(d_model, d_model)
        self.head_drop   = nn.Dropout(dropout)
        self.head_pelu   = PerChannelActivation(d_model, activation_type='pelu')

        L_after_conv = self.num_patches - self.conv_size + 1
        if L_after_conv < 1:
            raise ValueError(
                f"num_patches={self.num_patches} must be >= "
                f"conv_size={self.conv_size}")
        self.head_conv  = nn.Conv1d(in_channels=d_model,
                                    out_channels=pred_len,
                                    kernel_size=self.conv_size, padding=0)
        self.head_final = nn.Linear(L_after_conv, 1)

        # [ABLATION 3] Gated Residual Bypass around Conv1d to preserve high-frequency spikes
        if self.use_conv_residual:
            # Maps x_pelu of shape (B*C, d_model, num_patches) -> (B*C, pred_len, 1)
            self.conv_skip_proj = nn.Linear(d_model * self.num_patches, pred_len)
            # Initialized to logit(0.8) ~= 1.386 so model starts near original Conv1d path
            self.conv_gate_logit = nn.Parameter(torch.tensor(1.38629436))
        else:
            self.conv_skip_proj  = None
            self.register_parameter('conv_gate_logit', None)

    def forward(self, x):
        B, T, C = x.shape
        if self.use_channel_heads and C != self.num_features:
            raise ValueError(f'Expected {self.num_features} input channels in training order, got {C}')
        x = x.permute(0, 2, 1).reshape(B * C, T, 1)

        if self.use_revin:
            x = self.revin(x, reverse=False)

        x = x.squeeze(-1)

        if self.use_tslib_padding:
            x = F.pad(x, (0, self.stride), mode='replicate')

        x = x.unfold(1, self.patch_len, self.stride)
        x = self.patch_embedding(x)

        if self.use_learnable_pe:
            x = x + self.pos_embed
        else:
            x = x + self.pe[:, :self.num_patches, :]

        x = self.dropout_layer(x)
        x = self.encoder(x)

        x = self.head_linear(x)
        x = self.head_drop(x)
        x = x.permute(0, 2, 1)          # (B*C, d_model, num_patches)
        x_pelu = self.head_pelu(x)      # (B*C, d_model, num_patches)

        # Main Conv1d branch
        conv_out = self.head_conv(x_pelu)       # (B*C, pred_len, L_after_conv)
        conv_out = self.head_final(conv_out)    # (B*C, pred_len, 1)

        # [ABLATION 3] Gated Residual Bypass branch
        if self.use_conv_residual and self.conv_skip_proj is not None:
            skip_in  = x_pelu.reshape(B * C, -1)                    # (B*C, d_model * num_patches)
            skip_out = self.conv_skip_proj(skip_in).unsqueeze(-1)   # (B*C, pred_len, 1)
            lam      = torch.sigmoid(self.conv_gate_logit)
            x        = lam * conv_out + (1.0 - lam) * skip_out
        else:
            x = conv_out

        if self.use_revin:
            x = self.revin(x, reverse=True)

        x = x.reshape(B, C, self.pred_len, 1).squeeze(-1)
        return x.permute(0, 2, 1)

# =============================================================================
# 6. EVALUATION METRICS
# =============================================================================
def compute_metrics(preds, trues, train_naive_mse_per_channel=None):
    mse = float(np.mean((preds - trues) ** 2))
    mae = float(np.mean(np.abs(preds - trues)))

    if train_naive_mse_per_channel is not None:
        test_mse_per_channel = np.mean((preds - trues) ** 2, axis=(0, 1))
        safe = np.where(np.asarray(train_naive_mse_per_channel) > 0,
                        train_naive_mse_per_channel, np.nan)
        rmsse = float(np.nanmean(np.sqrt(test_mse_per_channel / safe)))
    else:
        rmsse = float(np.sqrt(mse))

    d_pred = np.diff(preds, axis=1)
    d_true = np.diff(trues, axis=1)
    mda    = float(np.mean((d_pred * d_true) > 0))

    pf, tf = preds.flatten(), trues.flatten()
    try:    spearman = float(stats.spearmanr(pf, tf)[0])
    except Exception: spearman = 0.0
    try:    pearson  = float(stats.pearsonr(pf, tf)[0])
    except Exception: pearson  = 0.0

    return {
        'main': {'MSE': mse, 'MAE': mae},
        'supplementary': {'RMSSE': rmsse, 'MDA': mda,
                          'Spearman': spearman, 'Pearson': pearson},
        'MSE': mse, 'MAE': mae,
        'RMSSE': rmsse, 'MDA': mda,
        'Spearman': spearman, 'Pearson': pearson,
    }

# =============================================================================
# 7. PLOTTING
# =============================================================================
def plot_train_val_losses(train_losses_by_seed, val_losses_by_seed,
                          horizon, dataset_name, model_variant, out_dir):
    if (not any(val_losses_by_seed.values())
            and not any(train_losses_by_seed.values())):
        return

    plt.figure(figsize=(11, 6))
    cmap  = plt.get_cmap('tab10')
    seeds = sorted(set(val_losses_by_seed.keys())
                   | set(train_losses_by_seed.keys()))
    for idx, seed in enumerate(seeds):
        color     = cmap(idx % 10)
        tr_losses = train_losses_by_seed.get(seed, [])
        va_losses = val_losses_by_seed.get(seed, [])
        if tr_losses:
            plt.plot(range(1, len(tr_losses) + 1), tr_losses,
                     linestyle='-', color=color, alpha=0.85, linewidth=1.5,
                     label=f'seed={seed} train')
        if va_losses:
            plt.plot(range(1, len(va_losses) + 1), va_losses,
                     linestyle='--', color=color, alpha=0.85, linewidth=1.5,
                     label=f'seed={seed} val')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(f'{dataset_name} | H={horizon} | {model_variant}\n'
              f'Train (solid) vs Validation (dashed) Loss across seeds')
    plt.legend(loc='best', ncol=2, fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plot_path = os.path.abspath(os.path.join(
        out_dir or '.',
        f"{dataset_name}_{model_variant}_H{horizon}_train_val_loss.png"))
    os.makedirs(os.path.dirname(plot_path) or '.', exist_ok=True)
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"\n  Plot saved -> {plot_path}")

# =============================================================================
# 8. TRAINING PIPELINE (with full Q1 ablation overrides)
# =============================================================================
def train_rubickformer(dataset_name, data_root, final_dir,
                       device='cuda', model_variant='full_cm',
                       epochs=EPOCHS, batch_size=BATCH_SIZE,
                       lr=LR, patience=PATIENCE,
                       d_model_override=None,
                       n_heads_override=None,
                       patch_len_override=None,
                       stride_override=None,
                       horizons_override=None,
                       seeds_override=None,
                       arch_version_override=None,
                       loss_alpha_override=None,
                       use_learnable_tau_override=None,
                       tau_init_override=None,
                       use_conv_residual_override=None):
    """Train RubickFormer with full support for all 4 Q1 ablations."""

    # ---- resolve locals (ablation overrides win over globals) ----
    _PL   = patch_len_override if patch_len_override is not None else PATCH_LEN
    _ST   = stride_override    if stride_override    is not None else STRIDE
    _AV   = arch_version_override if arch_version_override is not None else ARCH_VERSION
    if model_variant == 'full_cm_selhead':  # Legacy alias for the final architecture.
        model_variant = 'full_cm'
    _CHHEAD = 'chhead' in model_variant.split('_')
    if ('selhead' in model_variant.split('_') or
            (not model_variant.startswith('abl_') and not _CHHEAD)):
        _CHHEAD = 'selective'
        _AV += '_selhead_rank1_v1'
    elif _CHHEAD:
        _AV += '_chhead_v1_r020'
    _HZ   = list(horizons_override) if horizons_override is not None else list(HORIZONS)
    _SD   = list(seeds_override)    if seeds_override    is not None else list(SEEDS)

    _L_ALPHA = loss_alpha_override if loss_alpha_override is not None else LOSS_ALPHA
    _USE_TAU = use_learnable_tau_override if use_learnable_tau_override is not None else USE_LEARNABLE_TAU
    _TAU_INI = tau_init_override if tau_init_override is not None else TAU_INIT
    _USE_RES = use_conv_residual_override if use_conv_residual_override is not None else USE_CONV_RESIDUAL

    final_dir = os.path.abspath(final_dir or '.')
    os.makedirs(final_dir, exist_ok=True)

    print(f"[PATHS] cwd       : {os.getcwd()}")
    print(f"[PATHS] final_dir : {final_dir}")
    print(f"[ARCH] ARCH_VERSION = {_AV} | NORM_TYPE = {NORM_TYPE}")
    print(f"[ABLATION CONFIG] patch_len={_PL}, stride={_ST}, "
          f"loss_alpha={_L_ALPHA}, learnable_tau={_USE_TAU} (init={_TAU_INI}), "
          f"conv_residual={_USE_RES}")
    print(f"[RUN SCOPE] horizons={_HZ}, seeds={_SD}")

    dev = torch.device(
        'cuda' if (device == 'cuda' and torch.cuda.is_available()) else 'cpu')
    print(f"Device: {dev}")

    (use_revin, use_tslib_padding,
     use_tslib_training, capacity_matched) = _parse_variant(model_variant)

    if capacity_matched:
        if d_model_override is None:
            d_model_override = CM_D_MODEL
        if n_heads_override is None:
            n_heads_override = CM_N_HEADS

    effective_d_model = d_model_override if d_model_override is not None else D_MODEL
    effective_n_heads = n_heads_override if n_heads_override is not None else N_HEADS

    if use_tslib_padding:
        _padded_len = LOOKBACK + _ST
        _preview_patches = (_padded_len - _PL) // _ST + 1
        _preview_desc = "TSLib padding (PatchTST-style)"
    else:
        _pl_fit, _st_fit, _cs_fit = _fit_patching(LOOKBACK, _PL, _ST, CONV_SIZE)
        _preview_patches = (LOOKBACK - _pl_fit) // _st_fit + 1
        _preview_desc = "Rubick padding (_fit_patching)"

    if _PL == 1 and _ST == 1:
        print(f"  [NO-PATCH MODE] {_preview_patches} tokens "
              f"(one per timestep), patch_embedding = Linear(1, d_model)")
    elif _PL == 1 and _ST != 1:
        print(f"  [WARN] PATCH_LEN=1 with STRIDE={_ST} will SKIP timesteps. "
              f"Set STRIDE=1 for true no-patch mode.")
        print(f"  patch preview = {_preview_patches} tokens via {_preview_desc}")
    else:
        print(f"  patch preview = {_preview_patches} patches via {_preview_desc}")

    print(f"[VARIANT] {model_variant}")
    print(f"  use_revin          = {use_revin}")
    print(f"  use_tslib_padding  = {use_tslib_padding}")
    print(f"  use_tslib_training = {use_tslib_training}")
    print(f"  capacity_matched   = {capacity_matched}")
    print(f"  effective d_model  = {effective_d_model}")
    print(f"  effective n_heads  = {effective_n_heads}")
    print(f"  post_softmax_norm  = {NORM_TYPE}")
    print(f"  loss_alpha         = {_L_ALPHA} ({_L_ALPHA:.2f}*MSE + {1.0-_L_ALPHA:.2f}*MAE)")
    print(f"  use_learnable_tau  = {_USE_TAU}")
    print(f"  use_conv_residual  = {_USE_RES}")

    if use_tslib_training:
        print(f"[TRAINING CONFIG] Adam (no wd) + lradj='type1' + no grad clip")
        print(f"  WARNING: TSLib training was observed UNSTABLE for RubickAttention.")
    else:
        print(f"[TRAINING CONFIG] AdamW (wd=1e-4) + Cosine + "
              f"grad_clip={GRAD_CLIP}  (stable)")

    if 'SEASONAL_MAP' in globals() and dataset_name in SEASONAL_MAP:
        seasonal_period = SEASONAL_MAP[dataset_name]
    elif dataset_name.startswith('ETTh'):
        seasonal_period = 24
    elif dataset_name.startswith('ETTm'):
        seasonal_period = 96
    else:
        seasonal_period = 24
    print(f"[SEASONAL] m={seasonal_period}")

    train_naive_mse_per_channel = compute_train_seasonal_naive_mse_per_channel(
        dataset_name, data_root, seasonal_period=seasonal_period)

    print(f"[RMSSE denom] seasonal-naive MSE per channel (train, m={seasonal_period}):")
    for i, v in enumerate(train_naive_mse_per_channel):
        print(f"    ch{i}: {v:.6f}")
    print(f"    mean: {train_naive_mse_per_channel.mean():.6f}")

    all_results = {}
    expected_npy_paths = []

    for horizon in _HZ:
        all_results[horizon] = {}
        val_losses_by_seed   = {}
        train_losses_by_seed = {}

        for seed in _SD:
            cache_path = _cache_path(final_dir, dataset_name, model_variant, horizon, seed)
            expected_npy_paths.append(cache_path)

            if os.path.exists(cache_path):
                try:
                    cached = _load_npy_dict(cache_path)

                    cached_arch = cached.get('variant_meta', {}).get(
                        'arch_version', 'unknown')
                    if cached_arch != _AV:
                        print(f"\n[CACHE INVALIDATED] "
                              f"{os.path.basename(cache_path)}: "
                              f"arch_version='{cached_arch}' != "
                              f"'{_AV}' -> retraining")
                    else:
                        all_results[horizon][seed] = cached['metrics']
                        val_losses_by_seed[seed]   = cached.get('val_losses', [])
                        train_losses_by_seed[seed] = cached.get('train_losses', [])
                        print(f"\n{'='*60}\n"
                              f"[CACHE HIT] {os.path.abspath(cache_path)}")
                        m      = cached['metrics']
                        m_main = m.get('main', {'MSE': m.get('MSE', 0),
                                                'MAE': m.get('MAE', 0)})
                        m_supp = m.get('supplementary',
                                       {'RMSSE': m.get('RMSSE', 0),
                                        'MDA': m.get('MDA', 0),
                                        'Spearman': m.get('Spearman', 0),
                                        'Pearson': m.get('Pearson', 0)})
                        print(f"  [MAIN]  MSE={m_main['MSE']:.4f}  "
                              f"MAE={m_main['MAE']:.4f}")
                        print(f"  [SUPP]  RMSSE={m_supp['RMSSE']:.4f}  "
                              f"MDA={m_supp['MDA']:.4f}  "
                              f"Spearman={m_supp['Spearman']:.4f}  "
                              f"Pearson={m_supp['Pearson']:.4f}")
                        print(f"  [SKIP SAVE] cached file reused.")
                        continue
                except Exception as e:
                    print(f"\n[CACHE LOAD FAILED] {cache_path}: {e}")
                    print("  -> retraining this seed ...")

            set_seed(seed)
            tag = f"{dataset_name} | H={horizon} | seed={seed} | {model_variant}"
            print(f"\n{'='*60}\n{tag}")

            trX, trY, vaX, vaY, teX, teY, scaler = load_multivariate_data(
                dataset_name, pred_len=horizon,
                seq_len=LOOKBACK, data_root=data_root)
            C = trX.shape[-1]

            def _loader(X, Y, shuffle):
                return DataLoader(
                    TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)),
                    batch_size=batch_size, shuffle=shuffle,
                    pin_memory=(dev.type == 'cuda'), num_workers=0)

            tr_loader = _loader(trX, trY, shuffle=True)
            va_loader = _loader(vaX, vaY, shuffle=False)
            te_loader = _loader(teX, teY, shuffle=False)

            va_size = len(va_loader.dataset)
            print(f"  Data: train={len(trX)}, val={va_size}, test={len(teX)}, C={C}")

            if va_size == 0:
                n_train_windows = len(trX)
                carve_n = max(100, int(n_train_windows * 0.1))
                carve_n = min(carve_n, n_train_windows - 500)
                if carve_n > 0:
                    vaX = trX[-carve_n:]; vaY = trY[-carve_n:]
                    trX = trX[:-carve_n]; trY = trY[:-carve_n]
                    tr_loader = _loader(trX, trY, shuffle=True)
                    va_loader = _loader(vaX, vaY, shuffle=False)
                    va_size   = len(va_loader.dataset)
                    print(f"  [SAFETY] Validation was empty. "
                          f"Carved {carve_n} windows from training.")
                else:
                    print(f"  WARNING: dataset too small.")

            model = RubickFormer(
                input_len=LOOKBACK, pred_len=horizon,
                patch_len=_PL, stride=_ST,
                d_model=effective_d_model,
                n_heads=effective_n_heads,
                e_layers=E_LAYERS, dropout=DROPOUT,
                num_features=C, use_learnable_pe=True,
                conv_size=CONV_SIZE,
                use_revin=use_revin,
                use_tslib_padding=use_tslib_padding,
                use_learnable_tau=_USE_TAU,
                tau_init=_TAU_INI,
                tau_min=TAU_MIN,
                use_conv_residual=_USE_RES, use_channel_heads=_CHHEAD,
            ).to(dev)

            print(f"  Patch config: patch_len={model.patch_len}, "
                  f"stride={model.stride}, "
                  f"num_patches={model.num_patches}, "
                  f"conv_size={model.conv_size}, "
                  f"padding={model.use_tslib_padding}")

            with torch.no_grad():
                _scales = [float(l.attention.output_scale.item())
                           for l in model.encoder]
                if _USE_TAU:
                    _taus_init = [
                        float(l.attention.temperature.clamp(min=TAU_MIN).mean().item())
                        for l in model.encoder
                    ]
                else:
                    _taus_init = None
                if _USE_RES and model.conv_gate_logit is not None:
                    _conv_lam_init = float(torch.sigmoid(model.conv_gate_logit).item())
                else:
                    _conv_lam_init = None

            print(f"  output_scale (per layer, init): {_scales}")
            if _taus_init is not None:
                print(f"  tau mean     (per layer, init): {_taus_init}")
            if _conv_lam_init is not None:
                print(f"  conv_gate lambda         (init): {_conv_lam_init:.4f}")

            n_params = sum(p.numel() for p in model.parameters()
                           if p.requires_grad)
            print(f"  Params: {n_params:,}")

            if use_tslib_training:
                optimizer = optim.Adam(model.parameters(), lr=lr)
                scheduler = None
                use_grad_clip = False
            else:
                optimizer = optim.AdamW(model.parameters(), lr=lr,
                                        weight_decay=1e-4)
                scheduler = CosineAnnealingLR(optimizer,
                                              T_max=epochs, eta_min=lr * 0.01)
                use_grad_clip = True

            # [ABLATION 1] CombinedLoss for training
            criterion = CombinedLoss(alpha=_L_ALPHA)

            best_val_loss  = float('inf')
            no_improve     = 0
            best_state     = None
            val_losses     = []
            train_losses   = []
            epochs_trained = 0

            if dev.type == 'cuda':
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
            training_start_time = time.time()

            for epoch in range(1, epochs + 1):
                epochs_trained = epoch

                if use_tslib_training:
                    current_lr = _tslib_type1_lr(lr, epoch)
                    for pg in optimizer.param_groups:
                        pg['lr'] = current_lr
                else:
                    current_lr = scheduler.get_last_lr()[0]

                model.train()
                tr_loss = 0.0
                for xb, yb in tr_loader:
                    xb, yb = xb.to(dev), yb.to(dev)
                    optimizer.zero_grad(set_to_none=True)
                    pred  = model(xb)
                    loss  = criterion(pred, yb)
                    loss.backward()
                    if use_grad_clip:
                        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    tr_loss += loss.item() * xb.size(0)
                tr_loss /= len(tr_loader.dataset)

                if not use_tslib_training:
                    scheduler.step()

                if va_size > 0:
                    model.eval()
                    va_loss = 0.0
                    with torch.no_grad():
                        for xb, yb in va_loader:
                            xb, yb = xb.to(dev), yb.to(dev)
                            va_loss += criterion(model(xb), yb).item() * xb.size(0)
                    va_loss /= va_size
                else:
                    va_loss = float('nan')

                val_losses.append(va_loss)
                train_losses.append(tr_loss)

                va_str = "nan" if math.isnan(va_loss) else f"{va_loss:.4f}"
                print(f"  ep {epoch:03d}/{epochs} | "
                      f"tr={tr_loss:.4f}  va={va_str}  "
                      f"lr={current_lr:.2e}")

                if not math.isnan(va_loss) and va_loss < best_val_loss - 1e-6:
                    best_val_loss = va_loss
                    best_state    = {k: v.cpu().clone()
                                     for k, v in model.state_dict().items()}
                    no_improve    = 0
                elif not math.isnan(va_loss):
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"  Early stop at epoch {epoch}.")
                        break

            training_time_sec = time.time() - training_start_time
            if dev.type == 'cuda':
                peak_gpu_mem_train_gb = (
                    torch.cuda.max_memory_allocated() / 1024**3)
            else:
                peak_gpu_mem_train_gb = 0.0

            if best_state is None:
                print(f"  No finite validation loss. Using last-epoch model.")
                best_state = {k: v.cpu().clone()
                              for k, v in model.state_dict().items()}
                best_val_loss = float('nan')

            model.load_state_dict(
                {k: v.to(dev) for k, v in best_state.items()})

            with torch.no_grad():
                _scales_end = [float(l.attention.output_scale.item())
                               for l in model.encoder]
                if _USE_TAU:
                    _taus_end = [
                        float(l.attention.temperature.clamp(min=TAU_MIN).mean().item())
                        for l in model.encoder
                    ]
                else:
                    _taus_end = None
                if _USE_RES and model.conv_gate_logit is not None:
                    _conv_lam_end = float(torch.sigmoid(model.conv_gate_logit).item())
                else:
                    _conv_lam_end = None

            print(f"  output_scale (final)    : {_scales_end}")
            if _taus_end is not None:
                print(f"  tau mean     (final)    : {[round(x, 4) for x in _taus_end]}")
            if _conv_lam_end is not None:
                print(f"  conv_gate lambda (final): {_conv_lam_end:.4f}")

            if dev.type == 'cuda':
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
            inference_start_time = time.time()

            model.eval()
            preds_list, trues_list = [], []
            with torch.no_grad():
                for xb, yb in te_loader:
                    out = model(xb.to(dev)).cpu().numpy()
                    preds_list.append(out)
                    trues_list.append(yb.numpy())

            if dev.type == 'cuda':
                torch.cuda.synchronize()
            inference_time_sec = time.time() - inference_start_time

            if dev.type == 'cuda':
                peak_gpu_mem_infer_gb = (
                    torch.cuda.max_memory_allocated() / 1024**3)
            else:
                peak_gpu_mem_infer_gb = 0.0

            preds = np.concatenate(preds_list, axis=0)
            trues = np.concatenate(trues_list, axis=0)

            metrics = compute_metrics(
                preds, trues,
                train_naive_mse_per_channel=train_naive_mse_per_channel,
            )
            all_results[horizon][seed] = {
                'best_val_loss': best_val_loss,
                'main':          metrics['main'],
                'supplementary': metrics['supplementary'],
                'MSE':           metrics['MSE'],
                'MAE':           metrics['MAE'],
                'RMSSE':         metrics['RMSSE'],
                'MDA':           metrics['MDA'],
                'Spearman':      metrics['Spearman'],
                'Pearson':       metrics['Pearson'],
            }
            val_losses_by_seed[seed]   = val_losses
            train_losses_by_seed[seed] = train_losses

            _save_npy_dict(
                cache_path,
                {
                    'preds':              preds,
                    'trues':              trues,
                    'metrics':            metrics,
                    'best_val_loss':      best_val_loss,
                    'val_losses':         val_losses,
                    'train_losses':       train_losses,
                    'n_params':           n_params,
                    'training_time_sec':  training_time_sec,
                    'inference_time_sec': inference_time_sec,
                    'peak_gpu_mem_train_gb': peak_gpu_mem_train_gb,
                    'peak_gpu_mem_infer_gb': peak_gpu_mem_infer_gb,
                    'n_epochs_trained':   epochs_trained,
                    'variant_meta': {
                        'arch_version':          _AV,
                        'use_channel_heads':     _CHHEAD,
                        'channel_head_gain_final': [
                            layer.attention.channel_gain().detach().cpu().tolist()
                            for layer in model.encoder if layer.attention.head_weights is not None and _CHHEAD
                        ],
                        'post_softmax_norm_type': NORM_TYPE,
                        'use_revin':             use_revin,
                        'use_tslib_padding':     use_tslib_padding,
                        'use_tslib_training':    use_tslib_training,
                        'capacity_matched':      capacity_matched,
                        'd_model':               effective_d_model,
                        'n_heads':               effective_n_heads,
                        'patch_len':             model.patch_len,
                        'stride':                model.stride,
                        'num_patches':           model.num_patches,
                        'conv_size':             model.conv_size,
                        'loss_alpha':            _L_ALPHA,
                        'use_learnable_tau':     _USE_TAU,
                        'tau_init':              _taus_init,
                        'tau_final':             _taus_end,
                        'use_conv_residual':     _USE_RES,
                        'conv_lambda_init':      _conv_lam_init,
                        'conv_lambda_final':     _conv_lam_end,
                        'optimizer':             'Adam' if use_tslib_training else 'AdamW',
                        'scheduler':             ('type1 (L,L,L/2,L/4,...)'
                                                  if use_tslib_training else 'cosine'),
                        'weight_decay':          0.0 if use_tslib_training else 1e-4,
                        'grad_clip':             None if use_tslib_training else GRAD_CLIP,
                        'output_scale_init':     _scales,
                        'output_scale_final':    _scales_end,
                    },
                },
            )

            print(f"  TEST [MAIN]  MSE={metrics['main']['MSE']:.4f}  "
                  f"MAE={metrics['main']['MAE']:.4f}")
            print(f"  TEST [SUPP]  RMSSE={metrics['supplementary']['RMSSE']:.4f}  "
                  f"MDA={metrics['supplementary']['MDA']:.4f}  "
                  f"Spearman={metrics['supplementary']['Spearman']:.4f}  "
                  f"Pearson={metrics['supplementary']['Pearson']:.4f}")

            print(f"  Training: {training_time_sec:.1f}s | "
                  f"Inference: {inference_time_sec:.2f}s | "
                  f"Peak GPU (train): {peak_gpu_mem_train_gb:.2f} GB | "
                  f"Epochs: {epochs_trained}")

        plot_train_val_losses(train_losses_by_seed, val_losses_by_seed,
                              horizon, dataset_name, model_variant, final_dir)

        mse_vals = [all_results[horizon][s]['MSE'] for s in _SD
                    if s in all_results[horizon]]
        mae_vals = [all_results[horizon][s]['MAE'] for s in _SD
                    if s in all_results[horizon]]
        if mse_vals:
            print(f"\n[H={horizon}]  "
                  f"MSE={np.mean(mse_vals):.4f}+/-{np.std(mse_vals):.4f}  "
                  f"MAE={np.mean(mae_vals):.4f}+/-{np.std(mae_vals):.4f}")

    out_path = os.path.abspath(os.path.join(
        final_dir, f"{dataset_name}_{model_variant}_results.json"))
    serialisable = {str(h): {str(s): v for s, v in seeds.items()}
                    for h, seeds in all_results.items()}
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    with open(out_path, 'w') as f:
        json.dump(serialisable, f, indent=2)
    print(f"\nResults saved -> {out_path}  "
          f"({os.path.getsize(out_path):,} bytes)")

    print(f"\n[VERIFY] Expected .npy files in {final_dir}:")
    missing = []
    for p in expected_npy_paths:
        if os.path.exists(p):
            print(f"  OK    {os.path.basename(p)}")
        else:
            print(f"  MISSING {p}")
            missing.append(p)
    if missing:
        print(f"\n[VERIFY] WARNING: {len(missing)} .npy file(s) missing.")
    else:
        print(f"[VERIFY] All {len(expected_npy_paths)} .npy file(s) present.")

    return all_results

# =============================================================================
# 8b. [PATCHING-ABLATION] Runner (Preserved from v6)
# =============================================================================
def run_patching_ablation(dataset_name, data_root, final_dir, device='cuda',
                          ablation_horizons=None, ablation_seeds=None,
                          epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                          patience=PATIENCE,
                          d_model_override=None, n_heads_override=None):
    """Run patch vs no-patch comparison and print a table."""
    if ablation_horizons is None:
        ablation_horizons = list(ABLATION_HORIZONS)
    if ablation_seeds is None:
        ablation_seeds = list(ABLATION_SEEDS)

    print("\n" + "#" * 80)
    print("# PATCHING ABLATION")
    print("#" * 80)
    print(f"# Dataset   : {dataset_name}")
    print(f"# Horizons  : {ablation_horizons}")
    print(f"# Seeds     : {ablation_seeds}")
    print(f"# Variant A : patch_len={_ABL_PATCH_CFG['patch_len']}, "
          f"stride={_ABL_PATCH_CFG['stride']}")
    print(f"# Variant B : patch_len={_ABL_NOPATCH_CFG['patch_len']}, "
          f"stride={_ABL_NOPATCH_CFG['stride']}")
    print("#" * 80 + "\n")

    print("\n" + "=" * 80)
    print(f"[ABLATION A] WITH PATCHING "
          f"(patch_len={_ABL_PATCH_CFG['patch_len']}, "
          f"stride={_ABL_PATCH_CFG['stride']})")
    print("=" * 80)
    patch_results = train_rubickformer(
        dataset_name=dataset_name, data_root=data_root,
        final_dir=final_dir, device=device,
        model_variant=_ABL_PATCH_CFG['variant_name'],
        epochs=epochs, batch_size=batch_size, lr=lr, patience=patience,
        d_model_override=d_model_override, n_heads_override=n_heads_override,
        patch_len_override=_ABL_PATCH_CFG['patch_len'],
        stride_override=_ABL_PATCH_CFG['stride'],
        horizons_override=ablation_horizons,
        seeds_override=ablation_seeds,
        arch_version_override=_ABL_PATCH_CFG['arch_version'],
    )

    print("\n" + "=" * 80)
    print(f"[ABLATION B] NO PATCHING "
          f"(patch_len={_ABL_NOPATCH_CFG['patch_len']}, "
          f"stride={_ABL_NOPATCH_CFG['stride']})")
    print("=" * 80)
    nopatch_results = train_rubickformer(
        dataset_name=dataset_name, data_root=data_root,
        final_dir=final_dir, device=device,
        model_variant=_ABL_NOPATCH_CFG['variant_name'],
        epochs=epochs, batch_size=batch_size, lr=lr, patience=patience,
        d_model_override=d_model_override, n_heads_override=n_heads_override,
        patch_len_override=_ABL_NOPATCH_CFG['patch_len'],
        stride_override=_ABL_NOPATCH_CFG['stride'],
        horizons_override=ablation_horizons,
        seeds_override=ablation_seeds,
        arch_version_override=_ABL_NOPATCH_CFG['arch_version'],
    )

    print("\n" + "#" * 80)
    print("# ABLATION RESULTS: Patching vs No-Patching")
    print("#" * 80)
    print(f"\n{'Horizon':>8s} | {'Patch MSE':>12s} | {'NoPatch MSE':>13s} | "
          f"{'Delta':>10s} | {'Delta %':>9s} | Winner")
    print("-" * 80)

    for h in ablation_horizons:
        p_list = [patch_results[h][s]['MSE'] for s in ablation_seeds
                  if h in patch_results and s in patch_results[h]]
        n_list = [nopatch_results[h][s]['MSE'] for s in ablation_seeds
                  if h in nopatch_results and s in nopatch_results[h]]

        if not p_list or not n_list:
            print(f"{h:>8d} | {'(missing)':>12s} | {'(missing)':>13s} | "
                  f"{'-':>10s} | {'-':>9s} | -")
            continue

        p_mean = float(np.mean(p_list))
        n_mean = float(np.mean(n_list))
        delta  = n_mean - p_mean
        delta_pct = (delta / p_mean) * 100 if p_mean > 0 else 0.0
        winner = "Patch" if p_mean < n_mean else ("No-Patch" if n_mean < p_mean else "Tie")
        print(f"{h:>8d} | {p_mean:>12.4f} | {n_mean:>13.4f} | "
              f"{delta:>+10.4f} | {delta_pct:>+8.2f}% | {winner}")

    ablation_json_path = os.path.join(
        final_dir, f'{dataset_name}_patching_ablation.json')
    with open(ablation_json_path, 'w') as f:
        json.dump({
            'patch':   {str(h): {str(s): patch_results[h][s] for s in ablation_seeds
                                 if h in patch_results and s in patch_results[h]}
                        for h in ablation_horizons},
            'nopatch': {str(h): {str(s): nopatch_results[h][s] for s in ablation_seeds
                                 if h in nopatch_results and s in nopatch_results[h]}
                        for h in ablation_horizons},
        }, f, indent=2)
    print(f"\nAblation JSON saved -> {ablation_json_path}")
    return patch_results, nopatch_results

# =============================================================================
# 8c. [Q1 FULL 4-PART ABLATION SUITE] Runner
# =============================================================================
def run_full_q1_ablation_suite(dataset_name, data_root, final_dir, device='cuda',
                               ablation_horizons=None, ablation_seeds=None,
                               epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                               patience=PATIENCE,
                               d_model_override=None, n_heads_override=None):
    """Runs all 4 requested Q1 ablations systematically and prints a unified
    publication-ready comparison table (MSE, MAE, MDA, RMSSE):
      0. v5_baseline      : Pure MSE (alpha=1.0), fixed tau, no Conv residual, Patch(8,1)
      1. +CombinedLoss    : alpha=0.8 (0.8*MSE + 0.2*MAE)
      2. +LearnableTau    : Learnable attention temperature tau
      3. +ConvResidual    : Gated skip connection around Conv1d
      4. v7_Full_Combined : All 3 outlier-sensitive enhancements active together
      5. v7_NoPatch       : Full enhancements without patching (patch_len=1, stride=1)
    """
    if ablation_horizons is None:
        ablation_horizons = list(ABLATION_HORIZONS)
    if ablation_seeds is None:
        ablation_seeds = list(ABLATION_SEEDS)

    configs = [
        {
            'name': '0_v5_Baseline',
            'variant': 'abl_v5_base_cm',
            'arch_version': 'v7_abl0_base',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 1.0, 'use_tau': False, 'use_res': False,
        },
        {
            'name': '1_CombinedLoss(a=0.8)',
            'variant': 'abl_combloss_cm',
            'arch_version': 'v7_abl1_loss',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 0.8, 'use_tau': False, 'use_res': False,
        },
        {
            'name': '2_LearnableTau',
            'variant': 'abl_tau_cm',
            'arch_version': 'v7_abl2_tau',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 1.0, 'use_tau': True, 'use_res': False,
        },
        {
            'name': '3_ConvResidual',
            'variant': 'abl_convres_cm',
            'arch_version': 'v7_abl3_res',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 1.0, 'use_tau': False, 'use_res': True,
        },
        {
            'name': '4_v7_All_Combined',
            'variant': 'abl_v7_all_cm',
            'arch_version': 'v7_abl4_all',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 0.8, 'use_tau': True, 'use_res': True,
        },
        {
            'name': '5_v7_NoPatch(1,1)',
            'variant': 'abl_v7_nopatch_cm',
            'arch_version': 'v7_abl5_nopatch',
            'patch_len': 1, 'stride': 1,
            'loss_alpha': 0.8, 'use_tau': True, 'use_res': True,
        },
    ]

    all_cfg = next(cfg for cfg in configs if cfg['variant'] == 'abl_v7_all_cm')
    configs.append(dict(all_cfg, name='6_v7_All_ChannelHeads', variant='abl_v7_all_cm_chhead'))

    configs.append(dict(all_cfg, name='7_v7_Selective', variant='abl_v7_all_cm_selhead'))
    suite_results = {}
    for cfg in configs:
        print("\n" + "=" * 84)
        print(f">>> RUNNING Q1 ABLATION VARIANT: {cfg['name']} <<<")
        print("=" * 84)
        res = train_rubickformer(
            dataset_name=dataset_name,
            data_root=data_root,
            final_dir=final_dir,
            device=device,
            model_variant=cfg['variant'],
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
            d_model_override=d_model_override,
            n_heads_override=n_heads_override,
            patch_len_override=cfg['patch_len'],
            stride_override=cfg['stride'],
            horizons_override=ablation_horizons,
            seeds_override=ablation_seeds,
            arch_version_override=cfg['arch_version'],
            loss_alpha_override=cfg['loss_alpha'],
            use_learnable_tau_override=cfg['use_tau'],
            use_conv_residual_override=cfg['use_res'],
        )
        suite_results[cfg['name']] = res

    # ---- Print Publication Summary Table ----
    print("\n" + "#" * 96)
    print(f"# Q1 ABLATION STUDY SUMMARY TABLE — Dataset: {dataset_name} "
          f"(Horizons: {ablation_horizons}, Seeds: {ablation_seeds})")
    print("#" * 96)
    header = (f"{'Configuration':<24s} | {'MSE':>9s} | {'dMSE%':>8s} | "
              f"{'MAE':>9s} | {'MDA':>9s} | {'RMSSE':>9s}")
    print(header)
    print("-" * 96)

    base_mse = None
    summary_payload = {}

    for cfg in configs:
        name = cfg['name']
        res  = suite_results[name]
        mses, maes, mdas, rmsses = [], [], [], []
        for h in ablation_horizons:
            for s in ablation_seeds:
                if h in res and s in res[h]:
                    m = res[h][s]
                    mses.append(m['MSE'])
                    maes.append(m['MAE'])
                    mdas.append(m['MDA'])
                    rmsses.append(m['RMSSE'])
        if not mses:
            continue
        mean_mse   = float(np.mean(mses))
        mean_mae   = float(np.mean(maes))
        mean_mda   = float(np.mean(mdas))
        mean_rmsse = float(np.mean(rmsses))

        if base_mse is None:
            base_mse = mean_mse
            dmse_str = "   0.00%"
        else:
            dmse_pct = ((mean_mse - base_mse) / base_mse) * 100.0 if base_mse > 0 else 0.0
            dmse_str = f"{dmse_pct:>+7.2f}%"

        print(f"{name:<24s} | {mean_mse:>9.4f} | {dmse_str:>8s} | "
              f"{mean_mae:>9.4f} | {mean_mda:>9.4f} | {mean_rmsse:>9.4f}")

        summary_payload[name] = {
            'MSE_mean': mean_mse,
            'MAE_mean': mean_mae,
            'MDA_mean': mean_mda,
            'RMSSE_mean': mean_rmsse,
        }

    print("#" * 96)
    suite_json_path = os.path.join(final_dir, f"{dataset_name}_q1_full_ablation_suite.json")
    with open(suite_json_path, 'w') as f:
        json.dump(summary_payload, f, indent=2)
    print(f"\nQ1 Ablation Suite JSON saved -> {suite_json_path}")
    return suite_results

# =============================================================================
# 9. ENTRY POINT
# =============================================================================
if __name__ == '__main__':
    import argparse

    _default_dataset   = globals().get('Dataset_Name', 'ETTh1')
    _default_data_root = globals().get('data_root', '.')
    _default_out_dir   = globals().get(
        'final_dir', f'./results/{_default_dataset}_baselines')
    _default_epochs    = globals().get('EPOCHS', 40)

    parser = argparse.ArgumentParser(description='Train RubickFormer v7')
    parser.add_argument('--dataset',    type=str, default=_default_dataset)
    parser.add_argument('--data_root',  type=str, default=_default_data_root)
    parser.add_argument('--out_dir',    type=str, default=_default_out_dir)
    parser.add_argument('--variant',    type=str, default='full_cm')
    parser.add_argument('--device',     type=str, default='cuda')
    parser.add_argument('--epochs',     type=int, default=_default_epochs)
    parser.add_argument('--d_model',    type=int, default=None)
    parser.add_argument('--n_heads',    type=int, default=None)
    parser.add_argument('--loss_alpha', type=float, default=None,
                        help='Alpha in alpha*MSE + (1-alpha)*MAE')
    parser.add_argument('--ablation',   action='store_true',
                        help='Run patching ablation instead of standard training')
    parser.add_argument('--q1_suite',   action='store_true',
                        help='Run all 4 Q1 ablations and print comparison table')

    def is_jupyter():
        try:
            from IPython import get_ipython
            shell = get_ipython()
            if shell is None:
                return False
            s = str(type(shell)).lower()
            return ('ipykernel' in s) or ('zmqshell' in s) or ('colab' in s)
        except Exception:
            return False

    if is_jupyter():
        print("[INFO] Notebook detected -> using default args.")
        args, _ = parser.parse_known_args([])
    else:
        args, _ = parser.parse_known_args()

    _run_q1_suite = ABLATION_Q1_SUITE or args.q1_suite
    _run_ablation = ABLATION_PATCHING or args.ablation

    if _run_q1_suite:
        print("\n" + "=" * 80)
        print(">>> Q1 FULL 4-PART ABLATION SUITE MODE <<<")
        print("=" * 80)
        run_full_q1_ablation_suite(
            dataset_name=args.dataset,
            data_root=args.data_root,
            final_dir=args.out_dir,
            device=args.device,
            ablation_horizons=ABLATION_HORIZONS,
            ablation_seeds=ABLATION_SEEDS,
            epochs=args.epochs,
            d_model_override=args.d_model,
            n_heads_override=args.n_heads,
        )
    elif _run_ablation:
        print("\n" + "=" * 80)
        print(">>> PATCHING ABLATION MODE <<<")
        print(">>> Set ABLATION_PATCHING = False for full training <<<")
        print("=" * 80)
        run_patching_ablation(
            dataset_name=args.dataset,
            data_root=args.data_root,
            final_dir=args.out_dir,
            device=args.device,
            ablation_horizons=ABLATION_HORIZONS,
            ablation_seeds=ABLATION_SEEDS,
            epochs=args.epochs,
            d_model_override=args.d_model,
            n_heads_override=args.n_heads,
        )
    else:
        print("\n" + "=" * 80)
        print(">>> STANDARD TRAINING MODE (v7 Enhanced) <<<")
        print(f">>> Horizons={HORIZONS}, Seeds={SEEDS} <<<")
        print("=" * 80)
        train_rubickformer(
            dataset_name=args.dataset,
            data_root=args.data_root,
            final_dir=args.out_dir,
            device=args.device,
            model_variant=args.variant,
            epochs=args.epochs,
            d_model_override=args.d_model,
            n_heads_override=args.n_heads,
            loss_alpha_override=args.loss_alpha,
        )

[DATA] load_multivariate_data is TSLib-compatible (seq_len overlap).
[INFO] Notebook detected -> using default args.

>>> STANDARD TRAINING MODE (v7 Enhanced) <<<
>>> Horizons=[96, 192, 336, 720], Seeds=[0, 1, 2, 3, 4, 5] <<<
[PATHS] cwd       : /mnt/d/Project/RubickFormer
[PATHS] final_dir : /mnt/d/Project/RubickFormer/results/ETTh1_baselines
[ARCH] ARCH_VERSION = v7_q1_full_selhead_rank1_v1 | NORM_TYPE = rmsnorm
[ABLATION CONFIG] patch_len=8, stride=4, loss_alpha=0.8, learnable_tau=True (init=0.5), conv_residual=True
[RUN SCOPE] horizons=[96, 192, 336, 720], seeds=[0, 1, 2, 3, 4, 5]
Device: cuda
  patch preview = 23 patches via Rubick padding (_fit_patching)
[VARIANT] full_cm
  use_revin          = True
  use_tslib_padding  = False
  use_tslib_training = False
  capacity_matched   = True
  effective d_model  = 32
  effective n_heads  = 4
  post_softmax_norm  = rmsnorm
  loss_alpha         = 0.8 (0.80*MSE + 0.20*MAE)
  use_learnable_tau  = True
  use_conv_residual  = True
[TRAINING CO

# PivotFormer Final

In [ ]:
# =============================================================================
# PIVOTFORMER - Channel-Independent Multi-Input Multi-Output
# -----------------------------------------------------------------------------
# v7.1 - FINAL Q1 ARCHITECTURE + FULL ABLATION SUITE
#   1. Combined Loss: alpha * MSE + (1 - alpha) * MAE (default alpha=0.8)
#   2. Learnable Attention Temperature (tau) for tanh logit de-squashing
#   3. Gated Residual Bypass around Conv1d (preserves high-frequency spikes)
#   4. Patching vs. No-Patching + Full Component Ablation Runner & Table
#   5. Local-detail gating on the attention branch (optional, default='none')
# =============================================================================

import sys, subprocess
try:
    import docx  # noqa: F401
except ImportError:
    print("[SETUP] Installing python-docx ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-docx"],
                   check=False)
    print("[SETUP] python-docx installed.")

import os
import math
import json
import hashlib
import random
import time
import gc
import inspect
import traceback
import warnings
import numpy as np
import pandas as pd
from scipy import stats

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, TensorDataset

# =============================================================================
# 0. GUARD: required globals
# =============================================================================
_REQUIRED_GLOBALS = [
    'Dataset_Name', 'LOOKBACK', 'HORIZONS', 'SEEDS',
    'EPOCHS', 'BATCH_SIZE', 'LR', 'DROPOUT', 'D_MODEL', 'PATIENCE',
    'load_multivariate_data', 'compute_train_seasonal_naive_mse_per_channel',
]
_missing = [g for g in _REQUIRED_GLOBALS if g not in globals()]
if _missing:
    raise RuntimeError(
        f"Missing required globals: {_missing}.\n"
        f"Make sure the config cell and the data loading cell "
        f"have been executed BEFORE this cell."
    )

# =============================================================================
# 0b. GUARD: verify load_multivariate_data is the TSLib-compatible version
# =============================================================================
try:
    _ldm_src = inspect.getsource(load_multivariate_data)
except Exception as e:
    raise RuntimeError(
        f"Could not inspect load_multivariate_data: {e}") from e

if '_compute_tslib_borders' not in _ldm_src:
    raise RuntimeError(
        "\n" + "!" * 72 + "\n"
        "load_multivariate_data is NOT the TSLib-compatible version.\n"
        "The Config Cell has likely overwritten it with the simplified split\n"
        "(no seq_len overlap at validation/test boundaries). This would make\n"
        "the comparison with PatchTST / Autoformer / etc. unfair.\n"
        "\n"
        "Fix: re-run the Data Cell (unified data loading cell) BEFORE this\n"
        "     training cell, so that the TSLib-compatible version is active.\n"
        + "!" * 72
    )
print("[DATA] load_multivariate_data is TSLib-compatible (seq_len overlap).")

# =============================================================================
# 1. CONFIG & ABLATION FLAGS
# =============================================================================
PATCH_LEN = 8
STRIDE    = 4
N_HEADS   = 4
E_LAYERS  = 3
CONV_SIZE = 7
GRAD_CLIP = 1.0

# Architecture version
ARCH_VERSION = 'v7_pivotformer_final'

# Skip & Head Regularization (final v7 defaults: full-rank skip, dropout=0.1)
SKIP_RANK    = None   # Low-rank bottleneck rank (None = original full-rank Linear)
HEAD_DROPOUT = 0.1    # Dropout in forecasting head

# Inter-layer local-detail gate. HORIZONS still trains one independent model per H.
DETAIL_GATE_MODE      = 'none'   # 'none' | 'scalar' | 'targeted' (standard run)
DETAIL_GATE_KERNEL    = 3        # odd number of patches; replicate the edge patches
DETAIL_GATE_INIT_KEEP = 0.95     # initial fraction of local detail passed to attention
ABLATION_DETAIL_GATE  = False    # True: run only the matched 3-way comparison
DETAIL_GATE_HORIZONS  = [96, 720]
DETAIL_GATE_SEEDS     = list(SEEDS)  # use [SEEDS[0]] for a one-seed pilot

# Independent output root for ALL writes made by this version of the cell.
def _detail_root(parent):
    return os.path.join(os.path.abspath(parent or '.'), 'PivotFormer_v7_1')

# Normalization applied after softmax, before output_scale.
NORM_TYPE = 'rmsnorm'  # or 'layernorm'

# Defaults used by capacity-matched (_cm) variants.
CM_D_MODEL = 32
CM_N_HEADS = 4

# ---- NEW v7 Architectural & Loss Hyperparameters (Default Flags) ----
USE_LEARNABLE_TAU = True     # Ablation 2: Learnable temperature (tau) on attention
TAU_INIT          = 0.5      # Initial temperature (<1.0 sharpens tanh logits)
TAU_MIN           = 0.05     # Lower clamp bound for numerical stability

USE_CONV_RESIDUAL = True     # Ablation 3: Gated residual bypass around Conv1d
LOSS_ALPHA        = 0.8      # Ablation 1: alpha*MSE + (1-alpha)*MAE (1.0 = pure MSE)

# =============================================================================
# [ABLATION SUITE] Config
# -----------------------------------------------------------------------------
# ABLATION_PATCHING = True  -> run patch vs no-patch ablation
# ABLATION_Q1_SUITE = True  -> run complete 4-part Q1 ablation table
# =============================================================================
ABLATION_PATCHING = False      # set True to run Patch vs No-Patch only
ABLATION_Q1_SUITE = False      # set True to run all 4 Q1 ablations + table
ABLATION_HORIZONS = [96]       # horizons to use in the ablation
ABLATION_SEEDS    = [0]        # seeds to use in the ablation

# The two configs used by the patching ablation
_ABL_PATCH_CFG = {
    'variant_name': 'full_cm',
    'patch_len': 8,
    'stride': 4,
    'arch_version': 'v7_pivot_patch_postsoftmax_norm_rms',
}
_ABL_NOPATCH_CFG = {
    'variant_name': 'full_cm_nopatch',
    'patch_len': 1,
    'stride': 1,
    'arch_version': 'v7_pivot_nopatch_postsoftmax_norm_rms',
}

def _fit_patching(input_len, patch_len, stride, conv_size, min_patches=None):
    """PivotFormer patching helper.

    [PATCHING-ABLATION] Allows patch_len=1.
    With patch_len=1 and stride=1, each timestep becomes one token and
    patch_embedding reduces to Linear(1, d_model).
    """
    pl = max(1, int(patch_len))
    st = max(1, int(stride))
    if pl > input_len:
        pl = input_len
    if st > pl:
        st = pl
    while True:
        n_patches = (input_len - pl) // st + 1
        if n_patches < conv_size and pl > 1:
            pl = max(1, pl // 2)
            st = max(1, min(st, pl))
            continue
        break
    n_patches = (input_len - pl) // st + 1
    cs = min(int(conv_size), n_patches)
    return pl, st, cs

def _parse_variant(model_variant: str):
    """Parse variant name into independent flags."""
    use_revin         = 'no_revin' not in model_variant
    capacity_matched  = '_cm' in model_variant
    use_tslib_padding = '_tslib' in model_variant
    has_stable        = '_stable' in model_variant

    if has_stable:
        use_tslib_training = False
    elif use_tslib_padding:
        use_tslib_training = True
    else:
        use_tslib_training = False

    return use_revin, use_tslib_padding, use_tslib_training, capacity_matched

def _tslib_type1_lr(initial_lr, epoch):
    """Match TSLib's lradj='type1' exactly (1-indexed epoch)."""
    return initial_lr * (0.5 ** max(0, epoch - 2))

def set_seed(seed: int = 0):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _cache_path(final_dir, dataset_name, model_variant, horizon, seed):
    # Use a stable public filename; isolate variants to avoid overwriting baselines.
    directory = (final_dir if model_variant == 'full_cm'
                 else os.path.join(final_dir, model_variant))
    return os.path.join(directory, f'{dataset_name}_full_cm_H{horizon}_seed{seed}.npy')


def _save_npy_dict(cache_path: str, payload: dict) -> str:
    cache_path = os.path.abspath(cache_path)
    parent = os.path.dirname(cache_path) or '.'
    try:
        os.makedirs(parent, exist_ok=True)
    except Exception as e:
        raise RuntimeError(
            f"Could not create cache directory '{parent}': {e}") from e
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=UserWarning, module="numpy")
            np.save(cache_path, payload, allow_pickle=True)
    except Exception:
        print(f"  !!! np.save FAILED for {cache_path}")
        traceback.print_exc()
        raise
    if not os.path.exists(cache_path):
        raise RuntimeError(
            f"np.save returned without error but {cache_path} does not exist.")
    size = os.path.getsize(cache_path)
    print(f"  Saved -> {cache_path}  ({size:,} bytes)")
    return cache_path

def _load_npy_dict(cache_path: str) -> dict:
    obj = np.load(cache_path, allow_pickle=True)
    if isinstance(obj, np.ndarray) and obj.dtype == object and obj.shape == ():
        return obj.item()
    return obj.item() if hasattr(obj, 'item') else obj

# =============================================================================
# 1b. [ABLATION 1] COMBINED LOSS (alpha * MSE + (1 - alpha) * MAE)
# =============================================================================
class CombinedLoss(nn.Module):
    """Convex combination of MSE and MAE loss:
       L = alpha * MSE(pred, target) + (1 - alpha) * MAE(pred, target)
    When alpha == 1.0, reduces strictly to standard nn.MSELoss().
    """
    def __init__(self, alpha: float = 0.8):
        super().__init__()
        if not (0.0 <= alpha <= 1.0):
            raise ValueError(f"alpha must be in [0.0, 1.0], got {alpha}")
        self.alpha = float(alpha)

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if self.alpha >= 1.0 - 1e-8:
            return F.mse_loss(pred, target)
        if self.alpha <= 1e-8:
            return F.l1_loss(pred, target)
        mse = F.mse_loss(pred, target)
        mae = F.l1_loss(pred, target)
        return self.alpha * mse + (1.0 - self.alpha) * mae

# =============================================================================
# 2. RevIN
# =============================================================================
class RevIN(nn.Module):
    def __init__(self, num_features: int = 1, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.gamma = nn.Parameter(torch.ones(num_features))
            self.beta  = nn.Parameter(torch.zeros(num_features))
        else:
            self.register_parameter('gamma', None)
            self.register_parameter('beta',  None)
        self.mean  = None
        self.stdev = None

    def forward(self, x, reverse: bool = False):
        if reverse:
            if self.mean is None or self.stdev is None:
                raise RuntimeError("RevIN.forward(reverse=False) must be called first.")
            if self.affine:
                x = (x - self.beta.view(1, 1, -1)) / (
                    self.gamma.view(1, 1, -1) + self.eps)
            return x * self.stdev + self.mean
        else:
            self.mean  = x.mean(dim=1, keepdim=True).detach()
            self.stdev = torch.sqrt(
                x.var(dim=1, keepdim=True, unbiased=False) + self.eps).detach()
            x = (x - self.mean) / self.stdev
            if self.affine:
                x = x * self.gamma.view(1, 1, -1) + self.beta.view(1, 1, -1)
            return x

# =============================================================================
# 2b. RMSNorm
# =============================================================================
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, NeurIPS 2019)."""
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight

# =============================================================================
# 3. PIVOTATTENTION BLOCK (with Learnable Temperature tau)
# =============================================================================
class PivotAttentionBlock(nn.Module):
    def __init__(self, n_heads, d_model, multi_layer=True, use_residual=True,
                 use_scaling=True, soft_max_axis=-1, dropout=0.1,
                 use_learnable_tau=False, tau_init=0.5, tau_min=0.05,
                 head_channels=None, selective=False):
        super().__init__()
        self.n_heads           = n_heads
        self.d_model           = d_model
        self.multi_layer       = multi_layer
        self.use_residual      = use_residual
        self.use_scaling       = use_scaling
        self.soft_max_axis     = soft_max_axis
        self.use_learnable_tau = use_learnable_tau
        self.tau_min           = tau_min

        self.W = nn.Parameter(torch.empty(d_model, d_model))
        self.b = nn.Parameter(torch.zeros(d_model))
        nn.init.xavier_uniform_(self.W)

        self.c = nn.Parameter(torch.empty(n_heads, d_model, d_model))
        nn.init.xavier_uniform_(self.c)

        # [ABLATION 2] Learnable per-head temperature tau to counteract tanh saturation
        if self.use_learnable_tau:
            self.temperature = nn.Parameter(torch.full((1, n_heads, 1, 1), float(tau_init)))
        else:
            self.register_parameter('temperature', None)

        if self.multi_layer and self.n_heads > 1:
            self.head_weights = nn.Parameter(torch.ones(n_heads))
        else:
            self.register_parameter('head_weights', None)

        self.register_parameter('channel_strength', None)
        self.register_parameter('head_profile', None)
        if selective and head_channels is not None and self.head_weights is not None:
            self.channel_strength = nn.Parameter(torch.zeros(head_channels))
            self.head_profile = nn.Parameter(torch.linspace(-1., 1., n_heads))
        if not selective and head_channels is not None and self.head_weights is not None:
            self.channel_delta = nn.Parameter(torch.zeros(head_channels, n_heads))
        else:
            self.register_parameter('channel_delta', None)

        if not self.multi_layer and self.n_heads > 1:
            self.output_projection = nn.Linear(self.n_heads * d_model, d_model)
        else:
            self.output_projection = None

        if self.use_residual and not self.multi_layer:
            self.layer_norm = nn.LayerNorm(d_model, eps=1e-6)
        else:
            self.layer_norm = None

        if self.multi_layer:
            if NORM_TYPE == 'rmsnorm':
                self.post_softmax_norm = RMSNorm(
                    n_heads * d_model, eps=1e-6)
            else:
                self.post_softmax_norm = nn.LayerNorm(
                    n_heads * d_model, eps=1e-6)
            self.output_scale = nn.Parameter(torch.ones(1))
        else:
            self.post_softmax_norm = None
            self.register_parameter('output_scale', None)

        self.dropout = nn.Dropout(dropout)

    def channel_gain(self):
        if self.channel_strength is not None:
            return 1.0 + self.channel_strength.tanh()[:, None] * self.head_profile.tanh()[None, :]
        if self.channel_delta is not None:
            return 1.0 + 0.2 * self.channel_delta.tanh()
        return None

    def forward(self, x):
        B, L, F = x.shape
        H = self.n_heads
        c_it = torch.tanh(torch.matmul(x, self.W) + self.b)
        a_it = torch.einsum('bld,hfd->bhld', c_it, self.c)
        if self.use_scaling:
            a_it = a_it / math.sqrt(F)

        # [ABLATION 2] Apply learnable temperature before softmax
        if self.use_learnable_tau and self.temperature is not None:
            tau = self.temperature.clamp(min=self.tau_min)
            a_it = a_it / tau

        a_it_soft = torch.softmax(a_it, dim=-1)
        a_it_soft = self.dropout(a_it_soft)
        x_expanded = x.unsqueeze(1)
        weighted = x_expanded * a_it_soft
        if self.head_weights is not None:
            gain = self.channel_gain()
            if gain is None:
                weighted = weighted * self.head_weights.view(1, H, 1, 1)
            else:
                # B here is batch_size * C, with channel varying fastest.
                C = gain.shape[0]
                hw = self.head_weights * gain  # [C, H]
                weighted = (weighted.reshape(-1, C, H, L, F)
                            * hw[None, :, :, None, None]).reshape(B, H, L, F)
        weighted = weighted.permute(0, 2, 1, 3).contiguous()
        concat = weighted.view(B, L, H * F)
        if self.multi_layer:
            out = torch.softmax(concat, dim=self.soft_max_axis)
            out = self.post_softmax_norm(out)
            return out * self.output_scale
        else:
            if self.output_projection is not None:
                out = self.output_projection(concat)
                out = torch.relu(out)
            else:
                out = concat
            if self.use_residual:
                out = out + x
            if self.layer_norm is not None:
                out = self.layer_norm(out)
            return out

class _ZeroInitGateLinear(nn.Linear):
    """A Linear(D,D,bias=False) initialized without consuming RNG. This keeps
    all shared model weights and the first minibatch paired across gate modes."""
    def reset_parameters(self):
        nn.init.zeros_(self.weight)


class LocalDetailGate(nn.Module):
    """Gate patch-to-patch deviations within each (sample, channel) separately."""
    def __init__(self, d_model, mode, kernel_size=3, init_keep=0.95):
        super().__init__()
        if mode not in ('scalar', 'targeted'):
            raise ValueError(f'Invalid active gate mode: {mode!r}')
        if not isinstance(kernel_size, int) or kernel_size < 3 or kernel_size % 2 != 1:
            raise ValueError('detail gate kernel must be an odd integer >= 3')
        if not 0.0 < init_keep < 1.0:
            raise ValueError('detail gate init_keep must be in (0,1)')
        self.kernel_size = kernel_size
        self.beta = nn.Parameter(torch.tensor(math.log(init_keep / (1.0 - init_keep))))
        self.mix = (_ZeroInitGateLinear(d_model, d_model, bias=False)
                    if mode == 'targeted' else None)
        self.keep = nn.Sigmoid()

    def forward(self, x):                         # [B*C, P, D]
        pad = self.kernel_size // 2
        padded = F.pad(x.transpose(1, 2), (pad, pad), mode='replicate')
        smooth = F.avg_pool1d(padded, self.kernel_size, stride=1).transpose(1, 2)
        detail = x - smooth
        logits = self.beta
        if self.mix is not None:
            q = (detail.square().mean(dim=1) + 1e-6).sqrt()  # [B*C, D]
            logits = logits + self.mix(q).unsqueeze(1)       # [B*C, 1, D]
        return smooth + self.keep(logits) * detail


class PivotEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, soft_max_axis, dropout=0.1,
                 use_scaling=True, use_learnable_tau=False,
                 tau_init=0.5, tau_min=0.05, head_channels=None, selective=False,
                 detail_gate_mode='none', detail_kernel=3, detail_init_keep=0.95):
        super().__init__()
        self.detail_gate = (None if detail_gate_mode == 'none' else LocalDetailGate(
            d_model, detail_gate_mode, detail_kernel, detail_init_keep))
        self.attention = PivotAttentionBlock(
            n_heads=n_heads, d_model=d_model, multi_layer=True,
            use_residual=False, use_scaling=use_scaling,
            soft_max_axis=soft_max_axis, dropout=dropout,
            use_learnable_tau=use_learnable_tau,
            tau_init=tau_init, tau_min=tau_min, head_channels=head_channels, selective=selective
        )
        self.proj    = nn.Linear(n_heads * d_model, d_model)
        self.norm    = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x  # The skip path must retain the original, unfiltered representation.
        x = self.attention(x if self.detail_gate is None else self.detail_gate(x))
        x = self.proj(x)
        x = self.dropout(x)
        x = x + residual
        x = self.norm(x)
        return x

# =============================================================================
# 4. PER-CHANNEL ACTIVATION
# =============================================================================
class PerChannelActivation(nn.Module):
    def __init__(self, num_channels, activation_type='pelu'):
        super().__init__()
        self.num_channels    = num_channels
        self.activation_type = activation_type
        if activation_type == 'pelu':
            self.alpha = nn.Parameter(torch.ones(num_channels))
            self.beta  = nn.Parameter(torch.ones(num_channels))
        else:
            self.alpha = None
            self.beta  = None

    def forward(self, x):
        if self.activation_type == 'pelu' and self.alpha is not None:
            alpha = self.alpha.view(1, -1, 1)
            beta  = self.beta.view(1, -1, 1)
            return torch.where(x >= 0, beta * x, alpha * (torch.exp(x) - 1))
        return x

# =============================================================================
# 5. PIVOTFORMER (with Gated Conv1d Residual Bypass & Learnable Tau)
# =============================================================================
class PivotFormer(nn.Module):
    def __init__(self, input_len=336, pred_len=96, patch_len=16, stride=8,
                 d_model=64, n_heads=8, e_layers=3, dropout=0.1,
                 num_features=7, use_learnable_pe=True, conv_size=7,
                 use_revin=True, use_tslib_padding=False,
                 use_learnable_tau=False, tau_init=0.5, tau_min=0.05,
                 use_conv_residual=False, skip_rank=SKIP_RANK,
                 head_dropout=HEAD_DROPOUT, use_channel_heads='selective',
                 detail_gate_mode=DETAIL_GATE_MODE, detail_kernel=DETAIL_GATE_KERNEL,
                 detail_init_keep=DETAIL_GATE_INIT_KEEP):
        super().__init__()
        if detail_gate_mode not in ('none', 'scalar', 'targeted'):
            raise ValueError(f'Unknown detail_gate_mode: {detail_gate_mode!r}')
        if detail_gate_mode != 'none' and e_layers < 2:
            raise ValueError('The detail gate needs at least 2 encoder layers.')

        self.use_tslib_padding = use_tslib_padding
        self.use_learnable_tau = use_learnable_tau
        self.use_channel_heads = use_channel_heads
        self.use_conv_residual = use_conv_residual
        self.skip_rank         = skip_rank
        self.head_dropout      = head_dropout

        if use_tslib_padding:
            padded_len       = input_len + stride
            self.num_patches = (padded_len - patch_len) // stride + 1
            self.conv_size   = min(int(conv_size), self.num_patches)
        else:
            patch_len, stride, conv_size = _fit_patching(
                input_len, patch_len, stride, conv_size)
            self.num_patches = (input_len - patch_len) // stride + 1
            self.conv_size   = int(conv_size)

        self.input_len        = input_len
        self.pred_len         = pred_len
        self.patch_len        = patch_len
        self.stride           = stride
        self.d_model          = d_model
        self.num_features     = num_features
        self.use_learnable_pe = use_learnable_pe
        self.use_revin        = use_revin

        if use_revin:
            self.revin = RevIN(num_features=1, eps=1e-5, affine=True)
        else:
            self.revin = None

        self.patch_embedding = nn.Linear(patch_len, d_model)

        if use_learnable_pe:
            self.pos_embed = nn.Parameter(
                torch.randn(1, self.num_patches, d_model) * 0.02)
        else:
            pe       = torch.zeros(self.num_patches, d_model)
            position = torch.arange(self.num_patches).unsqueeze(1).float()
            div      = torch.exp(torch.arange(0, d_model, 2).float()
                                 * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div)
            pe[:, 1::2] = torch.cos(position * div)
            self.register_buffer('pe', pe.unsqueeze(0))

        self.dropout_layer = nn.Dropout(dropout)

        layers = []
        for i in range(e_layers):
            axis = -2 if i % 2 == 0 else -1
            layers.append(PivotEncoderLayer(
                d_model, n_heads, soft_max_axis=axis,
                dropout=dropout, use_scaling=True,
                use_learnable_tau=use_learnable_tau,
                tau_init=tau_init, tau_min=tau_min,
                head_channels=num_features if use_channel_heads else None,
                selective=(use_channel_heads == 'selective'),
                detail_gate_mode=detail_gate_mode if i >= 1 else 'none',
                detail_kernel=detail_kernel, detail_init_keep=detail_init_keep))
        self.encoder = nn.Sequential(*layers)

        self.head_linear = nn.Linear(d_model, d_model)
        self.head_drop   = nn.Dropout(head_dropout)
        self.head_pelu   = PerChannelActivation(d_model, activation_type='pelu')

        L_after_conv = self.num_patches - self.conv_size + 1
        if L_after_conv < 1:
            raise ValueError(
                f"num_patches={self.num_patches} must be >= "
                f"conv_size={self.conv_size}")
        self.head_conv  = nn.Conv1d(in_channels=d_model,
                                    out_channels=pred_len,
                                    kernel_size=self.conv_size, padding=0)
        self.head_final = nn.Linear(L_after_conv, 1)

        # [ABLATION 3] Gated Residual Bypass around Conv1d
        if self.use_conv_residual:
            in_dim = d_model * self.num_patches
            if skip_rank is not None and skip_rank < min(in_dim, pred_len):
                self.conv_skip_proj = nn.Sequential(
                    nn.Linear(in_dim, skip_rank, bias=False),
                    nn.Dropout(head_dropout),
                    nn.Linear(skip_rank, pred_len, bias=True),
                )
            else:
                self.conv_skip_proj = nn.Linear(in_dim, pred_len)
            # Initialized to logit(0.8) ~= 1.386 so model starts near original Conv1d path
            self.conv_gate_logit = nn.Parameter(torch.tensor(1.38629436))
        else:
            self.conv_skip_proj  = None
            self.register_parameter('conv_gate_logit', None)

    def forward(self, x):
        B, T, C = x.shape
        if self.use_channel_heads and C != self.num_features:
            raise ValueError(f'Expected {self.num_features} input channels in training order, got {C}')
        x = x.permute(0, 2, 1).reshape(B * C, T, 1)

        if self.use_revin:
            x = self.revin(x, reverse=False)

        x = x.squeeze(-1)

        if self.use_tslib_padding:
            x = F.pad(x, (0, self.stride), mode='replicate')

        x = x.unfold(1, self.patch_len, self.stride)
        x = self.patch_embedding(x)

        if self.use_learnable_pe:
            x = x + self.pos_embed
        else:
            x = x + self.pe[:, :self.num_patches, :]

        x = self.dropout_layer(x)
        x = self.encoder(x)

        x = self.head_linear(x)
        x = self.head_drop(x)
        x = x.permute(0, 2, 1)          # (B*C, d_model, num_patches)
        x_pelu = self.head_pelu(x)      # (B*C, d_model, num_patches)

        # Main Conv1d branch
        conv_out = self.head_conv(x_pelu)       # (B*C, pred_len, L_after_conv)
        conv_out = self.head_final(conv_out)    # (B*C, pred_len, 1)

        # [ABLATION 3] Gated Residual Bypass branch
        if self.use_conv_residual and self.conv_skip_proj is not None:
            skip_in  = x_pelu.reshape(B * C, -1)                    # (B*C, d_model * num_patches)
            skip_out = self.conv_skip_proj(skip_in).unsqueeze(-1)   # (B*C, pred_len, 1)
            lam      = torch.sigmoid(self.conv_gate_logit)
            x        = lam * conv_out + (1.0 - lam) * skip_out
        else:
            x = conv_out

        if self.use_revin:
            x = self.revin(x, reverse=True)

        x = x.reshape(B, C, self.pred_len, 1).squeeze(-1)
        return x.permute(0, 2, 1)

# =============================================================================
# 6. EVALUATION METRICS
# =============================================================================
def compute_metrics(preds, trues, train_naive_mse_per_channel=None):
    mse = float(np.mean((preds - trues) ** 2))
    mae = float(np.mean(np.abs(preds - trues)))

    if train_naive_mse_per_channel is not None:
        test_mse_per_channel = np.mean((preds - trues) ** 2, axis=(0, 1))
        safe = np.where(np.asarray(train_naive_mse_per_channel) > 0,
                        train_naive_mse_per_channel, np.nan)
        rmsse = float(np.nanmean(np.sqrt(test_mse_per_channel / safe)))
    else:
        rmsse = float(np.sqrt(mse))

    d_pred = np.diff(preds, axis=1)
    d_true = np.diff(trues, axis=1)
    mda    = float(np.mean((d_pred * d_true) > 0))

    pf, tf = preds.flatten(), trues.flatten()
    try:    spearman = float(stats.spearmanr(pf, tf)[0])
    except Exception: spearman = 0.0
    try:    pearson  = float(stats.pearsonr(pf, tf)[0])
    except Exception: pearson  = 0.0

    return {
        'main': {'MSE': mse, 'MAE': mae},
        'supplementary': {'RMSSE': rmsse, 'MDA': mda,
                          'Spearman': spearman, 'Pearson': pearson},
        'MSE': mse, 'MAE': mae,
        'RMSSE': rmsse, 'MDA': mda,
        'Spearman': spearman, 'Pearson': pearson,
    }

@torch.no_grad()
def _detail_validation(model, loader, dev):
    """One validation-only pass of the selected checkpoint; no predictions retained.
    Gate means refer to actual sigmoid outputs, not just sigmoid(beta)."""
    if len(loader.dataset) == 0:
        return {}, {}
    model.eval()
    H = model.pred_len
    se = torch.zeros(H, dtype=torch.float64, device=dev)
    ae = torch.zeros_like(se)
    n = 0
    sums, counts, hooks = {}, {}, []

    def observer(layer_id):
        def collect(_module, _args, g):
            sums[layer_id] += g.sum(dtype=torch.float64)
            counts[layer_id] += g.numel()
        return collect

    try:
        for i, layer in enumerate(model.encoder, start=1):
            if layer.detail_gate is not None:
                sums[i] = torch.zeros((), dtype=torch.float64, device=dev)
                counts[i] = 0
                hooks.append(layer.detail_gate.keep.register_forward_hook(observer(i)))
        for xb, yb in loader:
            err = model(xb.to(dev)) - yb.to(dev)  # [B, H, C]
            se += err.square().sum(dim=(0, 2), dtype=torch.float64)
            ae += err.abs().sum(dim=(0, 2), dtype=torch.float64)
            n += err.size(0) * err.size(2)
    finally:
        for h in hooks:
            h.remove()
    se, ae = (se / n).cpu().numpy(), (ae / n).cpu().numpy()
    blocks = {'all': {'MSE': float(se.mean()), 'MAE': float(ae.mean())}}
    start = 0
    for stop in sorted({min(H, b) for b in (96, 192, 336, H)}):
        blocks[f'{start+1}-{stop}'] = {'MSE': float(se[start:stop].mean()),
                                       'MAE': float(ae[start:stop].mean())}
        start = stop
    means = {f'L{i}': (sums[i] / counts[i]).item() for i in sums}
    return blocks, means


# =============================================================================
# 7. PLOTTING
# =============================================================================
def plot_train_val_losses(train_losses_by_seed, val_losses_by_seed,
                          horizon, dataset_name, model_variant, out_dir):
    if (not any(val_losses_by_seed.values())
            and not any(train_losses_by_seed.values())):
        return

    plt.figure(figsize=(11, 6))
    cmap  = plt.get_cmap('tab10')
    seeds = sorted(set(val_losses_by_seed.keys())
                   | set(train_losses_by_seed.keys()))
    for idx, seed in enumerate(seeds):
        color     = cmap(idx % 10)
        tr_losses = train_losses_by_seed.get(seed, [])
        va_losses = val_losses_by_seed.get(seed, [])
        if tr_losses:
            plt.plot(range(1, len(tr_losses) + 1), tr_losses,
                     linestyle='-', color=color, alpha=0.85, linewidth=1.5,
                     label=f'seed={seed} train')
        if va_losses:
            plt.plot(range(1, len(va_losses) + 1), va_losses,
                     linestyle='--', color=color, alpha=0.85, linewidth=1.5,
                     label=f'seed={seed} val')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title(f'{dataset_name} | H={horizon} | {model_variant}\n'
              f'Train (solid) vs Validation (dashed) Loss across seeds')
    plt.legend(loc='best', ncol=2, fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plot_path = os.path.abspath(os.path.join(
        out_dir or '.',
        f"{dataset_name}_{model_variant}_H{horizon}_train_val_loss.png"))
    os.makedirs(os.path.dirname(plot_path) or '.', exist_ok=True)
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f"\n  Plot saved -> {plot_path}")

# =============================================================================
# 8. TRAINING PIPELINE (with full Q1 ablation overrides)
# =============================================================================
def train_pivotformer(dataset_name, data_root, final_dir,
                      device='cuda', model_variant='full_cm',
                      epochs=EPOCHS, batch_size=BATCH_SIZE,
                      lr=LR, patience=PATIENCE,
                      d_model_override=None,
                      n_heads_override=None,
                      patch_len_override=None,
                      stride_override=None,
                      horizons_override=None,
                      seeds_override=None,
                      arch_version_override=None,
                      loss_alpha_override=None,
                      use_learnable_tau_override=None,
                      tau_init_override=None,
                      use_conv_residual_override=None,
                      skip_rank_override=None,
                      head_dropout_override=None,
                      detail_gate_mode_override=None):
    """Train PivotFormer with full Q1 ablation support."""

    # ---- resolve locals (ablation overrides win over globals) ----
    _PL   = patch_len_override if patch_len_override is not None else PATCH_LEN
    _ST   = stride_override    if stride_override    is not None else STRIDE
    _AV   = arch_version_override if arch_version_override is not None else ARCH_VERSION
    if model_variant == 'full_cm_selhead':  # Legacy alias for the final architecture.
        model_variant = 'full_cm'
    _CHHEAD = 'chhead' in model_variant.split('_')
    if ('selhead' in model_variant.split('_') or
            (not model_variant.startswith('abl_') and not _CHHEAD)):
        _CHHEAD = 'selective'
        _AV += '_selhead_rank1_v1'
    elif _CHHEAD:
        _AV += '_chhead_v1_r020'
    _HZ   = list(horizons_override) if horizons_override is not None else list(HORIZONS)
    _SD   = list(seeds_override)    if seeds_override    is not None else list(SEEDS)

    _L_ALPHA   = loss_alpha_override if loss_alpha_override is not None else LOSS_ALPHA
    _USE_TAU   = use_learnable_tau_override if use_learnable_tau_override is not None else USE_LEARNABLE_TAU
    _TAU_INI   = tau_init_override if tau_init_override is not None else TAU_INIT
    _USE_RES   = use_conv_residual_override if use_conv_residual_override is not None else USE_CONV_RESIDUAL
    _SKIP_RANK = SKIP_RANK if skip_rank_override is None else skip_rank_override
    _HEAD_DROP = HEAD_DROPOUT if head_dropout_override is None else head_dropout_override

    mode = DETAIL_GATE_MODE if detail_gate_mode_override is None else detail_gate_mode_override
    if mode not in ('none', 'scalar', 'targeted'):
        raise ValueError(f'Unknown gate mode: {mode!r}')
    upper_dir = _detail_root(final_dir)
    _AV += '_detailgate_v1_' + mode
    print(f"[PATHS] cwd       : {os.getcwd()}")
    print(f"[ARCH] ARCH_VERSION = {_AV} | NORM_TYPE = {NORM_TYPE}")
    print(f"[ABLATION CONFIG] patch_len={_PL}, stride={_ST}, "
          f"loss_alpha={_L_ALPHA}, learnable_tau={_USE_TAU} (init={_TAU_INI}), "
          f"conv_residual={_USE_RES}, skip_rank={_SKIP_RANK}, head_dropout={_HEAD_DROP}")
    print(f"[RUN SCOPE] horizons={_HZ}, seeds={_SD}")

    dev = torch.device(
        'cuda' if (device == 'cuda' and torch.cuda.is_available()) else 'cpu')
    print(f"Device: {dev}")

    (use_revin, use_tslib_padding,
     use_tslib_training, capacity_matched) = _parse_variant(model_variant)

    if capacity_matched:
        if d_model_override is None:
            d_model_override = CM_D_MODEL
        if n_heads_override is None:
            n_heads_override = CM_N_HEADS

    effective_d_model = d_model_override if d_model_override is not None else D_MODEL
    effective_n_heads = n_heads_override if n_heads_override is not None else N_HEADS

    # Constant across seeds/horizons (both are already in each cache filename).
    run_config = dict(architecture=_AV, model_variant=model_variant, mode=mode,
                      kernel=DETAIL_GATE_KERNEL if mode != 'none' else None,
                      init_keep=DETAIL_GATE_INIT_KEEP if mode != 'none' else None,
                      lookback=LOOKBACK, d_model=effective_d_model, heads=effective_n_heads,
                      layers=E_LAYERS, patch=_PL, stride=_ST, conv=CONV_SIZE,
                      dropout=DROPOUT, head_dropout=_HEAD_DROP, skip_rank=_SKIP_RANK,
                      norm=NORM_TYPE, alpha=_L_ALPHA, tau=_USE_TAU,
                      tau_init=_TAU_INI, tau_min=TAU_MIN, conv_residual=_USE_RES,
                      epochs=epochs, batch_size=batch_size, patience=patience, lr=lr,
                      grad_clip=GRAD_CLIP, device=str(dev), data_root=os.path.abspath(data_root),
                      loader_sha=hashlib.sha256(_ldm_src.encode()).hexdigest())
    config_id = hashlib.sha256(json.dumps(run_config, sort_keys=True, default=str).encode()).hexdigest()[:16]
    final_dir = os.path.join(upper_dir, mode, config_id)
    os.makedirs(final_dir, exist_ok=True)
    print(f'[PATHS] final_dir : {final_dir}')
    print(f'[DETAIL] mode={mode} | kernel={run_config["kernel"]} | init_keep={run_config["init_keep"]}')

    if use_tslib_padding:
        _padded_len = LOOKBACK + _ST
        _preview_patches = (_padded_len - _PL) // _ST + 1
        _preview_desc = "TSLib padding (PatchTST-style)"
    else:
        _pl_fit, _st_fit, _cs_fit = _fit_patching(LOOKBACK, _PL, _ST, CONV_SIZE)
        _preview_patches = (LOOKBACK - _pl_fit) // _st_fit + 1
        _preview_desc = "Pivot padding (_fit_patching)"

    if _PL == 1 and _ST == 1:
        print(f"  [NO-PATCH MODE] {_preview_patches} tokens "
              f"(one per timestep), patch_embedding = Linear(1, d_model)")
    elif _PL == 1 and _ST != 1:
        print(f"  [WARN] PATCH_LEN=1 with STRIDE={_ST} will SKIP timesteps. "
              f"Set STRIDE=1 for true no-patch mode.")
        print(f"  patch preview = {_preview_patches} tokens via {_preview_desc}")
    else:
        print(f"  patch preview = {_preview_patches} patches via {_preview_desc}")

    print(f"[VARIANT] {model_variant}")
    print(f"  use_revin          = {use_revin}")
    print(f"  use_tslib_padding  = {use_tslib_padding}")
    print(f"  use_tslib_training = {use_tslib_training}")
    print(f"  capacity_matched   = {capacity_matched}")
    print(f"  effective d_model  = {effective_d_model}")
    print(f"  effective n_heads  = {effective_n_heads}")
    print(f"  post_softmax_norm  = {NORM_TYPE}")
    print(f"  loss_alpha         = {_L_ALPHA} ({_L_ALPHA:.2f}*MSE + {1.0-_L_ALPHA:.2f}*MAE)")
    print(f"  use_learnable_tau  = {_USE_TAU}")
    print(f"  use_conv_residual  = {_USE_RES} (skip_rank={_SKIP_RANK}, head_drop={_HEAD_DROP})")

    if use_tslib_training:
        print(f"[TRAINING CONFIG] Adam (no wd) + lradj='type1' + no grad clip")
        print(f"  WARNING: TSLib training was observed UNSTABLE for PivotAttention.")
    else:
        print(f"[TRAINING CONFIG] AdamW (wd=1e-4) + Cosine + "
              f"grad_clip={GRAD_CLIP}  (stable)")

    if 'SEASONAL_MAP' in globals() and dataset_name in SEASONAL_MAP:
        seasonal_period = SEASONAL_MAP[dataset_name]
    elif dataset_name.startswith('ETTh'):
        seasonal_period = 24
    elif dataset_name.startswith('ETTm'):
        seasonal_period = 96
    else:
        seasonal_period = 24
    print(f"[SEASONAL] m={seasonal_period}")

    train_naive_mse_per_channel = compute_train_seasonal_naive_mse_per_channel(
        dataset_name, data_root, seasonal_period=seasonal_period)

    print(f"[RMSSE denom] seasonal-naive MSE per channel (train, m={seasonal_period}):")
    for i, v in enumerate(train_naive_mse_per_channel):
        print(f"    ch{i}: {v:.6f}")
    print(f"    mean: {train_naive_mse_per_channel.mean():.6f}")

    all_results = {}
    expected_npy_paths = []

    for horizon in _HZ:
        all_results[horizon] = {}
        val_losses_by_seed   = {}
        train_losses_by_seed = {}

        for seed in _SD:
            cache_path = _cache_path(final_dir, dataset_name, model_variant, horizon, seed)
            expected_npy_paths.append(cache_path)

            if os.path.exists(cache_path):
                try:
                    cached = _load_npy_dict(cache_path)

                    cached_arch = cached.get('variant_meta', {}).get(
                        'arch_version', 'unknown')
                    if cached_arch != _AV:
                        print(f"\n[CACHE INVALIDATED] "
                              f"{os.path.basename(cache_path)}: "
                              f"arch_version='{cached_arch}' != "
                              f"'{_AV}' -> retraining")
                    else:
                        all_results[horizon][seed] = dict(cached['metrics'], best_val_loss=cached['best_val_loss'])
                        val_losses_by_seed[seed]   = cached.get('val_losses', [])
                        train_losses_by_seed[seed] = cached.get('train_losses', [])
                        print(f"\n{'='*60}\n"
                              f"[CACHE HIT] {os.path.abspath(cache_path)}")
                        m      = cached['metrics']
                        m_main = m.get('main', {'MSE': m.get('MSE', 0),
                                                'MAE': m.get('MAE', 0)})
                        m_supp = m.get('supplementary',
                                       {'RMSSE': m.get('RMSSE', 0),
                                        'MDA': m.get('MDA', 0),
                                        'Spearman': m.get('Spearman', 0),
                                        'Pearson': m.get('Pearson', 0)})
                        print(f"  [MAIN]  MSE={m_main['MSE']:.4f}  "
                              f"MAE={m_main['MAE']:.4f}")
                        print(f"  [SUPP]  RMSSE={m_supp['RMSSE']:.4f}  "
                              f"MDA={m_supp['MDA']:.4f}  "
                              f"Spearman={m_supp['Spearman']:.4f}  "
                              f"Pearson={m_supp['Pearson']:.4f}")
                        print(f"  [SKIP SAVE] cached file reused.")
                        continue
                except Exception as e:
                    print(f"\n[CACHE LOAD FAILED] {cache_path}: {e}")
                    print("  -> retraining this seed ...")

            set_seed(seed)
            tag = f"{dataset_name} | H={horizon} | seed={seed} | {model_variant}"
            print(f"\n{'='*60}\n{tag}")

            trX, trY, vaX, vaY, teX, teY, scaler = load_multivariate_data(
                dataset_name, pred_len=horizon,
                seq_len=LOOKBACK, data_root=data_root)
            C = trX.shape[-1]

            def _loader(X, Y, shuffle):
                return DataLoader(
                    TensorDataset(torch.from_numpy(X), torch.from_numpy(Y)),
                    batch_size=batch_size, shuffle=shuffle,
                    pin_memory=(dev.type == 'cuda'), num_workers=0)

            tr_loader = _loader(trX, trY, shuffle=True)
            va_loader = _loader(vaX, vaY, shuffle=False)
            te_loader = _loader(teX, teY, shuffle=False)

            va_size = len(va_loader.dataset)
            print(f"  Data: train={len(trX)}, val={va_size}, test={len(teX)}, C={C}")

            if va_size == 0:
                n_train_windows = len(trX)
                carve_n = max(100, int(n_train_windows * 0.1))
                carve_n = min(carve_n, n_train_windows - 500)
                if carve_n > 0:
                    vaX = trX[-carve_n:]; vaY = trY[-carve_n:]
                    trX = trX[:-carve_n]; trY = trY[:-carve_n]
                    tr_loader = _loader(trX, trY, shuffle=True)
                    va_loader = _loader(vaX, vaY, shuffle=False)
                    va_size   = len(va_loader.dataset)
                    print(f"  [SAFETY] Validation was empty. "
                          f"Carved {carve_n} windows from training.")
                else:
                    print(f"  WARNING: dataset too small.")

            model = PivotFormer(
                input_len=LOOKBACK, pred_len=horizon,
                patch_len=_PL, stride=_ST,
                d_model=effective_d_model,
                n_heads=effective_n_heads,
                e_layers=E_LAYERS, dropout=DROPOUT,
                num_features=C, use_learnable_pe=True,
                conv_size=CONV_SIZE,
                use_revin=use_revin,
                use_tslib_padding=use_tslib_padding,
                use_learnable_tau=_USE_TAU,
                tau_init=_TAU_INI,
                tau_min=TAU_MIN,
                use_conv_residual=_USE_RES,
                skip_rank=_SKIP_RANK,
                head_dropout=_HEAD_DROP,
                use_channel_heads=_CHHEAD,
                detail_gate_mode=mode, detail_kernel=DETAIL_GATE_KERNEL,
                detail_init_keep=DETAIL_GATE_INIT_KEEP,
            ).to(dev)

            print(f"  Patch config: patch_len={model.patch_len}, "
                  f"stride={model.stride}, "
                  f"num_patches={model.num_patches}, "
                  f"conv_size={model.conv_size}, "
                  f"padding={model.use_tslib_padding}")

            with torch.no_grad():
                _scales = [float(l.attention.output_scale.item())
                           for l in model.encoder]
                if _USE_TAU:
                    _taus_init = [
                        float(l.attention.temperature.clamp(min=TAU_MIN).mean().item())
                        for l in model.encoder
                    ]
                else:
                    _taus_init = None
                if _USE_RES and model.conv_gate_logit is not None:
                    _conv_lam_init = float(torch.sigmoid(model.conv_gate_logit).item())
                else:
                    _conv_lam_init = None

            print(f"  output_scale (per layer, init): {_scales}")
            if _taus_init is not None:
                print(f"  tau mean     (per layer, init): {_taus_init}")
            if _conv_lam_init is not None:
                print(f"  conv_gate lambda         (init): {_conv_lam_init:.4f}")

            n_params = sum(p.numel() for p in model.parameters()
                           if p.requires_grad)
            print(f"  Params: {n_params:,}")

            if use_tslib_training:
                optimizer = optim.Adam(model.parameters(), lr=lr)
                scheduler = None
                use_grad_clip = False
            else:
                optimizer = optim.AdamW(model.parameters(), lr=lr,
                                        weight_decay=1e-4)
                scheduler = CosineAnnealingLR(optimizer,
                                              T_max=epochs, eta_min=lr * 0.01)
                use_grad_clip = True

            # [ABLATION 1] CombinedLoss for training
            criterion = CombinedLoss(alpha=_L_ALPHA)

            best_val_loss  = float('inf')
            no_improve     = 0
            best_state     = None
            val_losses     = []
            train_losses   = []
            epochs_trained = 0

            if dev.type == 'cuda':
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
            training_start_time = time.time()

            for epoch in range(1, epochs + 1):
                epochs_trained = epoch

                if use_tslib_training:
                    current_lr = _tslib_type1_lr(lr, epoch)
                    for pg in optimizer.param_groups:
                        pg['lr'] = current_lr
                else:
                    current_lr = scheduler.get_last_lr()[0]

                model.train()
                tr_loss = 0.0
                for xb, yb in tr_loader:
                    xb, yb = xb.to(dev), yb.to(dev)
                    optimizer.zero_grad(set_to_none=True)
                    pred  = model(xb)
                    loss  = criterion(pred, yb)
                    loss.backward()
                    if use_grad_clip:
                        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    tr_loss += loss.item() * xb.size(0)
                tr_loss /= len(tr_loader.dataset)

                if not use_tslib_training:
                    scheduler.step()

                if va_size > 0:
                    model.eval()
                    va_loss = 0.0
                    with torch.no_grad():
                        for xb, yb in va_loader:
                            xb, yb = xb.to(dev), yb.to(dev)
                            va_loss += criterion(model(xb), yb).item() * xb.size(0)
                    va_loss /= va_size
                else:
                    va_loss = float('nan')

                val_losses.append(va_loss)
                train_losses.append(tr_loss)

                va_str = "nan" if math.isnan(va_loss) else f"{va_loss:.4f}"
                print(f"  ep {epoch:03d}/{epochs} | "
                      f"tr={tr_loss:.4f}  va={va_str}  "
                      f"lr={current_lr:.2e}")

                if not math.isnan(va_loss) and va_loss < best_val_loss - 1e-6:
                    best_val_loss = va_loss
                    best_state    = {k: v.cpu().clone()
                                     for k, v in model.state_dict().items()}
                    no_improve    = 0
                elif not math.isnan(va_loss):
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"  Early stop at epoch {epoch}.")
                        break

            training_time_sec = time.time() - training_start_time
            if dev.type == 'cuda':
                peak_gpu_mem_train_gb = (
                    torch.cuda.max_memory_allocated() / 1024**3)
            else:
                peak_gpu_mem_train_gb = 0.0

            if best_state is None:
                print(f"  No finite validation loss. Using last-epoch model.")
                best_state = {k: v.cpu().clone()
                              for k, v in model.state_dict().items()}
                best_val_loss = float('nan')

            model.load_state_dict(
                {k: v.to(dev) for k, v in best_state.items()})

            with torch.no_grad():
                _scales_end = [float(l.attention.output_scale.item())
                               for l in model.encoder]
                if _USE_TAU:
                    _taus_end = [
                        float(l.attention.temperature.clamp(min=TAU_MIN).mean().item())
                        for l in model.encoder
                    ]
                else:
                    _taus_end = None
                if _USE_RES and model.conv_gate_logit is not None:
                    _conv_lam_end = float(torch.sigmoid(model.conv_gate_logit).item())
                else:
                    _conv_lam_end = None

            print(f"  output_scale (final)    : {_scales_end}")
            if _taus_end is not None:
                print(f"  tau mean     (final)    : {[round(x, 4) for x in _taus_end]}")
            if _conv_lam_end is not None:
                print(f"  conv_gate lambda (final): {_conv_lam_end:.4f}")

            val_blocks, val_gate_means = _detail_validation(model, va_loader, dev)
            if val_gate_means:
                print('  Validation detail retained (mean g):', val_gate_means)

            if dev.type == 'cuda':
                torch.cuda.reset_peak_memory_stats()
                torch.cuda.synchronize()
            inference_start_time = time.time()

            model.eval()
            preds_list, trues_list = [], []
            with torch.no_grad():
                for xb, yb in te_loader:
                    out = model(xb.to(dev)).cpu().numpy()
                    preds_list.append(out)
                    trues_list.append(yb.numpy())

            if dev.type == 'cuda':
                torch.cuda.synchronize()
            inference_time_sec = time.time() - inference_start_time

            if dev.type == 'cuda':
                peak_gpu_mem_infer_gb = (
                    torch.cuda.max_memory_allocated() / 1024**3)
            else:
                peak_gpu_mem_infer_gb = 0.0

            preds = np.concatenate(preds_list, axis=0)
            trues = np.concatenate(trues_list, axis=0)

            metrics = compute_metrics(
                preds, trues,
                train_naive_mse_per_channel=train_naive_mse_per_channel,
            )
            metrics['val_by_lead'] = val_blocks  # all other metric keys above are TEST metrics
            metrics['val_gate_mean'] = val_gate_means
            all_results[horizon][seed] = dict(metrics, best_val_loss=best_val_loss)
            val_losses_by_seed[seed]   = val_losses
            train_losses_by_seed[seed] = train_losses

            _save_npy_dict(
                cache_path,
                {
                    'preds':              preds,
                    'trues':              trues,
                    'metrics':            metrics,
                    'best_val_loss':      best_val_loss,
                    'val_losses':         val_losses,
                    'train_losses':       train_losses,
                    'n_params':           n_params,
                    'training_time_sec':  training_time_sec,
                    'inference_time_sec': inference_time_sec,
                    'peak_gpu_mem_train_gb': peak_gpu_mem_train_gb,
                    'peak_gpu_mem_infer_gb': peak_gpu_mem_infer_gb,
                    'n_epochs_trained':   epochs_trained,
                    'variant_meta': {
                        'arch_version':          _AV,
                        'run_config':            run_config,
                        'skip_rank':             _SKIP_RANK,
                        'head_dropout':          _HEAD_DROP,
                        'detail_gate_layers':    [i for i, l in enumerate(model.encoder, 1) if l.detail_gate is not None],
                        'detail_beta_final':     [l.detail_gate.beta.item() for l in model.encoder if l.detail_gate is not None],
                        'use_channel_heads':     _CHHEAD,
                        'channel_head_gain_final': [
                            layer.attention.channel_gain().detach().cpu().tolist()
                            for layer in model.encoder if layer.attention.head_weights is not None and _CHHEAD
                        ],
                        'post_softmax_norm_type': NORM_TYPE,
                        'use_revin':             use_revin,
                        'use_tslib_padding':     use_tslib_padding,
                        'use_tslib_training':    use_tslib_training,
                        'capacity_matched':      capacity_matched,
                        'd_model':               effective_d_model,
                        'n_heads':               effective_n_heads,
                        'patch_len':             model.patch_len,
                        'stride':                model.stride,
                        'num_patches':           model.num_patches,
                        'conv_size':             model.conv_size,
                        'loss_alpha':            _L_ALPHA,
                        'use_learnable_tau':     _USE_TAU,
                        'tau_init':              _taus_init,
                        'tau_final':             _taus_end,
                        'use_conv_residual':     _USE_RES,
                        'conv_lambda_init':      _conv_lam_init,
                        'conv_lambda_final':     _conv_lam_end,
                        'optimizer':             'Adam' if use_tslib_training else 'AdamW',
                        'scheduler':             ('type1 (L,L,L/2,L/4,...)'
                                                  if use_tslib_training else 'cosine'),
                        'weight_decay':          0.0 if use_tslib_training else 1e-4,
                        'grad_clip':             None if use_tslib_training else GRAD_CLIP,
                        'output_scale_init':     _scales,
                        'output_scale_final':    _scales_end,
                    },
                },
            )

            print(f"  TEST [MAIN]  MSE={metrics['main']['MSE']:.4f}  "
                  f"MAE={metrics['main']['MAE']:.4f}")
            print(f"  TEST [SUPP]  RMSSE={metrics['supplementary']['RMSSE']:.4f}  "
                  f"MDA={metrics['supplementary']['MDA']:.4f}  "
                  f"Spearman={metrics['supplementary']['Spearman']:.4f}  "
                  f"Pearson={metrics['supplementary']['Pearson']:.4f}")

            print(f"  Training: {training_time_sec:.1f}s | "
                  f"Inference: {inference_time_sec:.2f}s | "
                  f"Peak GPU (train): {peak_gpu_mem_train_gb:.2f} GB | "
                  f"Epochs: {epochs_trained}")

        plot_train_val_losses(train_losses_by_seed, val_losses_by_seed,
                              horizon, dataset_name, model_variant, final_dir)

        mse_vals = [all_results[horizon][s]['MSE'] for s in _SD
                    if s in all_results[horizon]]
        mae_vals = [all_results[horizon][s]['MAE'] for s in _SD
                    if s in all_results[horizon]]
        if mse_vals:
            print(f"\n[H={horizon}]  "
                  f"MSE={np.mean(mse_vals):.4f}+/-{np.std(mse_vals):.4f}  "
                  f"MAE={np.mean(mae_vals):.4f}+/-{np.std(mae_vals):.4f}")

    out_path = os.path.abspath(os.path.join(
        final_dir, f"{dataset_name}_{model_variant}_results.json"))
    serialisable = {str(h): {str(s): v for s, v in seeds.items()}
                    for h, seeds in all_results.items()}
    os.makedirs(os.path.dirname(out_path) or '.', exist_ok=True)
    with open(out_path, 'w') as f:
        json.dump(serialisable, f, indent=2)
    print(f"\nResults saved -> {out_path}  "
          f"({os.path.getsize(out_path):,} bytes)")

    print(f"\n[VERIFY] Expected .npy files in {final_dir}:")
    missing = []
    for p in expected_npy_paths:
        if os.path.exists(p):
            print(f"  OK    {os.path.basename(p)}")
        else:
            print(f"  MISSING {p}")
            missing.append(p)
    if missing:
        print(f"\n[VERIFY] WARNING: {len(missing)} .npy file(s) missing.")
    else:
        print(f"[VERIFY] All {len(expected_npy_paths)} .npy file(s) present.")

    return all_results

# =============================================================================
# 8b. [PATCHING-ABLATION] Runner (Preserved from v6)
# =============================================================================
def run_patching_ablation(dataset_name, data_root, final_dir, device='cuda',
                          ablation_horizons=None, ablation_seeds=None,
                          epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                          patience=PATIENCE,
                          d_model_override=None, n_heads_override=None):
    """Run patch vs no-patch comparison and print a table."""
    if ablation_horizons is None:
        ablation_horizons = list(ABLATION_HORIZONS)
    if ablation_seeds is None:
        ablation_seeds = list(ABLATION_SEEDS)

    print("\n" + "#" * 80)
    print("# PATCHING ABLATION")
    print("#" * 80)
    print(f"# Dataset   : {dataset_name}")
    print(f"# Horizons  : {ablation_horizons}")
    print(f"# Seeds     : {ablation_seeds}")
    print(f"# Variant A : patch_len={_ABL_PATCH_CFG['patch_len']}, "
          f"stride={_ABL_PATCH_CFG['stride']}")
    print(f"# Variant B : patch_len={_ABL_NOPATCH_CFG['patch_len']}, "
          f"stride={_ABL_NOPATCH_CFG['stride']}")
    print("#" * 80 + "\n")

    print("\n" + "=" * 80)
    print(f"[ABLATION A] WITH PATCHING "
          f"(patch_len={_ABL_PATCH_CFG['patch_len']}, "
          f"stride={_ABL_PATCH_CFG['stride']})")
    print("=" * 80)
    patch_results = train_pivotformer(
        dataset_name=dataset_name, data_root=data_root,
        final_dir=final_dir, device=device,
        model_variant=_ABL_PATCH_CFG['variant_name'],
        epochs=epochs, batch_size=batch_size, lr=lr, patience=patience,
        d_model_override=d_model_override, n_heads_override=n_heads_override,
        patch_len_override=_ABL_PATCH_CFG['patch_len'],
        stride_override=_ABL_PATCH_CFG['stride'],
        horizons_override=ablation_horizons,
        seeds_override=ablation_seeds,
        arch_version_override=_ABL_PATCH_CFG['arch_version'],
        detail_gate_mode_override='none',
    )

    print("\n" + "=" * 80)
    print(f"[ABLATION B] NO PATCHING "
          f"(patch_len={_ABL_NOPATCH_CFG['patch_len']}, "
          f"stride={_ABL_NOPATCH_CFG['stride']})")
    print("=" * 80)
    nopatch_results = train_pivotformer(
        dataset_name=dataset_name, data_root=data_root,
        final_dir=final_dir, device=device,
        model_variant=_ABL_NOPATCH_CFG['variant_name'],
        epochs=epochs, batch_size=batch_size, lr=lr, patience=patience,
        d_model_override=d_model_override, n_heads_override=n_heads_override,
        patch_len_override=_ABL_NOPATCH_CFG['patch_len'],
        stride_override=_ABL_NOPATCH_CFG['stride'],
        horizons_override=ablation_horizons,
        seeds_override=ablation_seeds,
        arch_version_override=_ABL_NOPATCH_CFG['arch_version'],
        detail_gate_mode_override='none',
    )

    print("\n" + "#" * 80)
    print("# ABLATION RESULTS: Patching vs No-Patching")
    print("#" * 80)
    print(f"\n{'Horizon':>8s} | {'Patch MSE':>12s} | {'NoPatch MSE':>13s} | "
          f"{'Delta':>10s} | {'Delta %':>9s} | Winner")
    print("-" * 80)

    for h in ablation_horizons:
        p_list = [patch_results[h][s]['MSE'] for s in ablation_seeds
                  if h in patch_results and s in patch_results[h]]
        n_list = [nopatch_results[h][s]['MSE'] for s in ablation_seeds
                  if h in nopatch_results and s in nopatch_results[h]]

        if not p_list or not n_list:
            print(f"{h:>8d} | {'(missing)':>12s} | {'(missing)':>13s} | "
                  f"{'-':>10s} | {'-':>9s} | -")
            continue

        p_mean = float(np.mean(p_list))
        n_mean = float(np.mean(n_list))
        delta  = n_mean - p_mean
        delta_pct = (delta / p_mean) * 100 if p_mean > 0 else 0.0
        winner = "Patch" if p_mean < n_mean else ("No-Patch" if n_mean < p_mean else "Tie")
        print(f"{h:>8d} | {p_mean:>12.4f} | {n_mean:>13.4f} | "
              f"{delta:>+10.4f} | {delta_pct:>+8.2f}% | {winner}")

    ablation_json_path = os.path.join(
        _detail_root(final_dir), f'{dataset_name}_patching_ablation.json')
    with open(ablation_json_path, 'w') as f:
        json.dump({
            'patch':   {str(h): {str(s): patch_results[h][s] for s in ablation_seeds
                                 if h in patch_results and s in patch_results[h]}
                        for h in ablation_horizons},
            'nopatch': {str(h): {str(s): nopatch_results[h][s] for s in ablation_seeds
                                 if h in nopatch_results and s in nopatch_results[h]}
                        for h in ablation_horizons},
        }, f, indent=2)
    print(f"\nAblation JSON saved -> {ablation_json_path}")
    return patch_results, nopatch_results

# =============================================================================
# 8c. [Q1 FULL 4-PART ABLATION SUITE] Runner
# =============================================================================
def run_full_q1_ablation_suite(dataset_name, data_root, final_dir, device='cuda',
                               ablation_horizons=None, ablation_seeds=None,
                               epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                               patience=PATIENCE,
                               d_model_override=None, n_heads_override=None):
    """Runs all Q1 ablations systematically and prints a unified comparison table."""
    if ablation_horizons is None:
        ablation_horizons = list(ABLATION_HORIZONS)
    if ablation_seeds is None:
        ablation_seeds = list(ABLATION_SEEDS)

    configs = [
        {
            'name': '0_v5_Baseline',
            'variant': 'abl_v5_base_cm',
            'arch_version': 'v7_abl0_base',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 1.0, 'use_tau': False, 'use_res': False,
        },
        {
            'name': '1_CombinedLoss(a=0.8)',
            'variant': 'abl_combloss_cm',
            'arch_version': 'v7_abl1_loss',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 0.8, 'use_tau': False, 'use_res': False,
        },
        {
            'name': '2_LearnableTau',
            'variant': 'abl_tau_cm',
            'arch_version': 'v7_abl2_tau',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 1.0, 'use_tau': True, 'use_res': False,
        },
        {
            'name': '3_ConvResidual',
            'variant': 'abl_convres_cm',
            'arch_version': 'v7_abl3_res',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 1.0, 'use_tau': False, 'use_res': True,
        },
        {
            'name': '4_v7_All_Combined',
            'variant': 'abl_v7_all_cm',
            'arch_version': 'v7_abl4_all',
            'patch_len': PATCH_LEN, 'stride': STRIDE,
            'loss_alpha': 0.8, 'use_tau': True, 'use_res': True,
        },
        {
            'name': '5_v7_NoPatch(1,1)',
            'variant': 'abl_v7_nopatch_cm',
            'arch_version': 'v7_abl5_nopatch',
            'patch_len': 1, 'stride': 1,
            'loss_alpha': 0.8, 'use_tau': True, 'use_res': True,
        },
    ]

    all_cfg = next(cfg for cfg in configs if cfg['variant'] == 'abl_v7_all_cm')
    configs.append(dict(all_cfg, name='6_v7_All_ChannelHeads', variant='abl_v7_all_cm_chhead'))
    configs.append(dict(all_cfg, name='7_v7_Selective', variant='abl_v7_all_cm_selhead'))

    suite_results = {}
    for cfg in configs:
        print("\n" + "=" * 84)
        print(f">>> RUNNING Q1 ABLATION VARIANT: {cfg['name']} <<<")
        print("=" * 84)
        res = train_pivotformer(
            dataset_name=dataset_name,
            data_root=data_root,
            final_dir=final_dir,
            device=device,
            model_variant=cfg['variant'],
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
            patience=patience,
            d_model_override=d_model_override,
            n_heads_override=n_heads_override,
            patch_len_override=cfg['patch_len'],
            stride_override=cfg['stride'],
            horizons_override=ablation_horizons,
            seeds_override=ablation_seeds,
            arch_version_override=cfg['arch_version'],
            loss_alpha_override=cfg['loss_alpha'],
            use_learnable_tau_override=cfg['use_tau'],
            use_conv_residual_override=cfg['use_res'],
            detail_gate_mode_override='none',
        )
        suite_results[cfg['name']] = res

    print("\n" + "#" * 96)
    print(f"# Q1 ABLATION STUDY SUMMARY TABLE — Dataset: {dataset_name} "
          f"(Horizons: {ablation_horizons}, Seeds: {ablation_seeds})")
    print("#" * 96)
    header = (f"{'Configuration':<24s} | {'MSE':>9s} | {'dMSE%':>8s} | "
              f"{'MAE':>9s} | {'MDA':>9s} | {'RMSSE':>9s}")
    print(header)
    print("-" * 96)

    base_mse = None
    summary_payload = {}

    for cfg in configs:
        name = cfg['name']
        res  = suite_results[name]
        mses, maes, mdas, rmsses = [], [], [], []
        for h in ablation_horizons:
            for s in ablation_seeds:
                if h in res and s in res[h]:
                    m = res[h][s]
                    mses.append(m['MSE'])
                    maes.append(m['MAE'])
                    mdas.append(m['MDA'])
                    rmsses.append(m['RMSSE'])
        if not mses:
            continue
        mean_mse   = float(np.mean(mses))
        mean_mae   = float(np.mean(maes))
        mean_mda   = float(np.mean(mdas))
        mean_rmsse = float(np.mean(rmsses))

        if base_mse is None:
            base_mse = mean_mse
            dmse_str = "   0.00%"
        else:
            dmse_pct = ((mean_mse - base_mse) / base_mse) * 100.0 if base_mse > 0 else 0.0
            dmse_str = f"{dmse_pct:>+7.2f}%"

        print(f"{name:<24s} | {mean_mse:>9.4f} | {dmse_str:>8s} | "
              f"{mean_mae:>9.4f} | {mean_mda:>9.4f} | {mean_rmsse:>9.4f}")

        summary_payload[name] = {
            'MSE_mean': mean_mse,
            'MAE_mean': mean_mae,
            'MDA_mean': mean_mda,
            'RMSSE_mean': mean_rmsse,
        }

    print("#" * 96)
    suite_json_path = os.path.join(_detail_root(final_dir), f"{dataset_name}_q1_full_ablation_suite.json")
    with open(suite_json_path, 'w') as f:
        json.dump(summary_payload, f, indent=2)
    print(f"\nQ1 Ablation Suite JSON saved -> {suite_json_path}")
    return suite_results

def run_detail_gate_ablation(dataset_name, data_root, final_dir, *, horizons=None,
                             seeds=None, **train_kwargs):
    """Compare like with like at each horizon, on validation (not test)."""
    horizons = list(DETAIL_GATE_HORIZONS if horizons is None else horizons)
    seeds = list(DETAIL_GATE_SEEDS if seeds is None else seeds)
    if not horizons or not seeds:
        raise ValueError('At least one detail-gate horizon and seed are required.')
    rows, results = [], {}
    for mode in ('none', 'scalar', 'targeted'):
        runs = train_pivotformer(dataset_name, data_root, final_dir,
                                 horizons_override=horizons, seeds_override=seeds,
                                 detail_gate_mode_override=mode, **train_kwargs)
        results[mode] = runs
        for h in horizons:
            for s in seeds:
                blocks = runs[h][s]['val_by_lead']
                if not blocks:
                    raise RuntimeError(f'Cannot compare gate modes without validation: H={h}, seed={s}')
                for lead, scores in blocks.items():
                    base = results['none'][h][s]['val_by_lead'][lead]
                    rows.append(dict(mode=mode, H=h, seed=s, lead=lead, **scores,
                                     delta_MSE=scores['MSE'] - base['MSE'],
                                     delta_MAE=scores['MAE'] - base['MAE']))
    df = pd.DataFrame(rows)
    report = os.path.join(_detail_root(final_dir), f'{dataset_name}_last_gate_validation.csv')
    df.to_csv(report, index=False)
    print('\n[VALIDATION] Paired differences vs none; negative favors gate.')
    print(df.groupby(['H', 'lead', 'mode'], sort=False)[
        ['MSE', 'MAE', 'delta_MSE', 'delta_MAE']].mean().round(6).to_string())
    print(f'[VALIDATION] Per-seed rows: {report}')
    return results


# =============================================================================
# 9. ENTRY POINT
# =============================================================================
if __name__ == '__main__':
    import argparse

    _default_dataset   = globals().get('Dataset_Name', 'ETTh1')
    _default_data_root = globals().get('data_root', '.')
    _default_out_dir   = globals().get(
        'final_dir', f'./results/{_default_dataset}_baselines')
    _default_epochs    = globals().get('EPOCHS', 40)

    parser = argparse.ArgumentParser(description='Train PivotFormer v7.1')
    parser.add_argument('--gate_mode', choices=('none', 'scalar', 'targeted'), default=DETAIL_GATE_MODE)
    parser.add_argument('--gate_ablation', action='store_true', help='Compare all 3 detail-gate modes')
    parser.add_argument('--dataset',    type=str, default=_default_dataset)
    parser.add_argument('--data_root',  type=str, default=_default_data_root)
    parser.add_argument('--out_dir',    type=str, default=_default_out_dir)
    parser.add_argument('--variant',    type=str, default='full_cm')
    parser.add_argument('--device',     type=str, default='cuda')
    parser.add_argument('--epochs',     type=int, default=_default_epochs)
    parser.add_argument('--d_model',    type=int, default=None)
    parser.add_argument('--n_heads',    type=int, default=None)
    parser.add_argument('--loss_alpha', type=float, default=None,
                        help='Alpha in alpha*MSE + (1-alpha)*MAE')
    parser.add_argument('--skip_rank',  type=int, default=None,
                        help='Low-rank bottleneck rank for conv_skip_proj')
    parser.add_argument('--ablation',   action='store_true',
                        help='Run patching ablation instead of standard training')
    parser.add_argument('--q1_suite',   action='store_true',
                        help='Run all 4 Q1 ablations and print comparison table')

    def is_jupyter():
        try:
            from IPython import get_ipython
            shell = get_ipython()
            if shell is None:
                return False
            s = str(type(shell)).lower()
            return ('ipykernel' in s) or ('zmqshell' in s) or ('colab' in s)
        except Exception:
            return False

    if is_jupyter():
        print("[INFO] Notebook detected -> using default args.")
        args, _ = parser.parse_known_args([])
    else:
        args, _ = parser.parse_known_args()

    _run_q1_suite = ABLATION_Q1_SUITE or args.q1_suite
    _run_ablation = ABLATION_PATCHING or args.ablation
    _run_gate_ablation = ABLATION_DETAIL_GATE or args.gate_ablation

    if sum(map(bool, (_run_gate_ablation, _run_q1_suite, _run_ablation))) > 1:
        raise ValueError('Choose only one ablation suite at a time.')

    if _run_gate_ablation:
        print('[DETAIL] Validation comparison: none / scalar / targeted')
        detail_gate_results = run_detail_gate_ablation(
            args.dataset, args.data_root, args.out_dir, device=args.device,
            model_variant=args.variant, epochs=args.epochs, d_model_override=args.d_model,
            n_heads_override=args.n_heads, loss_alpha_override=args.loss_alpha,
            skip_rank_override=args.skip_rank)
    elif _run_q1_suite:
        print("\n" + "=" * 80)
        print(">>> Q1 FULL 4-PART ABLATION SUITE MODE <<<")
        print("=" * 80)
        run_full_q1_ablation_suite(
            dataset_name=args.dataset,
            data_root=args.data_root,
            final_dir=args.out_dir,
            device=args.device,
            ablation_horizons=ABLATION_HORIZONS,
            ablation_seeds=ABLATION_SEEDS,
            epochs=args.epochs,
            d_model_override=args.d_model,
            n_heads_override=args.n_heads,
        )
    elif _run_ablation:
        print("\n" + "=" * 80)
        print(">>> PATCHING ABLATION MODE <<<")
        print(">>> Set ABLATION_PATCHING = False for full training <<<")
        print("=" * 80)
        run_patching_ablation(
            dataset_name=args.dataset,
            data_root=args.data_root,
            final_dir=args.out_dir,
            device=args.device,
            ablation_horizons=ABLATION_HORIZONS,
            ablation_seeds=ABLATION_SEEDS,
            epochs=args.epochs,
            d_model_override=args.d_model,
            n_heads_override=args.n_heads,
        )
    else:
        print("\n" + "=" * 80)
        print(f">>> STANDARD TRAINING MODE (PivotFormer v7.1 | detail gate: {args.gate_mode}) <<<")
        print(f">>> Horizons={HORIZONS}, Seeds={SEEDS} <<<")
        print("=" * 80)
        train_pivotformer(
            dataset_name=args.dataset,
            data_root=args.data_root,
            final_dir=args.out_dir,
            device=args.device,
            model_variant=args.variant,
            epochs=args.epochs,
            d_model_override=args.d_model,
            n_heads_override=args.n_heads,
            loss_alpha_override=args.loss_alpha,
            skip_rank_override=args.skip_rank,
            detail_gate_mode_override=args.gate_mode,
        )

[DATA] load_multivariate_data is TSLib-compatible (seq_len overlap).
[INFO] Notebook detected -> using default args.

>>> STANDARD TRAINING MODE (v7.1 Low-Rank Skip | detail gate: none) <<<
>>> Horizons=[96, 192, 336, 720], Seeds=[0, 1, 2, 3, 4, 5] <<<
[PATHS] cwd       : /content/drive/MyDrive/RubickFormer
[ARCH] ARCH_VERSION = v7_lowrank_skip_r32_selhead_rank1_v1_detailgate_v1_none | NORM_TYPE = rmsnorm
[ABLATION CONFIG] patch_len=8, stride=4, loss_alpha=1, learnable_tau=True (init=0.5), conv_residual=True, skip_rank=None, head_dropout=0.1
[RUN SCOPE] horizons=[96, 192, 336, 720], seeds=[0, 1, 2, 3, 4, 5]
Device: cuda
[PATHS] final_dir : /content/drive/MyDrive/RubickFormer/results/ETTh1_baselines/RubickFormer_lowrank_v7_1/none/5dc1811c85ec2e6c
[DETAIL] mode=none | kernel=None | init_keep=None
  patch preview = 23 patches via Rubick padding (_fit_patching)
[VARIANT] full_cm
  use_revin          = True
  use_tslib_padding  = False
  use_tslib_training = False
  capacity_matched   = Tru

KeyboardInterrupt: 

# Compare

In [ ]:
# Comparison only: reuse the helpers already defined by the PivotFormer train cell.
try:
    _pf_metric = train_pivotformer.__globals__['compute_metrics']
    _pf_load = train_pivotformer.__globals__['_load_npy_dict']
except (NameError, KeyError) as exc:
    raise RuntimeError('Load the PivotFormer train-cell definitions in this kernel first (training is not needed).') from exc
import os, glob, re, sys, time, json
import numpy as np
import pandas as pd                                    # [NEW-1]
import matplotlib.pyplot as plt
from scipy import stats
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.table import WD_TABLE_ALIGNMENT

# ------------------------------------------------------------
# 0. Setup
# ------------------------------------------------------------
MAIN_MODEL = 'PivotFormer'

# The configuration selects the experiment; saved artifacts establish coverage.
from pathlib import Path
from inspect import signature
for _key in ('Dataset_Name','HORIZONS','SEEDS','LOOKBACK','final_dir'):
    if _key not in globals():
        raise RuntimeError(f'Missing {_key}: run the configuration cell first.')
for _key, _values in [('HORIZONS', HORIZONS), ('SEEDS', SEEDS)]:
    if not len(_values) or len(set(_values)) != len(_values):
        raise ValueError(f'{_key} must be nonempty and unique')
METRIC_KEYS = ['MSE','MAE','RMSSE','MDA','Spearman','Pearson','Params']
LOWER_BETTER = {'MSE','MAE','RMSSE','Params'}
REFERENCE_H = HORIZONS[0]
_root = Path(final_dir)
if not _root.is_dir(): raise FileNotFoundError(_root)
MAIN_VARIANT = globals().get('COMPARISON_VARIANT', signature(train_pivotformer).parameters['model_variant'].default)
_pattern = re.compile(rf'(?P<m>.+)_{re.escape(Dataset_Name)}_(?P<h>\d+)_seed(?P<s>\d+)(?:_preds)?\.npy')
_found = set()
for _f in _root.glob('*.npy'):
    _match = _pattern.fullmatch(_f.name)
    if _match and int(_match['h']) in HORIZONS and int(_match['s']) in SEEDS:
        _found.add(_match['m'])
baseline_models = sorted(_found)
if not baseline_models: raise ValueError(f'No baseline files matching configured dataset, horizons and seeds in {_root}')
print(f'Dataset={Dataset_Name}, lookback={LOOKBACK}, horizons={HORIZONS}, seeds={SEEDS}')
print(f'Result directory={_root}; main variant={MAIN_VARIANT}; baseline candidates={baseline_models}')

def get_pivot_path(h, s):
    return os.path.join(
        final_dir, f"{Dataset_Name}_{MAIN_VARIANT}_H{h}_seed{s}.npy")

def get_base_path(m, h, s):
    return os.path.join(
        final_dir, f"{m}_{Dataset_Name}_{h}_seed{s}.npy")

def get_base_preds_path(m, h, s):
    return os.path.join(
        final_dir, f"{m}_{Dataset_Name}_{h}_seed{s}_preds.npy")

def load_pivotformer(fp):
    try:
        obj = _pf_load(fp)
    except Exception as e:
        print(f"   ERROR loading {os.path.basename(fp)}: {e}")
        return None
    if not isinstance(obj, dict):
        return None
    m    = obj.get('metrics', {})
    flat = {}
    flat.update(m.get('main', {}))
    flat.update(m.get('supplementary', {}))
    for k in METRIC_KEYS:
        if k in m and k not in flat:
            flat[k] = m[k]
    flat['Params'] = int(obj.get('n_params', 0))
    return {
        'metrics':            flat,
        'preds':              obj.get('preds'),
        'trues':              obj.get('trues'),
        'TrainTime_sec':      float(obj.get('training_time_sec', np.nan)),
        'InferenceTime_sec':  float(obj.get('inference_time_sec', np.nan)),
        'PeakGPU_train_GB':   float(obj.get('peak_gpu_mem_train_gb', np.nan)),
        'PeakGPU_infer_GB':   float(obj.get('peak_gpu_mem_infer_gb', np.nan)),
    }

def load_baseline(fp):
    try:
        arr = np.load(fp, allow_pickle=True)
    except Exception as e:
        print(f"   ERROR loading {os.path.basename(fp)}: {e}")
        return None
    if isinstance(arr, np.ndarray) and arr.dtype == object and arr.ndim == 0:
        obj = arr.item()
        if isinstance(obj, dict):
            m = obj.get('metrics', {})
            flat = {}
            flat.update(m.get('main', {}))
            flat.update(m.get('supplementary', {}))
            flat['Params'] = int(obj.get('n_params', 0))
            return {
                'metrics':            flat,
                'preds':              obj.get('preds'),
                'trues':              obj.get('trues'),
                'TrainTime_sec':      np.nan,
                'InferenceTime_sec':  np.nan,
                'PeakGPU_train_GB':   np.nan,
                'PeakGPU_infer_GB':   np.nan,
            }
    if isinstance(arr, np.ndarray) and arr.dtype != object and arr.ndim == 1:
        if arr.size >= 7:
            rec = {k: float(arr[i]) for i, k in enumerate(METRIC_KEYS)}
            train_t = float(arr[7]) if arr.size >= 9 else np.nan
            return {
                'metrics':            rec,
                'preds':              None,
                'trues':              None,
                'TrainTime_sec':      train_t,
                'InferenceTime_sec':  np.nan,
                'PeakGPU_train_GB':   np.nan,
                'PeakGPU_infer_GB':   np.nan,
            }
    return None

def try_load_baseline_preds(m, h, s):
    fp = get_base_preds_path(m, h, s)
    if not os.path.exists(fp):
        return None, None
    try:
        obj = np.load(fp, allow_pickle=True).item()
        return obj.get('preds'), obj.get('trues')
    except Exception as e:
        print(f"   could not load preds {os.path.basename(fp)}: {e}")
        return None, None

# ------------------------------------------------------------
# 5. Load everything
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("STEP 3 - Load data")
print("=" * 72)

data = {m: {} for m in [MAIN_MODEL] + baseline_models}

for h in HORIZONS:
    recs = []
    for s in SEEDS:
        fp = get_pivot_path(h, s)
        if os.path.exists(fp):
            r = load_pivotformer(fp)
            if r:
                r['seed'] = s
                recs.append(r)
    if recs:
        data[MAIN_MODEL][h] = recs

for bm in baseline_models:
    for h in HORIZONS:
        recs = []
        for s in SEEDS:
            fp = get_base_path(bm, h, s)
            if os.path.exists(fp):
                r = load_baseline(fp)
                if r:
                    preds, trues = try_load_baseline_preds(bm, h, s)
                    if preds is not None:
                        r['preds'] = preds
                        r['trues'] = trues
                    r['seed'] = s
                    recs.append(r)
        if recs:
            data[bm][h] = recs

model_names = [MAIN_MODEL] + baseline_models
coverage = []
for m in model_names:
    for h in HORIZONS:
        records = {r['seed']:r for r in data[m].get(h, [])}
        for seed in SEEDS:
            r = records.get(seed)
            status = ('missing/unreadable record' if r is None else 'missing predictions/targets'
                      if r.get('preds') is None or r.get('trues') is None else 'available')
            coverage.append(dict(model=m,horizon=h,seed=seed,status=status))
coverage = pd.DataFrame(coverage)
output_dir = _root / 'comparison_configured'
output_dir.mkdir(exist_ok=True)
coverage.to_csv(output_dir / f'{Dataset_Name}_coverage.csv',index=False)
_bad = coverage[coverage.status != 'available']
if not _bad.empty: raise ValueError('Incomplete configured comparison:\n'+_bad.to_string(index=False))
print("\n   Loaded data summary:")
for m in model_names:
    parts = []
    for h in HORIZONS:
        n    = len(data[m].get(h, []))
        hasp = (n > 0 and data[m][h][0].get('preds') is not None)
        parts.append(f"H={h}: {n}s{'p' if hasp else ' '}")
    print(f"   {m:14s} -> {' | '.join(parts)}")

print("\n   Split verification (per-horizon test window counts):")
for h in HORIZONS:
    counts = {}
    for m in model_names:
        recs = data[m].get(h, [])
        if recs and recs[0].get('preds') is not None:
            counts[m] = recs[0]['preds'].shape[0]
    if counts:
        vals = set(counts.values())
        status = "OK" if len(vals) == 1 else "MISMATCH"
        print(f"   H={h:3d}  {status}  {counts}")
    else:
        print(f"   H={h:3d}  no preds stored")

# ------------------------------------------------------------
# 6. Statistical tests
# ------------------------------------------------------------
def rank_biserial(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    d = x - y; nz = d != 0
    if not np.any(nz):
        return 0.0
    r  = stats.rankdata(np.abs(d[nz]))
    rp = r[d[nz] > 0].sum(); rn = r[d[nz] < 0].sum()
    t  = rp + rn
    return 0.0 if t == 0 else (rp - rn) / t

def paired_wilcoxon(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    if len(a) != len(b) or len(a) < 3:
        return np.nan, np.nan
    if np.array_equal(a, b):
        return 1.0, 0.0
    try:
        _, p = stats.wilcoxon(a, b, alternative='two-sided',
                              zero_method='wilcox')
        return p, rank_biserial(a, b)
    except Exception as e:
        print(f"   Wilcoxon failed: {e}")
        return np.nan, np.nan


# Strict paired design: missing files, different targets or shuffled windows
# must not silently change the sample behind a comparison. Test stride = 1.
def origin_scores(p, t):
    # Columns: squared error, absolute error, strict directional hits.
    scores = np.empty((len(p), 3), dtype=np.float64)
    if not (np.isfinite(p).all() and np.isfinite(t).all()):
        raise ValueError('Non-finite saved predictions/targets')
    for i in range(0, len(p), 64):
        sl = slice(i, i+64)
        a,b = p[sl],t[sl]
        e = a-b
        scores[sl,0] = (e**2).mean(axis=(1,2))
        scores[sl,1] = np.abs(e).mean(axis=(1,2))
        scores[sl,2] = ((np.diff(a,axis=1)*np.diff(b,axis=1))>0).mean(axis=(1,2))
    ref = _pf_metric(p[:1],t[:1])
    np.testing.assert_allclose(scores[0],[ref[k] for k in ('MSE','MAE','MDA')],rtol=1e-5,atol=1e-8)
    return scores

def origin_dm(d, h):
    # Bartlett HAC, lags 1,...,H-1 for unit-stride overlapping origins.
    n = len(d); u = d - d.mean()
    if n <= h: return np.nan, np.nan
    v = np.dot(u, u) / n
    for k in range(1, h):
        v += 2 * (1 - k/h) * np.dot(u[k:], u[:-k]) / n
    if not v > 0: return np.nan, np.nan
    z = d.mean() / np.sqrt(v/n)
    return z, 2 * stats.norm.sf(abs(z))

def block_ci(d, length, rng, reps=5000):
    # Circular moving-block percentile CI for a vector of paired means.
    n, k = d.shape
    if n <= length: return np.full((2,k), np.nan)
    prefix = np.vstack([np.zeros(k), np.cumsum(np.vstack([d,d[:length]]),axis=0)])
    full, rem = divmod(n, length)
    starts = rng.integers(n, size=(reps,full))
    totals = (prefix[starts+length]-prefix[starts]).sum(axis=1)
    if rem:
        starts = rng.integers(n,size=reps)
        totals += prefix[starts+rem]-prefix[starts]
    return np.quantile(totals/n,[.025,.975],axis=0)

agg, wilcox, dm_results, mda_analysis, ci_rows = {}, {}, {}, {}, []
for h in HORIZONS:
    agg[h], wilcox[h], dm_results[h] = {}, {}, {}
    origin_mean = {}; reference = None
    for m in model_names:
        rs = sorted(data[m].get(h, []), key=lambda r:r['seed'])
        if [r['seed'] for r in rs] != sorted(SEEDS):
            raise ValueError(f'Incomplete paired seed set: {m}, H={h}; expected {SEEDS}')
        total = None
        for r in rs:
            tag = (m,h,r['seed'])
            p,t = np.asarray(r['preds']),np.asarray(r['trues'])
            if t.ndim != 3 or p.shape != t.shape or t.shape[1] != h or min(t.shape)<1 or h<2:
                raise ValueError(f'Missing or invalid [origin,horizon,channel] arrays: {tag}')
            if reference is None:
                reference = t
                if not np.array_equal(t[:-1,1:],t[1:,:-1]):
                    raise ValueError(f'Targets do not form chronological stride-one windows: {tag}')
            elif not np.array_equal(reference,t):
                raise ValueError(f'Targets differ across models/seeds: check split, scale, order: {tag}')
            print(f'Scoring {m}, H={h}, seed={r["seed"]}', flush=True)
            scores = origin_scores(p,t)
            for j,key in enumerate(('MSE','MAE','MDA')):
                r['metrics'][key] = scores[:,j].mean()
            if total is None: total = np.zeros_like(scores)
            total += scores
        origin_mean[m] = total/len(rs)
        for key in ('MSE','MAE','RMSSE','MDA','Spearman','Pearson'):
            vals = [r['metrics'].get(key,np.nan) for r in rs]
            agg[h].setdefault(key,{})[m] = (np.mean(vals),np.std(vals))
    for m in model_names:
        if m == MAIN_MODEL: continue
        by_seed = lambda name: {r['seed']:r['metrics']['MSE'] for r in data[name][h]}
        a,b = by_seed(MAIN_MODEL),by_seed(m)
        pv,eff = paired_wilcoxon([a[s] for s in SEEDS],[b[s] for s in SEEDS])
        wilcox[h][m] = dict(p=pv,effect=eff)
        d = origin_mean[m] - origin_mean[MAIN_MODEL]
        d[:,2] *= -1  # positive favors main on all three metrics
        z,pv = origin_dm(d[:,0],h)
        dm_results[h][m] = dict(stat=z,p=pv,mode='origin-MSE')
        lo,hi = block_ci(d,h,np.random.default_rng(2026+h))
        for j,key in enumerate(('MSE','MAE','MDA')):
            ci_rows.append(dict(horizon=h,baseline=m,metric=key,n_origins=len(d),
                n_seeds=len(SEEDS),block_length=h,equivalent_blocks=len(d)/h,
                advantage=d[:,j].mean(),ci_low=lo[j],ci_high=hi[j]))
        adv=d[:,2]; tol=1e-12
        assert np.isclose(adv.mean(),agg[h]['MDA'][MAIN_MODEL][0]-agg[h]['MDA'][m][0],rtol=0,atol=1e-12)
        mda_analysis[h,m] = dict(win=(adv>tol).mean(),tie=(abs(adv)<=tol).mean(),
                                 loss=(adv < -tol).mean(),mean=adv.mean())
ci_table = pd.DataFrame(ci_rows)
# output_dir set during coverage check
os.makedirs(output_dir,exist_ok=True)
ci_table.to_csv(os.path.join(output_dir,f'{Dataset_Name}_origin_intervals.csv'),index=False)
print(ci_table.to_string(index=False))

# 7c. Per-horizon Params
# ------------------------------------------------------------
params_by_horizon     = {h: {} for h in HORIZONS}
train_time_by_horizon = {h: {} for h in HORIZONS}
peak_gpu_by_horizon   = {h: {} for h in HORIZONS}
infer_time_by_horizon = {h: {} for h in HORIZONS}

for h in HORIZONS:
    for m in model_names:
        if h in data.get(m, {}) and data[m][h]:
            p_list = [int(d['metrics']['Params']) for d in data[m][h]
                      if 'Params' in d['metrics']]
            if p_list:
                params_by_horizon[h][m] = int(round(np.mean(p_list)))
            else:
                params_by_horizon[h][m] = None

            tt = np.array([d.get('TrainTime_sec', np.nan)
                           for d in data[m][h]], dtype=float)
            it = np.array([d.get('InferenceTime_sec', np.nan)
                           for d in data[m][h]], dtype=float)
            gp = np.array([d.get('PeakGPU_train_GB', np.nan)
                           for d in data[m][h]], dtype=float)
            train_time_by_horizon[h][m] = (np.nanmean(tt)
                                            if not np.all(np.isnan(tt))
                                            else np.nan)
            infer_time_by_horizon[h][m] = (np.nanmean(it)
                                            if not np.all(np.isnan(it))
                                            else np.nan)
            peak_gpu_by_horizon[h][m] = (np.nanmean(gp)
                                          if not np.all(np.isnan(gp))
                                          else np.nan)
        else:
            params_by_horizon[h][m]     = None
            train_time_by_horizon[h][m] = np.nan
            infer_time_by_horizon[h][m] = np.nan
            peak_gpu_by_horizon[h][m]   = np.nan

def get_mean_params_over_all(m):
    all_p = []
    for h in HORIZONS:
        if h in data.get(m, {}):
            for d in data[m][h]:
                if 'Params' in d['metrics']:
                    all_p.append(int(d['metrics']['Params']))
    return int(round(np.mean(all_p))) if all_p else None

def get_params_range_over_all(m):
    all_p = [params_by_horizon[h][m] for h in HORIZONS
             if params_by_horizon[h].get(m) is not None]
    if not all_p:
        return None, None
    return min(all_p), max(all_p)

def get_mean_train_time_over_all(m):
    vals = []
    for h in HORIZONS:
        if h in data.get(m, {}):
            for d in data[m][h]:
                tt = d.get('TrainTime_sec', np.nan)
                if not np.isnan(tt):
                    vals.append(tt)
    return float(np.mean(vals)) if vals else np.nan

def get_mean_peak_gpu_over_all(m):
    vals = []
    for h in HORIZONS:
        if h in data.get(m, {}):
            for d in data[m][h]:
                gp = d.get('PeakGPU_train_GB', np.nan)
                if not np.isnan(gp):
                    vals.append(gp)
    return float(np.mean(vals)) if vals else np.nan

# ------------------------------------------------------------
# 7e. Per-channel MSE
# ------------------------------------------------------------
per_channel_mse = {}
for m in model_names:
    per_channel_mse[m] = {}
    for h in HORIZONS:
        recs = data[m].get(h, [])
        ch_mses = []
        for r in recs:
            p = r.get('preds'); t = r.get('trues')
            if p is None or t is None:
                continue
            p = np.asarray(p); t = np.asarray(t)
            if p.shape != t.shape:
                continue
            mse_c = np.mean((p - t) ** 2, axis=(0, 1))
            ch_mses.append(mse_c)
        if ch_mses:
            per_channel_mse[m][h] = np.mean(ch_mses, axis=0)

if any(per_channel_mse[m] for m in model_names):
    try:
        per_ch_frames = []
        for m in model_names:
            for h in HORIZONS:
                if h in per_channel_mse[m]:
                    arr = per_channel_mse[m][h]
                    df = pd.DataFrame({
                        'model':   m,
                        'horizon': h,
                        'channel': np.arange(len(arr)),
                        'MSE':     arr,
                    })
                    per_ch_frames.append(df)
        if per_ch_frames:
            all_per_ch = pd.concat(per_ch_frames, ignore_index=True)
            csv_path = os.path.join(
                output_dir, f"{Dataset_Name}_per_channel_mse.csv")
            all_per_ch.to_csv(csv_path, index=False)
            print(f"\n   Per-channel MSE CSV -> {csv_path}")
    except Exception as e:
        print(f"   Per-channel CSV save failed: {e}")

if mda_analysis:
    print("\n" + "=" * 72)
    print("DIRECTIONAL ADVANTAGE (per-window, over test set)")
    print("=" * 72)
    print(f"  For each test window, directional accuracy (MDA) is")
    print(f"  computed for {MAIN_MODEL} and each baseline, then compared.")
    print(f"  {'Horizon':>8s} | {'Baseline':14s} | "
          f"{'Win%':>6s} | {'Tie%':>6s} | {'Loss%':>6s} | {'MeanΔ':>8s}")
    print("  " + "-" * 72)
    for (h, m), s in sorted(mda_analysis.items()):
        print(f"  {h:>8d} | {m:14s} | "
              f"{s['win']*100:5.1f}% | {s['tie']*100:5.1f}% | "
              f"{s['loss']*100:5.1f}% | {s['mean']:+.4f}")
    print("=" * 72)

# ------------------------------------------------------------
# 8. Console summary
# ------------------------------------------------------------
def get_significance(p):
    if p is None or np.isnan(p): return '-'
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

print("\n" + "=" * 110)
print(f"FAIR COMPARISON - {Dataset_Name} "
      f"(LOOKBACK={LOOKBACK}, n_seeds={len(SEEDS)})")
print("=" * 110)
print(f"NOTE: DM stat is signed so POSITIVE = {MAIN_MODEL} better.")
print("      Effect size is rank-biserial on MSE:")
print(f"      negative = {MAIN_MODEL} better, positive = worse.")

for h in HORIZONS:
    print(f"\nHorizon = {h}")
    print("-" * 110)
    print(f"{'Model':14s} | {'MSE':>18s} | {'MAE':>18s} | "
          f"{'Wilcoxon p':>11s} | {'DM stat':>8s} | {'DM p':>10s} | Mode")
    print("-" * 110)
    for m in model_names:
        mse_m, mse_s = agg[h]['MSE'].get(m, (np.nan, np.nan))
        mae_m, mae_s = agg[h]['MAE'].get(m, (np.nan, np.nan))
        wp = wilcox[h].get(m, {}).get('p', np.nan)
        dm = dm_results[h].get(m, {'stat': np.nan, 'p': np.nan, 'mode': '-'})

        mse_s_str = f"{mse_m:8.4f}+/-{mse_s:.4f}" if not np.isnan(mse_m) else "     -      "
        mae_s_str = f"{mae_m:8.4f}+/-{mae_s:.4f}" if not np.isnan(mae_m) else "     -      "
        wp_str    = (f"{get_significance(wp):>4s} {wp:6.3g}"
                     if not np.isnan(wp) else f"{'--':>4s} {'--':>6s}")

        print(f"{m:14s} | {mse_s_str}  | "
              f"{mae_s_str}  | "
              f"{wp_str} | "
              f"{dm['stat']:8.3f} | {dm['p']:10.3g} | {dm['mode']}")
print("=" * 110)

print("\n" + "=" * 110)
print("AGGREGATED RESULTS (mean over ALL horizons AND ALL seeds)")
print("=" * 110)
print(f"{'Model':14s} | {'MSE':>10s} | {'MAE':>10s} | {'RMSSE':>10s} | "
      f"{'MDA':>10s} | {'Spearman':>10s} | {'Params(mean)':>14s}")
print("-" * 110)

def _mean_metric_over_all(m, met):
    vals = []
    for h in HORIZONS:
        if h in data.get(m, {}):
            for d in data[m][h]:
                v = d['metrics'].get(met, np.nan)
                if not np.isnan(v):
                    vals.append(v)
    return float(np.mean(vals)) if vals else np.nan

for m in model_names:
    mse_v = _mean_metric_over_all(m, 'MSE')
    mae_v = _mean_metric_over_all(m, 'MAE')
    rms_v = _mean_metric_over_all(m, 'RMSSE')
    mda_v = _mean_metric_over_all(m, 'MDA')
    sp_v  = _mean_metric_over_all(m, 'Spearman')
    p_v   = get_mean_params_over_all(m)
    p_str = f"{p_v:,}" if p_v is not None else "-"
    print(f"{m:14s} | {mse_v:10.4f} | {mae_v:10.4f} | {rms_v:10.4f} | "
          f"{mda_v:10.4f} | {sp_v:10.4f} | {p_str:>14s}")

print("=" * 110)

# [NEW-2b] Metric winners summary
print("\n" + "=" * 72)
print("METRIC WINNERS (aggregated over horizons and seeds)")
print("=" * 72)
for met in ['MSE', 'MAE', 'RMSSE', 'MDA', 'Spearman', 'Pearson']:
    vals = {m: _mean_metric_over_all(m, met) for m in model_names}
    vals = {m: v for m, v in vals.items() if not np.isnan(v)}
    if not vals:
        continue
    if met in LOWER_BETTER:
        winner = min(vals, key=vals.get)
    else:
        winner = max(vals, key=vals.get)
    print(f"  {met:10s} -> {winner:15s} ({vals[winner]:.4f})")
print("=" * 72)

print("\n" + "=" * 100)
print("COMPUTATIONAL COST (mean over ALL horizons AND ALL seeds)")
print("=" * 100)
print(f"{'Model':14s} | {'Params(mean)':>13s} | {'Params range':>22s} | "
      f"{'TrainTime(s)':>13s} | {'PeakGPU(GB)':>12s}")
print("-" * 100)
for m in model_names:
    p_mean = get_mean_params_over_all(m)
    p_min, p_max = get_params_range_over_all(m)
    tt_mean = get_mean_train_time_over_all(m)
    gp_mean = get_mean_peak_gpu_over_all(m)
    p_mean_s = f"{p_mean:,}" if p_mean is not None else "-"
    if p_min is not None and p_max is not None:
        p_range_s = (f"{p_min:,} - {p_max:,}"
                     if p_min != p_max else f"{p_min:,}")
    else:
        p_range_s = "-"
    tt_s = f"{tt_mean:.1f}" if not np.isnan(tt_mean) else "-"
    gp_s = f"{gp_mean:.2f}" if not np.isnan(gp_mean) else "-"
    print(f"{m:14s} | {p_mean_s:>13s} | {p_range_s:>22s} | "
          f"{tt_s:>13s} | {gp_s:>12s}")
print("=" * 100)

# ------------------------------------------------------------
# [NEW-4] Full flat metrics CSV (one row per model × horizon × seed)
# ------------------------------------------------------------
try:
    csv_rows = []
    for h in HORIZONS:
        for m in model_names:
            recs = data[m].get(h, [])
            for s_idx, rec in enumerate(recs):
                csv_rows.append({
                    'dataset': Dataset_Name,
                    'model': m,
                    'horizon': h,
                    'seed': rec['seed'],
                    'MSE': rec['metrics'].get('MSE', np.nan),
                    'MAE': rec['metrics'].get('MAE', np.nan),
                    'RMSSE': rec['metrics'].get('RMSSE', np.nan),
                    'MDA': rec['metrics'].get('MDA', np.nan),
                    'Spearman': rec['metrics'].get('Spearman', np.nan),
                    'Pearson': rec['metrics'].get('Pearson', np.nan),
                    'Params': rec['metrics'].get('Params', np.nan),
                    'TrainTime_sec': rec.get('TrainTime_sec', np.nan),
                    'InferenceTime_sec': rec.get('InferenceTime_sec', np.nan),
                    'PeakGPU_train_GB': rec.get('PeakGPU_train_GB', np.nan),
                })
    if csv_rows:
        csv_path = os.path.join(output_dir,
                                f'{Dataset_Name}_all_metrics.csv')
        pd.DataFrame(csv_rows).to_csv(csv_path, index=False)
        print(f"\n   Full metrics CSV -> {csv_path}  "
              f"({len(csv_rows)} rows)")
except Exception as e:
    print(f"   [WARN] Could not write full metrics CSV: {e}")

# ------------------------------------------------------------
# 9. Plots
# ------------------------------------------------------------
def get_color(m):
    return '#d62728' if m == MAIN_MODEL else '#7f8f7f'

colors = [get_color(m) for m in model_names]

def plot_metric(metric_name, filename_suffix):
    fig, axes = plt.subplots(1, len(HORIZONS),
                             figsize=(5 * len(HORIZONS), 5))
    if len(HORIZONS) == 1:
        axes = [axes]
    for i, h in enumerate(HORIZONS):
        ax = axes[i]
        means  = [agg[h][metric_name].get(m, (np.nan, np.nan))[0]
                  for m in model_names]
        errors = [agg[h][metric_name].get(m, (np.nan, np.nan))[1]
                  for m in model_names]
        means  = [0 if np.isnan(v) else v for v in means]
        errors = [0 if np.isnan(v) else v for v in errors]
        x = np.arange(len(model_names))
        ax.bar(x, means, yerr=errors, capsize=5, color=colors, alpha=0.8)
        y_max = max(means + [e for e in errors if e > 0]) if means else 1
        for j, m in enumerate(model_names):
            if m != MAIN_MODEL and metric_name == 'MSE':
                sig = get_significance(wilcox[h].get(m, {}).get('p', np.nan))
                if sig and sig != 'ns' and sig != '-':
                    ax.text(x[j], means[j] + errors[j] + y_max * 0.03,
                            sig, ha='center', fontsize=11, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(model_names, rotation=45, ha='right', fontsize=8)
        ax.set_title(f"{metric_name} @ horizon {h}")
        ax.set_ylabel(metric_name)
    fig.suptitle(f"{Dataset_Name} - {metric_name} "
                 f"(mean across {len(SEEDS)} seeds; error bars = SD)")
    fig.tight_layout()
    out = os.path.join(output_dir, f"{Dataset_Name}_{filename_suffix}.png")
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"   Saved: {out}")

plot_metric('MSE', 'compare_mse')
plot_metric('MAE', 'compare_mae')

fig, ax = plt.subplots(figsize=(8, 5.5))
for m in model_names:
    maes = [_mean_metric_over_all(m, 'MAE')]
    if np.isnan(maes[0]):
        continue
    p_mean = get_mean_params_over_all(m)
    p_min, p_max = get_params_range_over_all(m)
    if p_mean is None:
        continue
    mean_mae = maes[0]
    if p_min is not None and p_max is not None and p_min != p_max:
        ax.plot([p_min, p_max], [mean_mae, mean_mae],
                color=get_color(m), alpha=0.35, linewidth=6,
                solid_capstyle='butt', zorder=1)
    ax.scatter(p_mean, mean_mae, s=140, color=get_color(m),
               label=m, edgecolors='black', linewidth=1.2,
               zorder=3 if m == MAIN_MODEL else 2)
    ax.annotate(m, (p_mean, mean_mae), fontsize=8,
                xytext=(6, 4), textcoords='offset points')
ax.set_xscale('log')
ax.set_xlabel('Mean Parameters (over horizons and seeds, log scale)')
ax.set_ylabel('Mean MAE (over horizons and seeds)')
ax.set_title(
    f"{Dataset_Name} - Efficiency: Accuracy vs Model Size\n"
    f"Marker = mean Params; bar = range of per-horizon Params")
ax.grid(True, which='both', ls='--', alpha=0.3)
fig.tight_layout()
out = os.path.join(output_dir, f"{Dataset_Name}_efficiency.png")
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f"   Saved: {out}")

fig, ax = plt.subplots(figsize=(8, 4))
dm_matrix = np.full((len(baseline_models), len(HORIZONS)), np.nan)
for i, m in enumerate(baseline_models):
    if m not in model_names:
        continue
    for j, h in enumerate(HORIZONS):
        dm_matrix[i, j] = dm_results[h].get(m, {}).get('stat', np.nan)
im = ax.imshow(dm_matrix, cmap='RdBu_r', aspect='auto',
               vmin=-5, vmax=5)
ax.set_xticks(range(len(HORIZONS)))
ax.set_xticklabels([f"H={h}" for h in HORIZONS])
ax.set_yticks(range(len(baseline_models)))
ax.set_yticklabels(baseline_models)
for i in range(len(baseline_models)):
    for j in range(len(HORIZONS)):
        v = dm_matrix[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha='center', va='center',
                    color='white' if abs(v) > 2.5 else 'black', fontsize=9)
ax.set_title(f"DM statistic ({MAIN_MODEL} vs baselines) - "
             f"positive = {MAIN_MODEL} better")
fig.colorbar(im, ax=ax, label='DM stat')
fig.tight_layout()
out = os.path.join(output_dir, f"{Dataset_Name}_dm_heatmap.png")
fig.savefig(out, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f"   Saved: {out}")

# ------------------------------------------------------------
# 10. Word report
# ------------------------------------------------------------
doc = Document()
doc.add_heading(f"{Dataset_Name}: Extended Comparison "
                f"({MAIN_MODEL} vs. evaluated baselines)", level=1)

def add_note(document, txt):
    p = document.add_paragraph()
    r = p.add_run(txt)
    r.font.size = Pt(8); r.italic = True

def find_best_model_for_column(metric, h):
    valid = {}
    for m in model_names:
        mv, _ = agg[h][metric].get(m, (np.nan, np.nan))
        if not np.isnan(mv):
            valid[m] = mv
    if not valid:
        return None
    if metric in LOWER_BETTER:
        return min(valid, key=valid.get)
    return max(valid, key=valid.get)

def set_cell_bold(cell, bold=True):
    for p in cell.paragraphs:
        for r in p.runs:
            r.bold = bold

doc.add_heading("Detailed Results (per horizon)", level=2)
for h in HORIZONS:
    doc.add_heading(f"Horizon = {h}", level=3)
    cols = ['Model', 'MSE', 'MAE', 'RMSSE', 'MDA', 'Spearman',
            'Params', 'Wilcoxon p', 'Effect', 'DM stat', 'DM p']
    t = doc.add_table(rows=1, cols=len(cols))
    t.style = 'Light Grid Accent 1'
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, c in enumerate(cols):
        t.rows[0].cells[i].paragraphs[0].add_run(c).bold = True

    best_per_metric = {}
    for met in ['MSE', 'MAE', 'RMSSE', 'MDA', 'Spearman']:
        best_per_metric[met] = find_best_model_for_column(met, h)

    for m in model_names:
        cells = t.add_row().cells
        cells[0].text = m
        for ci, met in enumerate(['MSE', 'MAE', 'RMSSE',
                                  'MDA', 'Spearman'], 1):
            mv, sv = agg[h][met].get(m, (np.nan, np.nan))
            cells[ci].text = ('-' if np.isnan(mv)
                              else f"{mv:.4f}+/-{sv:.4f}")
            if (not np.isnan(mv)) and best_per_metric[met] == m:
                set_cell_bold(cells[ci], True)

        prm = params_by_horizon[h].get(m)
        cells[6].text = '-' if prm is None else f"{prm:,}"
        if m == MAIN_MODEL:
            for k in range(7, 11):
                cells[k].text = '-'
        else:
            wp = wilcox[h].get(m, {}).get('p', np.nan)
            we = wilcox[h].get(m, {}).get('effect', np.nan)
            dm = dm_results[h].get(m, {'stat': np.nan, 'p': np.nan})
            cells[7].text = ('-' if np.isnan(wp)
                             else f"{wp:.3g} {get_significance(wp)}")
            cells[8].text = '-' if np.isnan(we) else f"{we:.3f}"
            cells[9].text = ('-' if np.isnan(dm['stat'])
                             else f"{dm['stat']:.3f}")
            cells[10].text = ('-' if np.isnan(dm['p'])
                              else f"{dm['p']:.3g} {get_significance(dm['p'])}")

add_note(doc,
    f"Params column shows the parameter count at this specific horizon "
    f"(the head scales with pred_len). "
    f"Best value per metric column is bolded. "
    f"Wilcoxon signed-rank on per-seed MSE: *** p<0.001, ** p<0.01, "
    f"* p<0.05, ns not significant. "
    f"Wilcoxon effect size is rank-biserial on MSE; "
    f"negative effect = {MAIN_MODEL} better, positive = worse. "
    f"DM on squared loss: one seed-averaged difference per origin; Bartlett HAC through lag H-1; unadjusted p-values; "
    f"positive DM stat = {MAIN_MODEL} better.")

doc.add_heading("Aggregated Results (mean over all horizons and all seeds)",
                level=2)
cols = ['Model', 'MSE', 'MAE', 'RMSSE', 'MDA', 'Spearman',
        'Params (mean)']
t = doc.add_table(rows=1, cols=len(cols))
t.style = 'Light Grid Accent 1'
t.alignment = WD_TABLE_ALIGNMENT.CENTER
for i, c in enumerate(cols):
    t.rows[0].cells[i].paragraphs[0].add_run(c).bold = True

best_agg = {}
for met in ['MSE', 'MAE', 'RMSSE', 'MDA', 'Spearman']:
    valid = {}
    for m in model_names:
        v = _mean_metric_over_all(m, met)
        if not np.isnan(v):
            valid[m] = v
    if valid:
        best_agg[met] = (min(valid, key=valid.get)
                         if met in LOWER_BETTER
                         else max(valid, key=valid.get))

for m in model_names:
    cells = t.add_row().cells
    cells[0].text = m
    for ci, met in enumerate(['MSE', 'MAE', 'RMSSE',
                              'MDA', 'Spearman'], 1):
        v = _mean_metric_over_all(m, met)
        cells[ci].text = f"{v:.4f}" if not np.isnan(v) else '-'
        if (not np.isnan(v)) and best_agg.get(met) == m:
            set_cell_bold(cells[ci], True)

    p_mean = get_mean_params_over_all(m)
    cells[6].text = f"{p_mean:,}" if p_mean is not None else '-'


note = doc.add_paragraph()
rr = note.add_run(
    f"All values in this table (MSE, MAE, RMSSE, MDA, Spearman, Params) "
    f"are averaged over ALL {len(HORIZONS)} horizons AND ALL {len(SEEDS)} "
    f"seeds (n = {len(HORIZONS)*len(SEEDS)}). "
    f"Protocol: LOOKBACK={LOOKBACK}. "
    "MDA: consecutive steps within the forecast; either change equal to zero is a miss.")
rr.font.size = Pt(8); rr.italic = True

doc.add_heading('Paired advantage by forecast origin', level=2)
add_note(doc, f'Positive favors {MAIN_MODEL}. Pointwise 95% circular block-bootstrap intervals; block length H, 5000 draws, conditional on the available seeds. No pooling across horizons. Interpret intervals cautiously when there are few blocks, especially at long horizons.')
t = doc.add_table(rows=1,cols=6)
for cell,label in zip(t.rows[0].cells,['H','Baseline','Metric','Advantage','95% CI','Origins / H']): cell.text=label
for row in ci_table.itertuples():
    for cell,val in zip(t.add_row().cells,[row.horizon,row.baseline,row.metric,f'{row.advantage:+.5f}',f'[{row.ci_low:+.5f}, {row.ci_high:+.5f}]',f'{row.equivalent_blocks:.1f}']): cell.text=str(val)

if len(SEEDS) < 6:
    warn_p = doc.add_paragraph()
    rw = warn_p.add_run(
        f"IMPORTANT: Only {len(SEEDS)} seed(s) available. "
        f"With n={len(SEEDS)}, the minimum achievable two-sided Wilcoxon "
        f"p-value is {2.0/(2**len(SEEDS)):.4g}.")
    rw.font.size = Pt(8); rw.italic = True

doc.add_heading(
    "Computational Cost (mean over all horizons and all seeds)", level=2)
cols = ['Model', 'Params (mean)', 'Params (range over H)',
        'Train Time (s)', 'Peak GPU (GB)', 'Note']
t = doc.add_table(rows=1, cols=len(cols))
t.style = 'Light Grid Accent 1'
t.alignment = WD_TABLE_ALIGNMENT.CENTER
for i, c in enumerate(cols):
    t.rows[0].cells[i].paragraphs[0].add_run(c).bold = True

for m in model_names:
    cells = t.add_row().cells
    cells[0].text = m
    p_mean = get_mean_params_over_all(m)
    p_min, p_max = get_params_range_over_all(m)
    tt_mean = get_mean_train_time_over_all(m)
    gp_mean = get_mean_peak_gpu_over_all(m)

    cells[1].text = f"{p_mean:,}" if p_mean is not None else '-'
    if p_min is not None and p_max is not None:
        cells[2].text = (f"{p_min:,} - {p_max:,}"
                         if p_min != p_max else f"{p_min:,}")
    else:
        cells[2].text = '-'
    cells[3].text = f"{tt_mean:.1f}" if not np.isnan(tt_mean) else '-'
    cells[4].text = f"{gp_mean:.2f}" if not np.isnan(gp_mean) else '-'
    cells[5].text = ("training loop only" if m == MAIN_MODEL
                     else "subprocess wall-time")

add_note(doc,
    f"Params (mean) is averaged over ALL {len(HORIZONS)} horizons AND ALL "
    f"{len(SEEDS)} seeds. The range column reports the min and max of the "
    f"per-horizon seed-mean Params. Peak GPU is reported only for "
    f"{MAIN_MODEL} (measured in-process).")

if any(per_channel_mse[m] for m in model_names):
    doc.add_heading(
        f"Per-Channel MSE (mean across seeds, H={REFERENCE_H})", level=2)
    ch_count = None
    for m in model_names:
        if REFERENCE_H in per_channel_mse[m]:
            ch_count = len(per_channel_mse[m][REFERENCE_H])
            break
    if ch_count:
        cols = ['Model'] + [f'ch{i}' for i in range(ch_count)] + ['mean']
        t = doc.add_table(rows=1, cols=len(cols))
        t.style = 'Light Grid Accent 1'
        t.alignment = WD_TABLE_ALIGNMENT.CENTER
        for i, c in enumerate(cols):
            t.rows[0].cells[i].paragraphs[0].add_run(c).bold = True

        best_ch = {}
        for ci in range(ch_count):
            valid = {}
            for m in model_names:
                if REFERENCE_H in per_channel_mse[m]:
                    v = per_channel_mse[m][REFERENCE_H][ci]
                    if not np.isnan(v):
                        valid[m] = v
            if valid:
                best_ch[ci] = min(valid, key=valid.get)

        for m in model_names:
            if REFERENCE_H not in per_channel_mse[m]:
                continue
            cells = t.add_row().cells
            cells[0].text = m
            arr = per_channel_mse[m][REFERENCE_H]
            for ci in range(ch_count):
                cells[ci + 1].text = f"{arr[ci]:.4f}"
                if best_ch.get(ci) == m:
                    set_cell_bold(cells[ci + 1], True)
            cells[ch_count + 1].text = f"{np.mean(arr):.4f}"

        add_note(doc,
            f"Per-channel MSE at H={REFERENCE_H} (mean across seeds). "
            f"Best per channel is bolded.")

# ============================================================
# [NEW-3] Directional Advantage section
# ============================================================
if mda_analysis:
    doc.add_heading(
        "Directional Advantage (per-window, over test set)", level=2)
    doc.add_paragraph(
        f"For each test window, directional accuracy (MDA) is computed "
        f"for {MAIN_MODEL} and each baseline. The table reports the "
        f"fraction of windows where {MAIN_MODEL} has higher MDA (Win%), "
        f"the fraction where they tie (Tie%), and the fraction where the "
        f"baseline has higher MDA (Loss%). MeanΔ is the average "
        f"difference in MDA (positive = {MAIN_MODEL} better).")
    cols = ['Horizon', 'Baseline', 'Win%', 'Tie%', 'Loss%', 'MeanΔ MDA']
    t = doc.add_table(rows=1, cols=len(cols))
    t.style = 'Light Grid Accent 1'
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for i, c in enumerate(cols):
        t.rows[0].cells[i].paragraphs[0].add_run(c).bold = True
    for (h, m), s in sorted(mda_analysis.items()):
        cells = t.add_row().cells
        cells[0].text = str(h)
        cells[1].text = m
        cells[2].text = f"{s['win']*100:.1f}"
        cells[3].text = f"{s['tie']*100:.1f}"
        cells[4].text = f"{s['loss']*100:.1f}"
        cells[5].text = f"{s['mean']:+.4f}"
    add_note(doc,
        f"Win% + Tie% + Loss% = 100% per row. MeanΔ close to zero "
        f"indicates similar directional accuracy on average; a high "
        f"Win% indicates that {MAIN_MODEL} is more frequently correct "
        f"on direction even if the average gap is small.")

# ============================================================
# [NEW-3b] Patching Ablation section (reads *_patching_ablation.json)
# ============================================================
ablation_json_path = os.path.join(
    final_dir, f'{Dataset_Name}_patching_ablation.json')
if os.path.exists(ablation_json_path):
    try:
        with open(ablation_json_path) as f:
            abl = json.load(f)
        doc.add_heading("Patching Ablation", level=2)
        doc.add_paragraph(
            "Two tokenization schemes are compared: patch-level tokens "
            "(patch_len=8, stride=4) and per-timestep tokens "
            "(patch_len=1, stride=1). All other components (RevIN, "
            "RMSNorm, output_scale, PELU, attention, head) are shared. "
            "Negative Δ% indicates that removing patching degrades MSE.")
        abl_horizons = abl.get('config', {}).get('horizons', [])
        abl_seeds = abl.get('config', {}).get('seeds', [])
        cols = ['Horizon', 'Patch MSE', 'NoPatch MSE', 'Δ MSE',
                'Δ%', 'Winner']
        t = doc.add_table(rows=1, cols=len(cols))
        t.style = 'Light Grid Accent 1'
        t.alignment = WD_TABLE_ALIGNMENT.CENTER
        for i, c in enumerate(cols):
            t.rows[0].cells[i].paragraphs[0].add_run(c).bold = True

        for h in abl_horizons:
            h_key = str(h)
            p_vals, n_vals = [], []
            for s in abl_seeds:
                s_key = str(s)
                p_rec = (abl.get('results', {}).get('patch', {})
                            .get(h_key, {}).get(s_key))
                n_rec = (abl.get('results', {}).get('nopatch', {})
                            .get(h_key, {}).get(s_key))
                if p_rec and 'MSE' in p_rec:
                    p_vals.append(float(p_rec['MSE']))
                if n_rec and 'MSE' in n_rec:
                    n_vals.append(float(n_rec['MSE']))
            if not p_vals or not n_vals:
                continue
            p_mean = float(np.mean(p_vals))
            n_mean = float(np.mean(n_vals))
            delta = n_mean - p_mean
            delta_pct = (delta / p_mean * 100) if p_mean > 0 else 0.0
            if p_mean < n_mean:
                winner = "Patch"
            elif n_mean < p_mean:
                winner = "No-Patch"
            else:
                winner = "Tie"
            cells = t.add_row().cells
            cells[0].text = str(h)
            cells[1].text = f"{p_mean:.4f}"
            cells[2].text = f"{n_mean:.4f}"
            cells[3].text = f"{delta:+.4f}"
            cells[4].text = f"{delta_pct:+.2f}%"
            cells[5].text = winner

        add_note(doc,
            "Δ MSE = NoPatch MSE − Patch MSE. Positive Δ% means "
            "removing patching increases error, i.e., patching helps.")
    except Exception as e:
        print(f"   [WARN] Could not add patching ablation to docx: {e}")

out_docx = os.path.join(output_dir,
                        f"{Dataset_Name}_extended_comparison.docx")
doc.save(out_docx)
print(f"\nSaved report: {out_docx}")

# ------------------------------------------------------------
# 11. Final summary
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("COMPARISON COMPLETE")
print("=" * 72)
print(f"   Dataset               : {Dataset_Name}")
print(f"   PivotFormer variant   : {MAIN_VARIANT}")
print(f"   Seeds                 : {SEEDS}")
print(f"   Horizons              : {HORIZONS}")
print(f"   Models                : {model_names}")
print(f"   Aggregation           : mean over ({len(HORIZONS)} horizons x "
      f"{len(SEEDS)} seeds = {len(HORIZONS)*len(SEEDS)}) for all metrics "
      f"including Params")
print(f"   Results dir           : {final_dir}")
print(f"   Report                : {out_docx}")
if mda_analysis:
    print(f"   MDA analysis          : yes ({len(mda_analysis)} entries)")
if os.path.exists(ablation_json_path):
    print(f"   Patching ablation     : yes ({ablation_json_path})")
print("=" * 72)

Dataset=ETTh1, lookback=96, horizons=[96, 192, 336, 720], seeds=[0, 1, 2, 3, 4, 5]
Result directory=/mnt/d/Project/RubickFormer/results/ETTh1_baselines; main variant=full_cm; baseline candidates=['Crossformer', 'DLinear', 'PatchTST', 'iTransformer']

STEP 3 - Load data

   Loaded data summary:
   RubickFormer   -> H=96: 6sp | H=192: 6sp | H=336: 6sp | H=720: 6sp
   Crossformer    -> H=96: 6sp | H=192: 6sp | H=336: 6sp | H=720: 6sp
   DLinear        -> H=96: 6sp | H=192: 6sp | H=336: 6sp | H=720: 6sp
   PatchTST       -> H=96: 6sp | H=192: 6sp | H=336: 6sp | H=720: 6sp
   iTransformer   -> H=96: 6sp | H=192: 6sp | H=336: 6sp | H=720: 6sp

   Split verification (per-horizon test window counts):
   H= 96  OK  {'RubickFormer': 2785, 'Crossformer': 2785, 'DLinear': 2785, 'PatchTST': 2785, 'iTransformer': 2785}
   H=192  OK  {'RubickFormer': 2689, 'Crossformer': 2689, 'DLinear': 2689, 'PatchTST': 2689, 'iTransformer': 2689}
   H=336  OK  {'RubickFormer': 2545, 'Crossformer': 2545, 'DLinear':

# Diag

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def diagnose_saved_scales(horizon, seed, variant):
    if horizon not in HORIZONS or seed not in SEEDS:
        raise ValueError(
            f"H={horizon}, seed={seed} خارج از تنظیمات است: "
            f"HORIZONS={HORIZONS}, SEEDS={SEEDS}"
        )

    path = Path(final_dir) / f"{Dataset_Name}_{variant}_H{horizon}_seed{seed}.npy"
    if not path.is_file():
        raise FileNotFoundError(f"فایل اجرای انتخاب‌شده پیدا نشد:\n{path}")

    # همان loader سلول آموزش PivotFormer؛ هیچ تابع آموزشی فراخوانی نمی‌شود.
    try:
        load_record = train_pivotformer.__globals__["_load_npy_dict"]
    except (NameError, KeyError) as exc:
        raise RuntimeError(
            "ابتدا سلول تعریف PivotFormer (train_pivotformer) را در این کرنل اجرا کنید."
        ) from exc

    record = load_record(str(path))
    if not isinstance(record, dict):
        raise ValueError(f"رکورد قابل خواندن نیست: {path}")

    meta = record.get("variant_meta") or {}
    keys = ("output_scale_init", "output_scale_final")
    if any(meta.get(k) is None for k in keys):
        raise KeyError(
            f"این فایل مقادیر اولیه/نهایی output_scale را ندارد: {path.name}"
        )

    initial, best = (np.asarray(meta[k], dtype=float) for k in keys)
    if (initial.ndim != 1 or not initial.size or best.shape != initial.shape
            or not np.isfinite(initial).all() or not np.isfinite(best).all()):
        raise ValueError(f"مقادیر output_scale نامعتبر یا تعداد لایه‌ها ناهماهنگ است: {path.name}")

    delta = best - initial
    relative = np.full(initial.shape, np.nan)
    np.divide(100 * delta, np.abs(initial), out=relative, where=initial != 0)

    print(f"[DIAG] {path.name} | arch={meta.get('arch_version', '?')}")
    if meta.get("tau_init") is not None and meta.get("tau_final") is not None:
        print(f"       tau_init={meta['tau_init']} -> tau_final={[round(x, 4) for x in meta['tau_final']]}")
    if meta.get("conv_lambda_init") is not None and meta.get("conv_lambda_final") is not None:
        print(f"       conv_lambda: {meta['conv_lambda_init']:.4f} -> {meta['conv_lambda_final']:.4f}")

    return pd.DataFrame({
        "layer": np.arange(len(initial)),
        "scale_init": initial,
        "scale_best_val": best,
        "delta": delta,
        "change_pct": relative,
    })

# همان اجرای موردنظر شما؛ واریانت از سلول مقایسه گرفته می‌شود.
scale_diag = diagnose_saved_scales(
    horizon=192, seed=0, variant=globals().get("MAIN_VARIANT", "full_cm")
)
print(scale_diag.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

[DIAG] ETTh1_full_cm_H192_seed0.npy | arch=v7_q1_full_selhead_rank1_v1
 layer  scale_init  scale_best_val     delta  change_pct
     0    1.000000        0.975917 -0.024083   -2.408296
     1    1.000000        0.976880 -0.023120   -2.311981
     2    1.000000        1.063031  0.063031    6.303060
